# Learned Hybrid Metric for Reference-Based Evaluation of AI-Generated Summaries

## RoSE Experiment

This notebook contains the reproducible implementation of the RoSE CNN/DailyMail experiment used in the thesis.

The experiment evaluates whether the Learned Hybrid methodology generalises to a second human-annotated summarisation benchmark using ACU-based human content-coverage scores.

## 1. Environment Setup and Dataset Acquisition

This section prepares the Python environment and retrieves the RoSE CNN/DailyMail evaluation data required for the experiment.

The RoSE annotation file is obtained from the Salesforce RoSE dataset using a fixed dataset revision, while the corresponding CNN/DailyMail test documents are loaded to recover the source articles and reference summaries.

In [4]:
# ============================================================
# PHASE 1 — ENVIRONMENT SETUP
# ============================================================

!pip install -q -U datasets pandas pyarrow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.0 which is incompatible.
streamlit 1.37.1 requires pandas<3,>=1.3.0, but you have pandas 3.0.5 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 14.3.3 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
import datasets

from datasets import load_dataset

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("datasets:", datasets.__version__)

pandas: 3.0.1
numpy: 1.26.4
datasets: 5.0.1


In [10]:
# ============================================================
# PHASE 1 — LOAD RoSE CNNDM TEST ANNOTATIONS
# Compatible with modern Hugging Face datasets versions
# ============================================================

!pip install -q -U huggingface_hub

from huggingface_hub import hf_hub_download
import pandas as pd

ROSE_REVISION = "752be3b85f370a46d7379cc051e067236dca5280"

rose_file = hf_hub_download(
    repo_id="Salesforce/rose",
    repo_type="dataset",
    filename="cnndm.test.acus.aggregated.jsonl",
    revision=ROSE_REVISION
)

print("RoSE file downloaded:")
print(rose_file)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.0 which is incompatible.
streamlit 1.37.1 requires pandas<3,>=1.3.0, but you have pandas 3.0.5 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 14.3.3 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


cnndm.test.acus.aggregated.jsonl: 0.00B [00:00, ?B/s]

RoSE file downloaded:
/Users/sudip/.cache/huggingface/hub/datasets--Salesforce--rose/snapshots/752be3b85f370a46d7379cc051e067236dca5280/cnndm.test.acus.aggregated.jsonl


In [12]:
# ============================================================
# READ RoSE ANNOTATION FILE
# ============================================================

rose_raw_df = pd.read_json(
    rose_file,
    lines=True
)

print("RoSE raw shape:", rose_raw_df.shape)

print("\nColumns:")
print(rose_raw_df.columns.tolist())

display(rose_raw_df.head(2))

RoSE raw shape: (500, 7)

Columns:
['source', 'reference', 'reference_acus', 'count_id', 'example_id', 'annotations', 'system_outputs']


,source,reference,reference_acus,count_id,example_id,annotations,system_outputs
0,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...,"[Juan Arango escaped punishment., Juan Arango ...",0,000571afe702684d90c1d222ce70b1e1375c1016,"{'bart': {'acu_labels': [1, 1, 1, 1, 1, 1, 1, ...",{'bart': 'Juan Arango bites Jesus Zavela in a ...
1,Tim Sherwood will welcome Gary Gardner back in...,Gary Gardner confirms he'll report to Aston Vi...,[Gary Gardner confirms he'll report to Aston V...,1,000642916e3a6c33411c617cf2f3c134a206fba8,"{'bart': {'acu_labels': [0, 0, 1, 1, 1, 0, 1, ...",{'bart': 'Gary Gardner has enjoyed a successfu...


In [2]:
# ============================================================
# LOAD ORIGINAL CNN/DAILYMAIL 3.0.0 TEST SET
# ============================================================

from datasets import load_dataset

cnn_test = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0",
    split="test"
)

print("CNN/DailyMail test examples:", len(cnn_test))
print("Columns:", cnn_test.column_names)

display(cnn_test.select(range(2)).to_pandas())

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

CNN/DailyMail test examples: 11490
Columns: ['article', 'highlights', 'id']


,article,highlights,id
0,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,f001ec5c4704938247d27a44948eebb37ae98d01
1,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...",230c522854991d053fe98a718b1defa077a8efef


In [6]:
# ============================================================
# RECOVERY CELL — CREATE rose_raw_df
# ============================================================

import pandas as pd
from huggingface_hub import hf_hub_download

ROSE_REVISION = "752be3b85f370a46d7379cc051e067236dca5280"

# Download RoSE file if rose_file is not already available
if "rose_file" not in globals():

    rose_file = hf_hub_download(
        repo_id="Salesforce/rose",
        repo_type="dataset",
        filename="cnndm.test.acus.aggregated.jsonl",
        revision=ROSE_REVISION
    )

# Read JSONL into dataframe
rose_raw_df = pd.read_json(
    rose_file,
    lines=True
)

print("RoSE dataframe created successfully.")
print("Shape:", rose_raw_df.shape)

print("\nColumns:")
print(rose_raw_df.columns.tolist())

display(rose_raw_df.head(2))

RoSE dataframe created successfully.
Shape: (500, 7)

Columns:
['source', 'reference', 'reference_acus', 'count_id', 'example_id', 'annotations', 'system_outputs']


,source,reference,reference_acus,count_id,example_id,annotations,system_outputs
0,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...,"[Juan Arango escaped punishment., Juan Arango ...",0,000571afe702684d90c1d222ce70b1e1375c1016,"{'bart': {'acu_labels': [1, 1, 1, 1, 1, 1, 1, ...",{'bart': 'Juan Arango bites Jesus Zavela in a ...
1,Tim Sherwood will welcome Gary Gardner back in...,Gary Gardner confirms he'll report to Aston Vi...,[Gary Gardner confirms he'll report to Aston V...,1,000642916e3a6c33411c617cf2f3c134a206fba8,"{'bart': {'acu_labels': [0, 0, 1, 1, 1, 0, 1, ...",{'bart': 'Gary Gardner has enjoyed a successfu...


In [8]:
print("example_id exists:",
      "example_id" in rose_raw_df.columns)

print("\nFirst 5 IDs:")
print(rose_raw_df["example_id"].head())

example_id exists: True

First 5 IDs:
0    000571afe702684d90c1d222ce70b1e1375c1016
1    000642916e3a6c33411c617cf2f3c134a206fba8
2    00110802bc6eae0e8e4d3d22e27f458f41be2b22
3    00119229166ae09a6ef25c0e10b101ef9eb9cca3
4    0013aa16650fbcfbe6edb16ac614ad174cb5d1cf
Name: example_id, dtype: str


In [10]:
# ============================================================
# MATCH RoSE DOCUMENTS TO CNN/DAILYMAIL
# ============================================================

cnn_lookup = {
    row["id"]: {
        "source": row["article"],
        "reference": row["highlights"]
    }
    for row in cnn_test
}

missing_ids = [
    example_id
    for example_id in rose_raw_df["example_id"]
    if example_id not in cnn_lookup
]

print("Missing RoSE document IDs:", len(missing_ids))

if len(missing_ids) > 0:
    print("First missing IDs:", missing_ids[:10])

Missing RoSE document IDs: 0


In [12]:
rose_raw_df["source"] = rose_raw_df[
    "example_id"
].map(
    lambda x: cnn_lookup[x]["source"]
)

rose_raw_df["reference"] = rose_raw_df[
    "example_id"
].map(
    lambda x: cnn_lookup[x]["reference"]
)

print("Final RoSE raw shape:", rose_raw_df.shape)

display(
    rose_raw_df[
        [
            "example_id",
            "source",
            "reference"
        ]
    ].head()
)

Final RoSE raw shape: (500, 7)


,example_id,source,reference
0,000571afe702684d90c1d222ce70b1e1375c1016,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...
1,000642916e3a6c33411c617cf2f3c134a206fba8,Tim Sherwood will welcome Gary Gardner back in...,Gary Gardner confirms he'll report to Aston Vi...
2,00110802bc6eae0e8e4d3d22e27f458f41be2b22,For years medical experts have warned about th...,Current federal government guidelines dictate ...
3,00119229166ae09a6ef25c0e10b101ef9eb9cca3,It is the smallest computer in the world - and...,Michigan Micro Mote is a complete computer sys...
4,0013aa16650fbcfbe6edb16ac614ad174cb5d1cf,Britons heading to Europe on holiday are facin...,Britons finding they cannot pick up hire cars ...


In [14]:
# ============================================================
# STEP 4 — INSPECT RoSE SYSTEM OUTPUTS AND HUMAN ANNOTATIONS
# ============================================================

example = rose_raw_df.iloc[0].to_dict()

print("Top-level columns:")
print(rose_raw_df.columns.tolist())

print("\nAnnotation systems:")
print(list(example["annotations"].keys()))

print("\nSystem-output systems:")
print(list(example["system_outputs"].keys()))

print("\nNumber of annotated systems:")
print(len(example["annotations"]))

print("\nNumber of system outputs:")
print(len(example["system_outputs"]))

Top-level columns:
['source', 'reference', 'reference_acus', 'count_id', 'example_id', 'annotations', 'system_outputs']

Annotation systems:
['bart', 'gold', 'pegasus', 'brio', 'gsum', 'simcls', 'cliff', 'ctrlsum', 'frost', 'glob', 'matchsum', 'brio-ext']

System-output systems:
['bart', 'gold', 'pegasus', 'brio', 'gsum', 'simcls', 'cliff', 'ctrlsum', 'frost', 'glob', 'matchsum', 'brio-ext']

Number of annotated systems:
12

Number of system outputs:
12


In [16]:
# ============================================================
# STEP 5 — INSPECT ONE SYSTEM
# ============================================================

system_names = list(example["system_outputs"].keys())

first_system = system_names[0]

print("Example system:", first_system)

print("\nCandidate summary:")
print(example["system_outputs"][first_system])

print("\nHuman annotation:")
print(example["annotations"][first_system])

Example system: bart

Candidate summary:
Juan Arango bites Jesus Zavela in a moment of madness in Club Tijuana's 4-3 defeat by Monterrey in the Mexican league. The Venezuelan icon sank his teeth into the shoulder of the opponent as his temper flared in the defeat. He was not booked by the referee but could face a heavy retrospective ban. Arango had earlier curled in a magnificent free kick for his team to bring them level after falling 2-0 down.

Human annotation:
{'acu_labels': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'acu': 1.0, 'normalized_acu': 0.5503382086753841}


## 2. RoSE Dataset Preparation and Standardisation

This section converts the RoSE annotations and system outputs into a summary-level experimental dataset with one row per candidate summary.

Each record contains the source document, reference summary, candidate summary, generating system, document identifier, and ACU-based human content-coverage score. The resulting dataset contains 6,000 candidate summaries from 500 source documents and 12 summarisation systems.

In [18]:
# ============================================================
# STEP 6 — CONVERT RoSE INTO STANDARD MR604 FORMAT
# 500 documents × 12 systems = 6000 candidate summaries
# ============================================================

rows = []

summary_id = 0

for doc_index, row in rose_raw_df.iterrows():

    doc_id = row["example_id"]

    source = row["source"]
    reference = row["reference"]

    annotations = row["annotations"]
    system_outputs = row["system_outputs"]

    for model_name, candidate_summary in system_outputs.items():

        # Make sure corresponding human annotation exists
        if model_name not in annotations:
            continue

        annotation = annotations[model_name]

        acu_score = annotation.get("acu", None)

        rows.append({
            "summary_id": summary_id,
            "doc_id": doc_id,
            "model": model_name,
            "source": source,
            "reference_summary": reference,
            "candidate_summary": candidate_summary,
            "human_content_coverage": acu_score
        })

        summary_id += 1


rose_df = pd.DataFrame(rows)

print("Standard RoSE dataframe created.")
print("Shape:", rose_df.shape)

print("\nColumns:")
print(rose_df.columns.tolist())

display(rose_df.head())

Standard RoSE dataframe created.
Shape: (6000, 7)

Columns:
['summary_id', 'doc_id', 'model', 'source', 'reference_summary', 'candidate_summary', 'human_content_coverage']


,summary_id,doc_id,model,source,reference_summary,candidate_summary,human_content_coverage
0,0,000571afe702684d90c1d222ce70b1e1375c1016,bart,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...,Juan Arango bites Jesus Zavela in a moment of ...,1.0
1,1,000571afe702684d90c1d222ce70b1e1375c1016,gold,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...,Juan Arango bit Jesus Zavala's shoulder in Clu...,0.3
2,2,000571afe702684d90c1d222ce70b1e1375c1016,pegasus,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...,Club Tijuana lost 4-3 to Monterrey in the Mexi...,0.3
3,3,000571afe702684d90c1d222ce70b1e1375c1016,brio,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...,Juan Arango bites the shoulder of opponent Jes...,0.8
4,4,000571afe702684d90c1d222ce70b1e1375c1016,gsum,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...,Juan Arango bites Jesus Zavela in Club Tijuana...,0.9


In [20]:
# ============================================================
# STEP 7 — VALIDATE REPLICATION DATASET
# ============================================================

print("Total summaries:", len(rose_df))
print("Unique documents:", rose_df["doc_id"].nunique())
print("Unique systems:", rose_df["model"].nunique())

print("\nSummaries per document:")
print(
    rose_df.groupby("doc_id")
    .size()
    .describe()
)

print("\nMissing values:")
print(
    rose_df[
        [
            "doc_id",
            "model",
            "source",
            "reference_summary",
            "candidate_summary",
            "human_content_coverage"
        ]
    ].isnull().sum()
)

print("\nHuman ACU score distribution:")
print(
    rose_df["human_content_coverage"].describe()
)

Total summaries: 6000
Unique documents: 500
Unique systems: 12

Summaries per document:
count    500.0
mean      12.0
std        0.0
min       12.0
25%       12.0
50%       12.0
75%       12.0
max       12.0
dtype: float64

Missing values:
doc_id                    0
model                     0
source                    0
reference_summary         0
candidate_summary         0
human_content_coverage    0
dtype: int64

Human ACU score distribution:
count    6000.000000
mean        0.404678
std         0.237396
min         0.000000
25%         0.230769
50%         0.400000
75%         0.571429
max         1.000000
Name: human_content_coverage, dtype: float64


## 3. Document-Grouped Development and Locked Holdout Split

This section partitions the RoSE dataset at the source-document level to prevent document leakage between development and evaluation data.

The 500 source documents are split into 400 development documents and 100 locked holdout documents using a fixed random seed. All candidate summaries belonging to the same source document remain within the same partition.

In [24]:
# ============================================================
# PHASE 2 — DOCUMENT-LEVEL DEVELOPMENT / HOLDOUT SPLIT
# FIXED VERSION FOR ARROW STRING ARRAYS
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Convert document IDs to a normal NumPy array
unique_docs = np.array(
    rose_df["doc_id"].astype(str).unique().tolist()
)

print("Total unique documents:", len(unique_docs))
print("Type:", type(unique_docs))
print("First 5 document IDs:", unique_docs[:5])

# 80% development, 20% final holdout
development_docs, holdout_docs = train_test_split(
    unique_docs,
    test_size=0.20,
    random_state=RANDOM_STATE,
    shuffle=True
)

print("\nDevelopment documents:", len(development_docs))
print("Holdout documents:", len(holdout_docs))

Total unique documents: 500
Type: <class 'numpy.ndarray'>
First 5 document IDs: ['000571afe702684d90c1d222ce70b1e1375c1016'
 '000642916e3a6c33411c617cf2f3c134a206fba8'
 '00110802bc6eae0e8e4d3d22e27f458f41be2b22'
 '00119229166ae09a6ef25c0e10b101ef9eb9cca3'
 '0013aa16650fbcfbe6edb16ac614ad174cb5d1cf']

Development documents: 400
Holdout documents: 100


In [26]:
# ============================================================
# STEP 9 — CREATE DEVELOPMENT AND HOLDOUT SETS
# ============================================================

development_df = (
    rose_df[
        rose_df["doc_id"].astype(str).isin(development_docs)
    ]
    .copy()
    .reset_index(drop=True)
)

holdout_df = (
    rose_df[
        rose_df["doc_id"].astype(str).isin(holdout_docs)
    ]
    .copy()
    .reset_index(drop=True)
)

print("Development shape:", development_df.shape)
print("Holdout shape:", holdout_df.shape)

print(
    "Development documents:",
    development_df["doc_id"].nunique()
)

print(
    "Holdout documents:",
    holdout_df["doc_id"].nunique()
)

Development shape: (4800, 7)
Holdout shape: (1200, 7)
Development documents: 400
Holdout documents: 100


In [28]:
development_doc_set = set(
    development_df["doc_id"].astype(str)
)

holdout_doc_set = set(
    holdout_df["doc_id"].astype(str)
)

overlap = development_doc_set.intersection(
    holdout_doc_set
)

print(
    "Documents appearing in BOTH sets:",
    len(overlap)
)

if len(overlap) == 0:
    print("✓ PASS — No document leakage")
else:
    print("✗ ERROR — Document leakage detected")

Documents appearing in BOTH sets: 0
✓ PASS — No document leakage


In [30]:
# ============================================================
# STEP 10 — SYSTEM REPRESENTATION CHECK
# ============================================================

print("Development systems:")
print(
    development_df["model"]
    .value_counts()
    .sort_index()
)

print("\nHoldout systems:")
print(
    holdout_df["model"]
    .value_counts()
    .sort_index()
)

Development systems:
model
bart        400
brio        400
brio-ext    400
cliff       400
ctrlsum     400
frost       400
glob        400
gold        400
gsum        400
matchsum    400
pegasus     400
simcls      400
Name: count, dtype: int64

Holdout systems:
model
bart        100
brio        100
brio-ext    100
cliff       100
ctrlsum     100
frost       100
glob        100
gold        100
gsum        100
matchsum    100
pegasus     100
simcls      100
Name: count, dtype: int64


In [32]:
# ============================================================
# STEP 11 — HUMAN ACU DISTRIBUTION CHECK
# ============================================================

distribution_check = pd.DataFrame({
    "Full Dataset":
        rose_df["human_content_coverage"].describe(),

    "Development":
        development_df["human_content_coverage"].describe(),

    "Holdout":
        holdout_df["human_content_coverage"].describe()
})

display(distribution_check.round(4))

,Full Dataset,Development,Holdout
count,6000.0000,4800.0000,1200.0000
mean,0.4047,0.4091,0.3871
std,0.2374,0.2367,0.2394
min,0.0000,0.0000,0.0000
25%,0.2308,0.2353,0.2222
50%,0.4000,0.4000,0.3750
75%,0.5714,0.5714,0.5556
max,1.0000,1.0000,1.0000


## 4. Structural Feature Extraction

This section preprocesses the candidate and reference summaries and derives structural features used by the Learned Hybrid metric.

The structural feature set captures summary-length adequacy, sentence-count similarity, redundancy quality, and average sentence length. These features are computed consistently across the development and locked holdout partitions.

In [34]:
# ============================================================
# PHASE 3 — PREPROCESSING + STRUCTURAL FEATURES
# Same definitions as Dataset 1
# ============================================================

import re
import numpy as np
import pandas as pd


def clean_summary(text: str) -> str:
    text = str(text)
    text = re.sub(
        r"</?t>",
        " ",
        text,
        flags=re.IGNORECASE
    )
    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def word_count(text: str) -> int:
    cleaned = clean_summary(text)
    return len(
        re.findall(
            r"\b\w+\b",
            cleaned
        )
    )


def sentence_count(text: str) -> int:
    cleaned = clean_summary(text)

    sentence_parts = re.split(
        r"[.!?]+",
        cleaned
    )

    return max(
        1,
        sum(
            bool(part.strip())
            for part in sentence_parts
        )
    )


def symmetric_ratio(
    value_a: float,
    value_b: float
) -> float:

    if value_a <= 0 or value_b <= 0:
        return 0.0

    return min(
        value_a / value_b,
        value_b / value_a
    )


def repeated_ngram_rate(
    text: str,
    n: int = 2
) -> float:

    words = re.findall(
        r"\b\w+\b",
        clean_summary(text).lower()
    )

    if len(words) < n:
        return 0.0

    ngrams = [
        tuple(
            words[index:index + n]
        )
        for index in range(
            len(words) - n + 1
        )
    ]

    unique_ngrams = len(
        set(ngrams)
    )

    return max(
        0.0,
        1.0 - (
            unique_ngrams /
            len(ngrams)
        )
    )


# ------------------------------------------------------------
# Safe working copy
# ------------------------------------------------------------

feature_df = rose_df.copy()


# ------------------------------------------------------------
# Clean text
# ------------------------------------------------------------

feature_df[
    "source_clean"
] = feature_df[
    "source"
].map(clean_summary)


feature_df[
    "reference_clean"
] = feature_df[
    "reference_summary"
].map(clean_summary)


feature_df[
    "candidate_clean"
] = feature_df[
    "candidate_summary"
].map(clean_summary)


# ------------------------------------------------------------
# Word counts
# ------------------------------------------------------------

feature_df[
    "candidate_word_count"
] = feature_df[
    "candidate_summary"
].map(word_count)


feature_df[
    "reference_word_count"
] = feature_df[
    "reference_summary"
].map(word_count)


# ------------------------------------------------------------
# Length adequacy
# ------------------------------------------------------------

feature_df[
    "length_adequacy"
] = [
    symmetric_ratio(
        candidate,
        reference
    )
    for candidate, reference
    in zip(
        feature_df[
            "candidate_word_count"
        ],
        feature_df[
            "reference_word_count"
        ]
    )
]


# ------------------------------------------------------------
# Sentence counts
# ------------------------------------------------------------

feature_df[
    "candidate_sentence_count"
] = feature_df[
    "candidate_summary"
].map(sentence_count)


feature_df[
    "reference_sentence_count"
] = feature_df[
    "reference_summary"
].map(sentence_count)


# ------------------------------------------------------------
# Sentence-count ratio
# ------------------------------------------------------------

feature_df[
    "sentence_count_ratio"
] = [
    symmetric_ratio(
        candidate,
        reference
    )
    for candidate, reference
    in zip(
        feature_df[
            "candidate_sentence_count"
        ],
        feature_df[
            "reference_sentence_count"
        ]
    )
]


# ------------------------------------------------------------
# Repetition / redundancy
# ------------------------------------------------------------

feature_df[
    "repeated_bigram_rate"
] = feature_df[
    "candidate_summary"
].map(
    lambda x: repeated_ngram_rate(
        x,
        n=2
    )
)


feature_df[
    "redundancy_quality"
] = (
    1.0 -
    feature_df[
        "repeated_bigram_rate"
    ]
)


# ------------------------------------------------------------
# Average sentence length
# ------------------------------------------------------------

feature_df[
    "average_sentence_length"
] = (
    feature_df[
        "candidate_word_count"
    ]
    /
    feature_df[
        "candidate_sentence_count"
    ]
)


print(
    "Feature dataframe shape:",
    feature_df.shape
)

display(
    feature_df[
        [
            "summary_id",
            "doc_id",
            "model",
            "human_content_coverage",
            "candidate_word_count",
            "reference_word_count",
            "length_adequacy",
            "candidate_sentence_count",
            "reference_sentence_count",
            "sentence_count_ratio",
            "repeated_bigram_rate",
            "redundancy_quality",
            "average_sentence_length"
        ]
    ].head()
)

Feature dataframe shape: (6000, 19)


,summary_id,doc_id,model,human_content_coverage,candidate_word_count,reference_word_count,length_adequacy,candidate_sentence_count,reference_sentence_count,sentence_count_ratio,repeated_bigram_rate,redundancy_quality,average_sentence_length
0,0,000571afe702684d90c1d222ce70b1e1375c1016,bart,1.0,77,34,0.441558,4,3,0.75,0.026316,0.973684,19.250000
1,1,000571afe702684d90c1d222ce70b1e1375c1016,gold,0.3,42,34,0.809524,3,3,1.00,0.000000,1.000000,14.000000
2,2,000571afe702684d90c1d222ce70b1e1375c1016,pegasus,0.3,35,34,0.971429,3,3,1.00,0.000000,1.000000,11.666667
3,3,000571afe702684d90c1d222ce70b1e1375c1016,brio,0.8,51,34,0.666667,4,3,0.75,0.000000,1.000000,12.750000
4,4,000571afe702684d90c1d222ce70b1e1375c1016,gsum,0.9,65,34,0.523077,4,3,0.75,0.000000,1.000000,16.250000


In [36]:
# ============================================================
# STRUCTURAL FEATURE CHECK
# ============================================================

structural_columns = [
    "candidate_word_count",
    "reference_word_count",
    "length_adequacy",
    "candidate_sentence_count",
    "reference_sentence_count",
    "sentence_count_ratio",
    "repeated_bigram_rate",
    "redundancy_quality",
    "average_sentence_length"
]

print("Missing values:")
print(
    feature_df[
        structural_columns
    ].isnull().sum()
)

print("\nStructural feature statistics:")
display(
    feature_df[
        structural_columns
    ].describe().T.round(4)
)

Missing values:
candidate_word_count        0
reference_word_count        0
length_adequacy             0
candidate_sentence_count    0
reference_sentence_count    0
sentence_count_ratio        0
repeated_bigram_rate        0
redundancy_quality          0
average_sentence_length     0
dtype: int64

Structural feature statistics:


,count,mean,std,min,25%,50%,75%,max
candidate_word_count,6000.0,61.6385,18.4758,10.0000,49.0000,58.0000,72.0000,162.00
reference_word_count,6000.0,52.5600,18.9921,10.0000,39.0000,49.0000,61.0000,131.00
length_adequacy,6000.0,0.7450,0.1606,0.1136,0.6250,0.7609,0.8772,1.00
candidate_sentence_count,6000.0,3.7552,1.1134,1.0000,3.0000,4.0000,4.0000,10.00
reference_sentence_count,6000.0,3.9660,1.3465,1.0000,3.0000,4.0000,5.0000,12.00
sentence_count_ratio,6000.0,0.7753,0.1798,0.2000,0.6667,0.7500,1.0000,1.00
repeated_bigram_rate,6000.0,0.0183,0.0239,0.0000,0.0000,0.0139,0.0278,0.45
redundancy_quality,6000.0,0.9817,0.0239,0.5500,0.9722,0.9861,1.0000,1.00
average_sentence_length,6000.0,17.2241,5.7815,4.6667,13.0000,16.0000,20.0000,81.00


## 5. Individual Automatic-Metric Baselines

This section computes the individual automatic evaluation metrics used as baselines and as input features for the Learned Hybrid metric.

The evaluated metrics are ROUGE-L, BERTScore F1, and MoverScore. Their scores are calculated for the RoSE candidate summaries and later compared with the ACU-based human content-coverage scores.

In [38]:
# ============================================================
# PHASE 4 — ROUGE-L BASELINE
# ============================================================

!pip install -q rouge-score


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [40]:
from rouge_score import rouge_scorer
import numpy as np
import pandas as pd

print("ROUGE package ready.")

ROUGE package ready.


In [42]:
# ============================================================
# CREATE ROUGE-L SCORER
# ============================================================

rougeL_scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

print("ROUGE-L scorer created.")

ROUGE-L scorer created.


In [44]:
# ============================================================
# TEST ROUGE-L ON ONE EXAMPLE
# ============================================================

test_reference = feature_df.loc[
    0,
    "reference_clean"
]

test_candidate = feature_df.loc[
    0,
    "candidate_clean"
]

test_rouge = rougeL_scorer.score(
    test_reference,
    test_candidate
)

print("ROUGE-L Precision:",
      test_rouge["rougeL"].precision)

print("ROUGE-L Recall:",
      test_rouge["rougeL"].recall)

print("ROUGE-L F1:",
      test_rouge["rougeL"].fmeasure)

ROUGE-L Precision: 0.23376623376623376
ROUGE-L Recall: 0.5294117647058824
ROUGE-L F1: 0.32432432432432434


In [46]:
# ============================================================
# CALCULATE ROUGE-L FOR ALL RoSE SUMMARIES
# ============================================================

rougeL_precision = []
rougeL_recall = []
rougeL_f1 = []

total_rows = len(feature_df)

for i, row in feature_df.iterrows():

    scores = rougeL_scorer.score(
        row["reference_clean"],
        row["candidate_clean"]
    )

    rouge_score = scores["rougeL"]

    rougeL_precision.append(
        rouge_score.precision
    )

    rougeL_recall.append(
        rouge_score.recall
    )

    rougeL_f1.append(
        rouge_score.fmeasure
    )

    # Progress display
    if (i + 1) % 500 == 0:
        print(
            f"Processed {i + 1} / {total_rows}"
        )


feature_df[
    "rougeL_precision"
] = rougeL_precision

feature_df[
    "rougeL_recall"
] = rougeL_recall

feature_df[
    "rougeL"
] = rougeL_f1


print("\nROUGE-L calculation complete.")

Processed 500 / 6000
Processed 1000 / 6000
Processed 1500 / 6000
Processed 2000 / 6000
Processed 2500 / 6000
Processed 3000 / 6000
Processed 3500 / 6000
Processed 4000 / 6000
Processed 4500 / 6000
Processed 5000 / 6000
Processed 5500 / 6000
Processed 6000 / 6000

ROUGE-L calculation complete.


In [48]:
# ============================================================
# ROUGE-L VALIDATION CHECK
# ============================================================

print("Missing ROUGE-L values:")
print(
    feature_df[
        [
            "rougeL_precision",
            "rougeL_recall",
            "rougeL"
        ]
    ].isnull().sum()
)

print("\nROUGE-L statistics:")

display(
    feature_df[
        [
            "rougeL_precision",
            "rougeL_recall",
            "rougeL"
        ]
    ]
    .describe()
    .T
    .round(4)
)

Missing ROUGE-L values:
rougeL_precision    0
rougeL_recall       0
rougeL              0
dtype: int64

ROUGE-L statistics:


,count,mean,std,min,25%,50%,75%,max
rougeL_precision,6000.0,0.2933,0.1263,0.0161,0.2041,0.2727,0.3579,1.0000
rougeL_recall,6000.0,0.3487,0.1478,0.0312,0.2444,0.3235,0.4259,1.0000
rougeL,6000.0,0.3091,0.1212,0.0215,0.2247,0.2895,0.3729,0.9815


In [50]:
# ============================================================
# MANUAL ROUGE-L SANITY CHECK
# ============================================================

display(
    feature_df[
        [
            "doc_id",
            "model",
            "human_content_coverage",
            "rougeL",
            "reference_summary",
            "candidate_summary"
        ]
    ].head(10)
)

,doc_id,model,human_content_coverage,rougeL,reference_summary,candidate_summary
0,000571afe702684d90c1d222ce70b1e1375c1016,bart,1.0,0.324324,Juan Arango escaped punishment from the refere...,Juan Arango bites Jesus Zavela in a moment of ...
1,000571afe702684d90c1d222ce70b1e1375c1016,gold,0.3,0.236842,Juan Arango escaped punishment from the refere...,Juan Arango bit Jesus Zavala's shoulder in Clu...
2,000571afe702684d90c1d222ce70b1e1375c1016,pegasus,0.3,0.318841,Juan Arango escaped punishment from the refere...,Club Tijuana lost 4-3 to Monterrey in the Mexi...
3,000571afe702684d90c1d222ce70b1e1375c1016,brio,0.8,0.376471,Juan Arango escaped punishment from the refere...,Juan Arango bites the shoulder of opponent Jes...
4,000571afe702684d90c1d222ce70b1e1375c1016,gsum,0.9,0.383838,Juan Arango escaped punishment from the refere...,Juan Arango bites Jesus Zavela in Club Tijuana...
5,000571afe702684d90c1d222ce70b1e1375c1016,simcls,0.5,0.430380,Juan Arango escaped punishment from the refere...,Club Tijuana lost 4-3 to Monterrey in the Mexi...
6,000571afe702684d90c1d222ce70b1e1375c1016,cliff,1.0,0.447059,Juan Arango escaped punishment from the refere...,Juan Arango bites Jesus Zavela in Tijuana's 4-...
7,000571afe702684d90c1d222ce70b1e1375c1016,ctrlsum,0.7,0.417582,Juan Arango escaped punishment from the refere...,Juan Arango bit the shoulder of opponent Jesus...
8,000571afe702684d90c1d222ce70b1e1375c1016,frost,0.3,0.411765,Juan Arango escaped punishment from the refere...,Juan Arango sank his teeth into the shoulder o...
9,000571afe702684d90c1d222ce70b1e1375c1016,glob,0.4,0.270270,Juan Arango escaped punishment from the refere...,Juan Arango bit Jesus Zavela in Club Tijuana's...


In [52]:
# ============================================================
# INITIAL ROUGE-L / HUMAN SCORE CHECK
# ============================================================

from scipy.stats import pearsonr, spearmanr, kendalltau


valid = feature_df[
    [
        "rougeL",
        "human_content_coverage"
    ]
].dropna()


pearson_value = pearsonr(
    valid["rougeL"],
    valid["human_content_coverage"]
).statistic


spearman_value = spearmanr(
    valid["rougeL"],
    valid["human_content_coverage"]
).statistic


kendall_value = kendalltau(
    valid["rougeL"],
    valid["human_content_coverage"]
).statistic


print(
    "ROUGE-L vs Human ACU"
)

print(
    "Pearson:",
    round(pearson_value, 4)
)

print(
    "Spearman:",
    round(spearman_value, 4)
)

print(
    "Kendall:",
    round(kendall_value, 4)
)

ROUGE-L vs Human ACU
Pearson: 0.5709
Spearman: 0.5646
Kendall: 0.4021


In [54]:
# ============================================================
# PHASE 5 — BERTSCORE
# ============================================================

!pip install -q -U bert-score


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [56]:
from bert_score import BERTScorer
import torch

print("PyTorch version:", torch.__version__)
print(
    "Device:",
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

PyTorch version: 2.10.0
Device: mps


In [58]:
# ============================================================
# CREATE BERTSCORE SCORER
# ============================================================

bertscore_scorer = BERTScorer(
    model_type="roberta-large",
    lang="en",
    rescale_with_baseline=True,
    device=(
        "cuda"
        if torch.cuda.is_available()
        else "mps"
        if torch.backends.mps.is_available()
        else "cpu"
    )
)

print("BERTScore scorer ready.")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore scorer ready.


In [60]:
candidate = feature_df.loc[
    0,
    "candidate_clean"
]

reference = feature_df.loc[
    0,
    "reference_clean"
]

P, R, F1 = bertscore_scorer.score(
    [candidate],
    [reference]
)

print("BERTScore Precision:", float(P[0]))
print("BERTScore Recall:", float(R[0]))
print("BERTScore F1:", float(F1[0]))

BERTScore Precision: 0.2747073471546173
BERTScore Recall: 0.5190006494522095
BERTScore F1: 0.3950452208518982


/opt/anaconda3/lib/python3.12/site-packages/bert_score/scorer.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self._baseline_vals = torch.from_numpy(


In [62]:
# ============================================================
# BERTSCORE — ALL 6000 RoSE SUMMARIES
# ============================================================

import numpy as np
import torch
import gc

candidates = feature_df[
    "candidate_clean"
].astype(str).tolist()

references = feature_df[
    "reference_clean"
].astype(str).tolist()

bert_precision = []
bert_recall = []
bert_f1 = []

CHUNK_SIZE = 256
BATCH_SIZE = 16

total = len(candidates)

for start in range(0, total, CHUNK_SIZE):

    end = min(
        start + CHUNK_SIZE,
        total
    )

    chunk_candidates = candidates[start:end]
    chunk_references = references[start:end]

    P, R, F1 = bertscore_scorer.score(
        chunk_candidates,
        chunk_references,
        batch_size=BATCH_SIZE,
        verbose=False
    )

    bert_precision.extend(
        P.detach().cpu().numpy().tolist()
    )

    bert_recall.extend(
        R.detach().cpu().numpy().tolist()
    )

    bert_f1.extend(
        F1.detach().cpu().numpy().tolist()
    )

    print(
        f"Processed {end} / {total}"
    )

    del P, R, F1

    gc.collect()

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


# ------------------------------------------------------------
# SAVE INTO DATAFRAME
# ------------------------------------------------------------

feature_df[
    "bertScore_precision"
] = bert_precision

feature_df[
    "bertScore_recall"
] = bert_recall

feature_df[
    "bertScoreF1"
] = bert_f1


print("\nBERTScore calculation complete.")

print(
    "Number of scores:",
    len(bert_f1)
)

Processed 256 / 6000
Processed 512 / 6000
Processed 768 / 6000
Processed 1024 / 6000
Processed 1280 / 6000
Processed 1536 / 6000
Processed 1792 / 6000
Processed 2048 / 6000
Processed 2304 / 6000
Processed 2560 / 6000
Processed 2816 / 6000
Processed 3072 / 6000
Processed 3328 / 6000
Processed 3584 / 6000
Processed 3840 / 6000
Processed 4096 / 6000
Processed 4352 / 6000
Processed 4608 / 6000
Processed 4864 / 6000
Processed 5120 / 6000
Processed 5376 / 6000
Processed 5632 / 6000
Processed 5888 / 6000
Processed 6000 / 6000

BERTScore calculation complete.
Number of scores: 6000


In [64]:
# ============================================================
# BERTSCORE VALIDATION
# ============================================================

bert_columns = [
    "bertScore_precision",
    "bertScore_recall",
    "bertScoreF1"
]

print("Missing values:")

print(
    feature_df[
        bert_columns
    ].isnull().sum()
)


print("\nBERTScore statistics:")

display(
    feature_df[
        bert_columns
    ]
    .describe()
    .T
    .round(4)
)

Missing values:
bertScore_precision    0
bertScore_recall       0
bertScoreF1            0
dtype: int64

BERTScore statistics:


,count,mean,std,min,25%,50%,75%,max
bertScore_precision,6000.0,0.3352,0.1460,-0.2046,0.2401,0.3347,0.4276,0.9568
bertScore_recall,6000.0,0.3368,0.1565,-0.3298,0.2354,0.3354,0.4344,0.9624
bertScoreF1,6000.0,0.3362,0.1354,-0.2008,0.2488,0.3367,0.4215,0.9463


In [66]:
# ============================================================
# INITIAL BERTSCORE / HUMAN ACU CHECK
# ============================================================

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau
)

valid = feature_df[
    [
        "bertScoreF1",
        "human_content_coverage"
    ]
].dropna()


bert_pearson = pearsonr(
    valid["bertScoreF1"],
    valid["human_content_coverage"]
).statistic


bert_spearman = spearmanr(
    valid["bertScoreF1"],
    valid["human_content_coverage"]
).statistic


bert_kendall = kendalltau(
    valid["bertScoreF1"],
    valid["human_content_coverage"]
).statistic


print("BERTScore F1 vs Human ACU")

print(
    "Pearson:",
    round(bert_pearson, 4)
)

print(
    "Spearman:",
    round(bert_spearman, 4)
)

print(
    "Kendall:",
    round(bert_kendall, 4)
)

BERTScore F1 vs Human ACU
Pearson: 0.6322
Spearman: 0.6221
Kendall: 0.4482


In [68]:
# ============================================================
# PHASE 6 — MOVERSCORE SETUP
# ============================================================

!pip install -q pyemd pytorch-pretrained-bert

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.12.3 requires botocore<1.34.70,>=1.34.41, but you have botocore 1.43.70 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [70]:
!git clone -q https://github.com/AIPHES/emnlp19-moverscore.git

In [72]:
from pathlib import Path

mover_file = Path(
    "emnlp19-moverscore/moverscore_v2.py"
)

text = mover_file.read_text()

# Official implementation defaults to CUDA.
# Change the module-level model device to CPU for Mac compatibility.
text = text.replace(
    "device = 'cuda'",
    "device = 'cpu'"
)

mover_file.write_text(text)

print("MoverScore patched for CPU.")

MoverScore patched for CPU.


In [74]:
import sys

sys.path.append(
    "emnlp19-moverscore"
)

from moverscore_v2 import (
    get_idf_dict,
    word_mover_score
)

print("MoverScore imported successfully.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MoverScore imported successfully.


In [78]:
# ============================================================
# PATCH MOVERSCORE FOR MODERN NUMPY
# ============================================================

from pathlib import Path
import importlib
import moverscore_v2

mover_file = Path(
    "emnlp19-moverscore/moverscore_v2.py"
)

text = mover_file.read_text()

# Replace deprecated NumPy alias
text = text.replace(
    "dtype=np.float",
    "dtype=float"
)

mover_file.write_text(text)

print("Patched np.float -> float")

# Reload the already-imported module
importlib.reload(moverscore_v2)

from moverscore_v2 import (
    get_idf_dict,
    word_mover_score
)

print("MoverScore reloaded successfully.")

Patched np.float -> float


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MoverScore reloaded successfully.


In [82]:
# ============================================================
# FIX MOVERSCORE NUMPY TYPES CORRECTLY
# ============================================================

from pathlib import Path
import importlib
import moverscore_v2

mover_file = Path(
    "emnlp19-moverscore/moverscore_v2.py"
)

text = mover_file.read_text()

# Fix accidental previous replacement
text = text.replace(
    "dtype=float32",
    "dtype=np.float32"
)

# Fix deprecated np.float ONLY where it is exactly np.float)
text = text.replace(
    "dtype=np.float)",
    "dtype=float)"
)

text = text.replace(
    "dtype=np.float,",
    "dtype=float,"
)

mover_file.write_text(text)

print("MoverScore NumPy compatibility patch completed.")

# Reload module
importlib.reload(moverscore_v2)

from moverscore_v2 import (
    get_idf_dict,
    word_mover_score
)

print("MoverScore reloaded.")

MoverScore NumPy compatibility patch completed.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MoverScore reloaded.


In [84]:
# ============================================================
# VERIFY PATCH
# ============================================================

for line_number, line in enumerate(
    mover_file.read_text().splitlines(),
    start=1
):
    if "dtype=" in line:
        print(line_number, line.strip())

53 def padding(arr, pad_token, dtype=torch.long):
56 padded = torch.ones(len(arr), max_len, dtype=dtype) * pad_token
57 mask = torch.zeros(len(arr), max_len, dtype=torch.long)
59 padded[i, :lens[i]] = torch.tensor(a, dtype=dtype)
85 padded, lens, mask = padding(arr, pad_token, dtype=torch.long)
86 padded_idf, _, _ = padding(idf_weights, pad_token, dtype=torch.float)
162 c1 = np.zeros(raw.shape[1], dtype=float)
163 c2 = np.zeros(raw.shape[1], dtype=float)
172 flow = np.array(flow, dtype=np.float32)
201 c1 = np.zeros(raw.shape[1], dtype=float)
202 c2 = np.zeros(raw.shape[1], dtype=float)
213 new_flow = np.array(flow, dtype=np.float32)


In [86]:
# ============================================================
# MOVERSCORE RETEST
# ============================================================

test_refs = [
    feature_df.loc[0, "reference_clean"]
]

test_candidates = [
    feature_df.loc[0, "candidate_clean"]
]

idf_ref_test = get_idf_dict(
    test_refs,
    nthreads=1
)

idf_candidate_test = get_idf_dict(
    test_candidates,
    nthreads=1
)

test_mover = word_mover_score(
    test_refs,
    test_candidates,
    idf_ref_test,
    idf_candidate_test,
    stop_words=[],
    n_gram=1,
    remove_subwords=True,
    batch_size=1,
    device="cpu"
)

print("MoverScore:", test_mover)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 25389.25it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MoverScore: [1.0]


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 32380.95it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [88]:
# ============================================================
# MOVERSCORE — DEVELOPMENT / HOLDOUT MASKS
# ============================================================

development_mask = (
    feature_df["doc_id"]
    .astype(str)
    .isin(development_docs)
)

holdout_mask = (
    feature_df["doc_id"]
    .astype(str)
    .isin(holdout_docs)
)

print(
    "Development summaries:",
    development_mask.sum()
)

print(
    "Holdout summaries:",
    holdout_mask.sum()
)

Development summaries: 4800
Holdout summaries: 1200


In [90]:
# ============================================================
# MOVERSCORE — FIT IDF USING DEVELOPMENT DATA ONLY
# ============================================================

development_references = (
    feature_df.loc[
        development_mask,
        "reference_clean"
    ]
    .astype(str)
    .tolist()
)

development_candidates = (
    feature_df.loc[
        development_mask,
        "candidate_clean"
    ]
    .astype(str)
    .tolist()
)

print(
    "Development references:",
    len(development_references)
)

print(
    "Development candidates:",
    len(development_candidates)
)

idf_dict_ref = get_idf_dict(
    development_references,
    nthreads=1
)

idf_dict_hyp = get_idf_dict(
    development_candidates,
    nthreads=1
)

print("Development IDF dictionaries created.")

print(
    "Reference IDF entries:",
    len(idf_dict_ref)
)

print(
    "Candidate IDF entries:",
    len(idf_dict_hyp)
)

Development references: 4800
Development candidates: 4800


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 28571.55it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 26484.21it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	

Development IDF dictionaries created.
Reference IDF entries: 6103
Candidate IDF entries: 10372


In [91]:
# ============================================================
# MOVERSCORE — CORRECT ONE-SUMMARY TEST
# ============================================================

test_refs = [
    feature_df.loc[
        0,
        "reference_clean"
    ]
]

test_candidates = [
    feature_df.loc[
        0,
        "candidate_clean"
    ]
]

test_mover = word_mover_score(
    test_refs,
    test_candidates,
    idf_dict_ref,
    idf_dict_hyp,
    stop_words=[],
    n_gram=1,
    remove_subwords=True,
    batch_size=1,
    device="cpu"
)

print(
    "MoverScore:",
    test_mover
)

MoverScore: [0.5805905597564016]


In [94]:
# ============================================================
# MOVERSCORE — SCORE ALL 6000 SUMMARIES
# IDF dictionaries remain fixed from DEVELOPMENT data only
# ============================================================

import numpy as np
import gc
import time

all_references = (
    feature_df["reference_clean"]
    .astype(str)
    .tolist()
)

all_candidates = (
    feature_df["candidate_clean"]
    .astype(str)
    .tolist()
)

mover_scores = []

CHUNK_SIZE = 100
BATCH_SIZE = 8

total = len(feature_df)

for start in range(0, total, CHUNK_SIZE):

    end = min(
        start + CHUNK_SIZE,
        total
    )

    chunk_refs = all_references[start:end]
    chunk_candidates = all_candidates[start:end]

    chunk_scores = word_mover_score(
        chunk_refs,
        chunk_candidates,
        idf_dict_ref,
        idf_dict_hyp,
        stop_words=[],
        n_gram=1,
        remove_subwords=True,
        batch_size=BATCH_SIZE,
        device="cpu"
    )

    mover_scores.extend(
        [float(x) for x in chunk_scores]
    )

    print(
        f"Processed {end} / {total}"
    )

    gc.collect()


feature_df["moverScore"] = mover_scores

print("\nMoverScore calculation complete.")
print("Number of scores:", len(mover_scores))

Processed 100 / 6000
Processed 200 / 6000
Processed 300 / 6000
Processed 400 / 6000
Processed 500 / 6000
Processed 600 / 6000
Processed 700 / 6000
Processed 800 / 6000
Processed 900 / 6000
Processed 1000 / 6000
Processed 1100 / 6000
Processed 1200 / 6000
Processed 1300 / 6000
Processed 1400 / 6000
Processed 1500 / 6000
Processed 1600 / 6000
Processed 1700 / 6000
Processed 1800 / 6000
Processed 1900 / 6000
Processed 2000 / 6000
Processed 2100 / 6000
Processed 2200 / 6000
Processed 2300 / 6000
Processed 2400 / 6000
Processed 2500 / 6000
Processed 2600 / 6000
Processed 2700 / 6000
Processed 2800 / 6000
Processed 2900 / 6000
Processed 3000 / 6000
Processed 3100 / 6000
Processed 3200 / 6000
Processed 3300 / 6000
Processed 3400 / 6000
Processed 3500 / 6000
Processed 3600 / 6000
Processed 3700 / 6000
Processed 3800 / 6000
Processed 3900 / 6000
Processed 4000 / 6000
Processed 4100 / 6000
Processed 4200 / 6000
Processed 4300 / 6000
Processed 4400 / 6000
Processed 4500 / 6000
Processed 4600 / 60

In [96]:
# ============================================================
# MOVERSCORE VALIDATION
# ============================================================

print(
    "Missing values:",
    feature_df["moverScore"].isnull().sum()
)

print(
    "Non-finite values:",
    (~np.isfinite(
        feature_df["moverScore"]
    )).sum()
)

print("\nMoverScore statistics:")

display(
    feature_df[
        ["moverScore"]
    ]
    .describe()
    .T
    .round(4)
)

Missing values: 0
Non-finite values: 0

MoverScore statistics:


,count,mean,std,min,25%,50%,75%,max
moverScore,6000.0,0.5891,0.0353,0.4717,0.5669,0.5863,0.6074,0.868


In [98]:
# ============================================================
# INITIAL MOVERSCORE / HUMAN ACU CHECK
# ============================================================

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau
)

valid = feature_df[
    [
        "moverScore",
        "human_content_coverage"
    ]
].dropna()

mover_pearson = pearsonr(
    valid["moverScore"],
    valid["human_content_coverage"]
).statistic

mover_spearman = spearmanr(
    valid["moverScore"],
    valid["human_content_coverage"]
).statistic

mover_kendall = kendalltau(
    valid["moverScore"],
    valid["human_content_coverage"]
).statistic

print("MoverScore vs Human ACU")

print(
    "Pearson:",
    round(mover_pearson, 4)
)

print(
    "Spearman:",
    round(mover_spearman, 4)
)

print(
    "Kendall:",
    round(mover_kendall, 4)
)

MoverScore vs Human ACU
Pearson: 0.5752
Spearman: 0.5781
Kendall: 0.4133


## 6. Source-Grounded NLI Factuality Features

This section derives source-grounded factuality features using a Natural Language Inference model.

Candidate-summary sentences are evaluated against the corresponding source document to estimate entailment, contradiction, and neutrality. These sentence-level predictions are aggregated into factuality features including mean entailment, mean contradiction, mean neutrality, supported-sentence ratio, contradicted-sentence ratio, maximum contradiction, and minimum entailment.

In [100]:
# ============================================================
# PHASE 7 — NLI FACTUALITY
# ============================================================

!pip install -q -U transformers sentencepiece scikit-learn


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [102]:
import torch
import numpy as np
import pandas as pd
import re

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("PyTorch:", torch.__version__)

DEVICE = (
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("NLI device:", DEVICE)

PyTorch: 2.10.0
NLI device: mps


In [106]:
# ============================================================
# SAVE CHECKPOINT BEFORE FIXING TRANSFORMERS ENVIRONMENT
# ============================================================

import numpy as np
import pandas as pd

feature_df.to_pickle(
    "ROSE_feature_df_before_NLI.pkl"
)

np.save(
    "ROSE_development_docs.npy",
    np.asarray(development_docs, dtype=str)
)

np.save(
    "ROSE_holdout_docs.npy",
    np.asarray(holdout_docs, dtype=str)
)

print("Checkpoint saved.")

print("feature_df shape:", feature_df.shape)
print("development docs:", len(development_docs))
print("holdout docs:", len(holdout_docs))

Checkpoint saved.
feature_df shape: (6000, 26)
development docs: 400
holdout docs: 100


In [108]:
%pip install -q --force-reinstall --no-cache-dir \
    "transformers==4.47.1" \
    "huggingface_hub==0.27.1" \
    "tokenizers==0.21.0" \
    "sentencepiece==0.2.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.5.2 which is incompatible.
scipy 1.13.1 requires numpy<2.3,>=1.22.4, but you have numpy 2.5.2 which is incompatible.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.5.2 which is incompatible.
s3fs 2024.6.1 requires fsspec==2024.6.1.*, but you have fsspec 2026.7.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.2 which is incompatible.
aiobotocore 2.12.3 requires botocore<1.34.70,>=1.34.41, but you have botocore 1.43.70 which is incompatible.
datasets 5.0.1 requires fsspec[http]<=2026.6.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.3 which is incompatible.
streamlit 1.37.1 requires pandas<3,>=1.3.0, but you hav

In [110]:
# ============================================================
# RESTORE RoSE EXPERIMENT AFTER ENVIRONMENT FIX
# ============================================================

import numpy as np
import pandas as pd
import torch

feature_df = pd.read_pickle(
    "ROSE_feature_df_before_NLI.pkl"
)

development_docs = np.load(
    "ROSE_development_docs.npy"
)

holdout_docs = np.load(
    "ROSE_holdout_docs.npy"
)

print("Experiment restored.")

print("feature_df:", feature_df.shape)
print("development docs:", len(development_docs))
print("holdout docs:", len(holdout_docs))

print("\nImportant columns available:")

for column in [
    "rougeL",
    "bertScoreF1",
    "moverScore",
    "human_content_coverage"
]:
    print(
        column,
        column in feature_df.columns
    )

Experiment restored.
feature_df: (6000, 26)
development docs: 400
holdout docs: 100

Important columns available:
rougeL True
bertScoreF1 True
moverScore True
human_content_coverage True


In [112]:
import transformers
import huggingface_hub
import tokenizers

print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("tokenizers:", tokenizers.__version__)

transformers: 5.3.0
huggingface_hub: 1.27.0
tokenizers: 0.22.2


In [114]:
# ============================================================
# FORCE CORRECT HF VERSIONS INTO THIS EXACT JUPYTER KERNEL
# ============================================================

import sys
import subprocess

print("Jupyter Python executable:")
print(sys.executable)

# Remove the incompatible versions from THIS kernel
subprocess.check_call([
    sys.executable, "-m", "pip",
    "uninstall", "-y",
    "transformers",
    "huggingface_hub",
    "tokenizers"
])

# Install the stable versions into THIS exact kernel
subprocess.check_call([
    sys.executable, "-m", "pip",
    "install",
    "--no-cache-dir",
    "transformers==4.47.1",
    "huggingface_hub==0.27.1",
    "tokenizers==0.21.0",
    "sentencepiece==0.2.0"
])

print("\nInstallation finished.")
print("NOW RESTART THE JUPYTER KERNEL.")

Jupyter Python executable:
/opt/anaconda3/bin/python
Found existing installation: transformers 4.47.1
Uninstalling transformers-4.47.1:
  Successfully uninstalled transformers-4.47.1
Found existing installation: huggingface-hub 0.27.1
Uninstalling huggingface-hub-0.27.1:
  Successfully uninstalled huggingface-hub-0.27.1
Found existing installation: tokenizers 0.21.0
Uninstalling tokenizers-0.21.0:
  Successfully uninstalled tokenizers-0.21.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 85.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 82.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]━━━━━ 2/3 [transformers]

Installation finished.
NOW RESTART THE JUPYTER KERNEL.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.0.1 requires fsspec[http]<=2026.6.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import transformers
import huggingface_hub
import tokenizers
import sys

print("Python:", sys.executable)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("tokenizers:", tokenizers.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Python: /opt/anaconda3/bin/python
transformers: 4.47.1
huggingface_hub: 0.27.1
tokenizers: 0.21.0


In [4]:
# ============================================================
# RESTORE RoSE EXPERIMENT
# ============================================================

import numpy as np
import pandas as pd
import torch

feature_df = pd.read_pickle(
    "ROSE_feature_df_before_NLI.pkl"
)

development_docs = np.load(
    "ROSE_development_docs.npy"
)

holdout_docs = np.load(
    "ROSE_holdout_docs.npy"
)

print("Experiment restored successfully.")

print("feature_df shape:", feature_df.shape)
print("Development documents:", len(development_docs))
print("Holdout documents:", len(holdout_docs))

print("\nCore metrics:")

for column in [
    "rougeL",
    "bertScoreF1",
    "moverScore",
    "human_content_coverage"
]:
    print(column, "✓" if column in feature_df.columns else "MISSING")

Experiment restored successfully.
feature_df shape: (6000, 26)
Development documents: 400
Holdout documents: 100

Core metrics:
rougeL ✓
bertScoreF1 ✓
moverScore ✓
human_content_coverage ✓


In [8]:
# ============================================================
# REPAIR NUMPY / SCIPY / SKLEARN BINARY COMPATIBILITY
# Python 3.12 + Transformers 4.47.1
# ============================================================

import sys
import subprocess

print("Using Python:")
print(sys.executable)

packages = [
    "numpy==1.26.4",
    "scipy==1.14.1",
    "scikit-learn==1.5.2",
    "pandas==2.2.3"
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--force-reinstall",
    "--no-cache-dir",
    *packages
])

print("\nRepair finished.")
print("RESTART THE JUPYTER KERNEL NOW.")

Using Python:
/opt/anaconda3/bin/python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 17.3 MB/s  0:00:00 20.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 18.2 MB/s  0:00:010.7 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 5.2 MB/s  0:00:02m 5.1 MB/s eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 35.2 MB/s  0:00:00101.1 MB/s eta 0:00:01
  Attempting uninstall: pytz
    Found existing installation: pytz 2024.1
    Uninstalling pytz-2024.1:
      Successfully uninstalled pytz-2024.1
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2023.3
    Uninstalling tzdata-2023.3:
      Successfully uninstalled tzdata-2023.3
  Attempting uninstall: threadpoolctl37m╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/10 [tzdata]
    Found existing installation: threadpoolctl 3.5.0━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/10 [tzdata]
    Uninstalling threadpoolctl-3.5.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.14.1 which is incompatible.
datasets 5.0.1 requires fsspec[http]<=2026.6.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.3 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 14.3.3 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [10]:
# ============================================================
# CLEAN REMAINING DEPENDENCY CONFLICTS
# ============================================================

import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--force-reinstall",
    "--no-cache-dir",

    "numpy==1.26.4",
    "scipy==1.13.1",
    "scikit-learn==1.5.2",
    "pandas==2.2.3",

    "fsspec==2026.6.0",
    "packaging==24.2",
    "rich==13.9.4"
])

print("Compatibility packages installed.")
print("Restart the Jupyter kernel.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 68.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.4/30.4 MB 9.8 MB/s  0:00:039.2 MB/s eta 0:00:01:02m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 84.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 88.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 108.1 MB/s  0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2026.3.post1
    Uninstalling pytz-2026.3.post1:
      Successfully uninstalled pytz-2026.3.post1
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2026.3
    Uninstalling tzdata-2026.3:
      Successfully uninstalled tzdata-2026.3
  Attempting uninstall: threadpoolctl
    Found existing installation: threadpoolctl 3.6.0
    Uninstalling threadpoolctl-3.6.0:
      Successfully uninstalled threadpoolctl-3.6.0;5;237m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/16 [threadpoolctl]
  Attempting uninstall: six╺━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mdit-py-plugins 0.3.0 requires markdown-it-py<3.0.0,>=1.0.0, but you have markdown-it-py 4.2.0 which is incompatible.
s3fs 2024.6.1 requires fsspec==2024.6.1.*, but you have fsspec 2026.6.0 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np
import scipy
import sklearn
import pandas as pd
import transformers
import huggingface_hub
import tokenizers
import fsspec
import packaging
import rich

print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("pandas:", pd.__version__)

print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("tokenizers:", tokenizers.__version__)

print("fsspec:", fsspec.__version__)
print("packaging:", packaging.__version__)

print("\n✓ Imports completed")

numpy: 1.26.4
scipy: 1.13.1
sklearn: 1.5.2
pandas: 2.2.3
transformers: 4.47.1
huggingface_hub: 0.27.1
tokenizers: 0.21.0
fsspec: 2026.6.0
packaging: 24.2

✓ Imports completed


In [6]:
# ============================================================
# RECOVER PANDAS-3 PICKLE WITHOUT CHANGING CURRENT ENVIRONMENT
# ============================================================

import sys
import os
import subprocess
from pathlib import Path

pickle_path = Path(
    "ROSE_feature_df_before_NLI.pkl"
).resolve()

parquet_path = Path(
    "ROSE_feature_df_before_NLI_PORTABLE.parquet"
).resolve()

recovery_env = Path(
    "rose_pickle_recovery_env"
).resolve()

print("Source pickle:")
print(pickle_path)

print("\nPortable output:")
print(parquet_path)


# ------------------------------------------------------------
# 1. Create isolated Python environment
# ------------------------------------------------------------

if not recovery_env.exists():
    subprocess.check_call([
        sys.executable,
        "-m",
        "venv",
        str(recovery_env)
    ])


# Mac / Linux Python path
recovery_python = (
    recovery_env /
    "bin" /
    "python"
)


# ------------------------------------------------------------
# 2. Install modern pandas ONLY inside recovery environment
# ------------------------------------------------------------

subprocess.check_call([
    str(recovery_python),
    "-m",
    "pip",
    "install",
    "-q",
    "pandas==3.0.4",
    "pyarrow"
])


# ------------------------------------------------------------
# 3. Read old pickle and export portable Parquet
# ------------------------------------------------------------

recovery_code = f'''
import pandas as pd

print("Recovery pandas:", pd.__version__)

df = pd.read_pickle(
    r"{pickle_path}"
)

print("Recovered shape:", df.shape)

# Convert pandas StringDtype columns to plain Python-object strings
# before exporting.
for col in df.columns:
    if isinstance(df[col].dtype, pd.StringDtype):
        df[col] = df[col].astype(object)

df.to_parquet(
    r"{parquet_path}",
    index=False
)

print("Portable Parquet saved successfully.")
'''

subprocess.check_call([
    str(recovery_python),
    "-c",
    recovery_code
])

print("\n✓ Recovery completed")

Source pickle:
/Users/sudip/Downloads/ROSE_feature_df_before_NLI.pkl

Portable output:
/Users/sudip/Downloads/ROSE_feature_df_before_NLI_PORTABLE.parquet


Reason for being yanked: Reported segfaults with datetime-related functionality

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


Recovery pandas: 3.0.4
Recovered shape: (6000, 26)
Portable Parquet saved successfully.

✓ Recovery completed


In [8]:
# ============================================================
# LOAD PORTABLE CHECKPOINT
# ============================================================

import pandas as pd
import numpy as np

feature_df = pd.read_parquet(
    "ROSE_feature_df_before_NLI_PORTABLE.parquet"
)

development_docs = np.load(
    "ROSE_development_docs.npy"
)

holdout_docs = np.load(
    "ROSE_holdout_docs.npy"
)

print("feature_df shape:", feature_df.shape)
print("Development documents:", len(development_docs))
print("Holdout documents:", len(holdout_docs))

print("\nCompleted metrics:")

for column in [
    "rougeL",
    "bertScoreF1",
    "moverScore",
    "human_content_coverage"
]:
    print(
        column,
        "✓" if column in feature_df.columns
        else "MISSING"
    )

feature_df shape: (6000, 26)
Development documents: 400
Holdout documents: 100

Completed metrics:
rougeL ✓
bertScoreF1 ✓
moverScore ✓
human_content_coverage ✓


In [10]:
# ============================================================
# PHASE 7 — NLI FACTUALITY MODEL
# ============================================================

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

NLI_MODEL_NAME = (
    "MoritzLaurer/"
    "DeBERTa-v3-base-mnli-fever-anli"
)

DEVICE = "cpu"

print("NLI device:", DEVICE)

nli_tokenizer = AutoTokenizer.from_pretrained(
    NLI_MODEL_NAME
)

nli_model = (
    AutoModelForSequenceClassification
    .from_pretrained(NLI_MODEL_NAME)
    .to(DEVICE)
)

nli_model.eval()

print("\n✓ NLI model loaded successfully")
print("Label mapping:", nli_model.config.id2label)

NLI device: cpu

✓ NLI model loaded successfully
Label mapping: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}


In [12]:
import re
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def split_sentences(text):

    text = str(text).strip()

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


def retrieve_top_source_evidence(
    source_text,
    hypothesis,
    top_k=3
):

    source_sentences = split_sentences(
        source_text
    )

    if not source_sentences:
        return ""

    corpus = (
        source_sentences +
        [hypothesis]
    )

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english"
    )

    matrix = vectorizer.fit_transform(
        corpus
    )

    similarities = cosine_similarity(
        matrix[-1],
        matrix[:-1]
    )[0]

    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    # Restore original article order
    top_indices = sorted(top_indices)

    return " ".join(
        source_sentences[i]
        for i in top_indices
    )

In [14]:
candidate_sentences = split_sentences(
    feature_df.loc[
        0,
        "candidate_clean"
    ]
)

test_hypothesis = candidate_sentences[0]

test_evidence = retrieve_top_source_evidence(
    feature_df.loc[
        0,
        "source_clean"
    ],
    test_hypothesis,
    top_k=3
)

print("CANDIDATE SENTENCE:")
print(test_hypothesis)

print("\nTOP-3 SOURCE EVIDENCE:")
print(test_evidence)

CANDIDATE SENTENCE:
Juan Arango bites Jesus Zavela in a moment of madness in Club Tijuana's 4-3 defeat by Monterrey in the Mexican league.

TOP-3 SOURCE EVIDENCE:
Club Tijuana star Juan Arango conjured memories Luis Suarez in his team's 4-3 defeat by Monterrey in the Mexican league - but it was not through prodigious scoring. The Venezuelan icon Arango sank his teeth into the shoulder of Jesus Zavela as his temper flared in the defeat. Juan Arango (left) bites the shoulder of opponent Jesus Zavela in a moment of madness .


In [16]:
# ============================================================
# ONE-SENTENCE NLI TEST
# ============================================================

inputs = nli_tokenizer(
    test_evidence,
    test_hypothesis,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {
    key: value.to(DEVICE)
    for key, value in inputs.items()
}

with torch.no_grad():

    logits = nli_model(
        **inputs
    ).logits

    probabilities = torch.softmax(
        logits,
        dim=-1
    )[0].cpu().numpy()


print("Label mapping:")
print(nli_model.config.id2label)

print("\nProbabilities:")
for index, probability in enumerate(probabilities):

    label = nli_model.config.id2label[
        index
    ]

    print(
        f"{label}: {probability:.4f}"
    )

print(
    "\nProbability sum:",
    probabilities.sum()
)

Label mapping:
{0: 'entailment', 1: 'neutral', 2: 'contradiction'}

Probabilities:
entailment: 0.9834
neutral: 0.0145
contradiction: 0.0022

Probability sum: 1.0


In [18]:
# ============================================================
# STEP 41 — BUILD NLI SENTENCE/EVIDENCE PAIRS
# ============================================================

import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def split_sentences(text):

    text = str(text).strip()

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


def retrieve_top_source_evidence(
    source_text,
    hypothesis,
    top_k=3
):

    source_sentences = split_sentences(
        source_text
    )

    if len(source_sentences) == 0:
        return ""

    corpus = source_sentences + [
        hypothesis
    ]

    try:

        vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english"
        )

        matrix = vectorizer.fit_transform(
            corpus
        )

        similarities = cosine_similarity(
            matrix[-1],
            matrix[:-1]
        )[0]

        top_indices = np.argsort(
            similarities
        )[::-1][:top_k]

        # Restore article sentence order
        top_indices = sorted(
            top_indices
        )

        evidence = " ".join(
            source_sentences[i]
            for i in top_indices
        )

        return evidence

    except ValueError:

        # Fallback for unusual empty-vocabulary cases
        return " ".join(
            source_sentences[:top_k]
        )

In [20]:
# ============================================================
# CREATE SENTENCE-LEVEL NLI PAIRS
# ============================================================

nli_pairs = []

for row_pos, row in feature_df.iterrows():

    candidate_sentences = split_sentences(
        row["candidate_clean"]
    )

    for sentence_number, hypothesis in enumerate(
        candidate_sentences
    ):

        evidence = retrieve_top_source_evidence(
            row["source_clean"],
            hypothesis,
            top_k=3
        )

        nli_pairs.append({
            "row_pos": row_pos,
            "summary_id": row["summary_id"],
            "sentence_number": sentence_number,
            "premise": evidence,
            "hypothesis": hypothesis
        })

    if (row_pos + 1) % 500 == 0:

        print(
            f"Prepared {row_pos + 1} / "
            f"{len(feature_df)} summaries"
        )


nli_pairs_df = pd.DataFrame(
    nli_pairs
)

print("\nSentence-level NLI dataset created.")

print(
    "Total candidate sentences:",
    len(nli_pairs_df)
)

print(
    "Average sentences per summary:",
    round(
        len(nli_pairs_df) /
        len(feature_df),
        3
    )
)

display(
    nli_pairs_df.head()
)

Prepared 500 / 6000 summaries
Prepared 1000 / 6000 summaries
Prepared 1500 / 6000 summaries
Prepared 2000 / 6000 summaries
Prepared 2500 / 6000 summaries
Prepared 3000 / 6000 summaries
Prepared 3500 / 6000 summaries
Prepared 4000 / 6000 summaries
Prepared 4500 / 6000 summaries
Prepared 5000 / 6000 summaries
Prepared 5500 / 6000 summaries
Prepared 6000 / 6000 summaries

Sentence-level NLI dataset created.
Total candidate sentences: 21807
Average sentences per summary: 3.635


,row_pos,summary_id,sentence_number,premise,hypothesis
0,0,0,0,Club Tijuana star Juan Arango conjured memorie...,Juan Arango bites Jesus Zavela in a moment of ...
1,0,0,1,The Venezuelan icon Arango sank his teeth into...,The Venezuelan icon sank his teeth into the sh...
2,0,0,2,He was not booked by the referee but could fac...,He was not booked by the referee but could fac...
3,0,0,3,Club Tijuana star Juan Arango conjured memorie...,Arango had earlier curled in a magnificent fre...
4,1,1,0,Club Tijuana star Juan Arango conjured memorie...,Juan Arango bit Jesus Zavala's shoulder in Clu...


In [22]:
print(
    "NLI pairs:",
    len(nli_pairs_df)
)

print(
    "Unique summaries:",
    nli_pairs_df[
        "summary_id"
    ].nunique()
)

print(
    "Missing premises:",
    nli_pairs_df[
        "premise"
    ].isna().sum()
)

print(
    "Missing hypotheses:",
    nli_pairs_df[
        "hypothesis"
    ].isna().sum()
)

print(
    "Empty premises:",
    (
        nli_pairs_df[
            "premise"
        ].str.len() == 0
    ).sum()
)

NLI pairs: 21807
Unique summaries: 6000
Missing premises: 0
Missing hypotheses: 0
Empty premises: 0


In [24]:
# ============================================================
# STEP 43 — BATCH NLI INFERENCE
# ============================================================

import torch
import numpy as np
import gc

label2id = {
    str(label).lower(): int(idx)
    for label, idx
    in nli_model.config.label2id.items()
}

print("Label mapping:")
print(label2id)

ENTAILMENT_ID = label2id[
    "entailment"
]

NEUTRAL_ID = label2id[
    "neutral"
]

CONTRADICTION_ID = label2id[
    "contradiction"
]


NLI_BATCH_SIZE = 16

entailment_scores = []
neutral_scores = []
contradiction_scores = []

total_pairs = len(
    nli_pairs_df
)

for start in range(
    0,
    total_pairs,
    NLI_BATCH_SIZE
):

    end = min(
        start + NLI_BATCH_SIZE,
        total_pairs
    )

    batch = nli_pairs_df.iloc[
        start:end
    ]

    premises = (
        batch["premise"]
        .astype(str)
        .tolist()
    )

    hypotheses = (
        batch["hypothesis"]
        .astype(str)
        .tolist()
    )

    inputs = nli_tokenizer(
        premises,
        hypotheses,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value
        in inputs.items()
    }

    with torch.no_grad():

        logits = nli_model(
            **inputs
        ).logits

        probs = torch.softmax(
            logits,
            dim=-1
        )

    probs = (
        probs
        .detach()
        .cpu()
        .numpy()
    )

    entailment_scores.extend(
        probs[
            :,
            ENTAILMENT_ID
        ].tolist()
    )

    neutral_scores.extend(
        probs[
            :,
            NEUTRAL_ID
        ].tolist()
    )

    contradiction_scores.extend(
        probs[
            :,
            CONTRADICTION_ID
        ].tolist()
    )

    if end % 500 == 0 or end == total_pairs:

        print(
            f"Processed {end} / "
            f"{total_pairs} NLI pairs"
        )

    del inputs
    del logits
    del probs

    gc.collect()


nli_pairs_df[
    "entailment"
] = entailment_scores

nli_pairs_df[
    "neutral"
] = neutral_scores

nli_pairs_df[
    "contradiction"
] = contradiction_scores


print("\n✓ NLI inference complete")

Label mapping:
{'contradiction': 2, 'entailment': 0, 'neutral': 1}
Processed 2000 / 21807 NLI pairs
Processed 4000 / 21807 NLI pairs
Processed 6000 / 21807 NLI pairs
Processed 8000 / 21807 NLI pairs
Processed 10000 / 21807 NLI pairs


KeyboardInterrupt: 

In [28]:
print("Current nli_pairs_df columns:")
print(nli_pairs_df.columns.tolist())

print("\nNumber of NLI pairs:", len(nli_pairs_df))

for name in [
    "entailment_scores",
    "neutral_scores",
    "contradiction_scores"
]:
    if name in globals():
        obj = globals()[name]
        print(name, "exists:", len(obj))
    else:
        print(name, "DOES NOT EXIST")

Current nli_pairs_df columns:
['row_pos', 'summary_id', 'sentence_number', 'premise', 'hypothesis']

Number of NLI pairs: 21807
entailment_scores exists: 10944
neutral_scores exists: 10944
contradiction_scores exists: 10944


In [33]:
# ============================================================
# STEP 43B — RESUME NLI INFERENCE FROM EXISTING PROGRESS
# ============================================================

import torch
import numpy as np
import gc

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert len(entailment_scores) == len(neutral_scores)
assert len(entailment_scores) == len(contradiction_scores)

total_pairs = len(nli_pairs_df)
start_from = len(entailment_scores)

print("Total NLI pairs:", total_pairs)
print("Already processed:", start_from)
print("Remaining:", total_pairs - start_from)

# ------------------------------------------------------------
# Get label IDs safely
# ------------------------------------------------------------

label2id = {
    str(label).lower(): int(idx)
    for label, idx
    in nli_model.config.label2id.items()
}

ENTAILMENT_ID = label2id["entailment"]
NEUTRAL_ID = label2id["neutral"]
CONTRADICTION_ID = label2id["contradiction"]

print("\nLabel IDs:")
print("Entailment:", ENTAILMENT_ID)
print("Neutral:", NEUTRAL_ID)
print("Contradiction:", CONTRADICTION_ID)

# Keep same stable batch size
NLI_BATCH_SIZE = 16


# ------------------------------------------------------------
# Resume from exactly where processing stopped
# ------------------------------------------------------------

for start in range(
    start_from,
    total_pairs,
    NLI_BATCH_SIZE
):

    end = min(
        start + NLI_BATCH_SIZE,
        total_pairs
    )

    batch = nli_pairs_df.iloc[start:end]

    premises = (
        batch["premise"]
        .astype(str)
        .tolist()
    )

    hypotheses = (
        batch["hypothesis"]
        .astype(str)
        .tolist()
    )

    inputs = nli_tokenizer(
        premises,
        hypotheses,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.inference_mode():

        logits = nli_model(
            **inputs
        ).logits

        probs = torch.softmax(
            logits,
            dim=-1
        )

    probs = (
        probs
        .detach()
        .cpu()
        .numpy()
    )

    entailment_scores.extend(
        probs[:, ENTAILMENT_ID].tolist()
    )

    neutral_scores.extend(
        probs[:, NEUTRAL_ID].tolist()
    )

    contradiction_scores.extend(
        probs[:, CONTRADICTION_ID].tolist()
    )

    # Progress output
    if end % 500 < NLI_BATCH_SIZE or end == total_pairs:

        print(
            f"Processed {end} / {total_pairs}"
        )

    del inputs
    del logits
    del probs

    gc.collect()


print("\n✓ Remaining NLI inference completed")

print(
    "Final number of scores:",
    len(entailment_scores)
)

Total NLI pairs: 21807
Already processed: 12352
Remaining: 9455

Label IDs:
Entailment: 0
Neutral: 1
Contradiction: 2
Processed 12512 / 21807
Processed 13008 / 21807
Processed 13504 / 21807
Processed 14000 / 21807
Processed 14512 / 21807
Processed 15008 / 21807
Processed 15504 / 21807
Processed 16000 / 21807
Processed 16512 / 21807
Processed 17008 / 21807
Processed 17504 / 21807
Processed 18000 / 21807
Processed 18512 / 21807
Processed 19008 / 21807
Processed 19504 / 21807
Processed 20000 / 21807
Processed 20512 / 21807
Processed 21008 / 21807
Processed 21504 / 21807
Processed 21807 / 21807

✓ Remaining NLI inference completed
Final number of scores: 21807


In [36]:
# ============================================================
# ATTACH NLI SCORES + SAVE CHECKPOINT
# ============================================================

assert len(entailment_scores) == len(nli_pairs_df)
assert len(neutral_scores) == len(nli_pairs_df)
assert len(contradiction_scores) == len(nli_pairs_df)

nli_pairs_df["entailment"] = np.asarray(
    entailment_scores,
    dtype=float
)

nli_pairs_df["neutral"] = np.asarray(
    neutral_scores,
    dtype=float
)

nli_pairs_df["contradiction"] = np.asarray(
    contradiction_scores,
    dtype=float
)

print("✓ NLI columns attached")

print(
    nli_pairs_df[
        [
            "entailment",
            "neutral",
            "contradiction"
        ]
    ].head()
)


# Permanent checkpoint
nli_pairs_df.to_parquet(
    "ROSE_NLI_SENTENCE_LEVEL.parquet",
    index=False
)

print("\n✓ NLI checkpoint saved")

✓ NLI columns attached
   entailment   neutral  contradiction
0    0.983371  0.014463       0.002165
1    0.977184  0.019419       0.003396
2    0.984451  0.014021       0.001527
3    0.959834  0.028226       0.011940
4    0.259537  0.470703       0.269759

✓ NLI checkpoint saved


In [38]:
# ============================================================
# STEP 44 — FAST VALIDATION
# ============================================================

nli_cols = [
    "entailment",
    "neutral",
    "contradiction"
]

print("Missing values:")
print(
    nli_pairs_df[
        nli_cols
    ].isna().sum()
)

nli_array = (
    nli_pairs_df[
        nli_cols
    ]
    .to_numpy(dtype=float)
)

probability_sums = (
    nli_array.sum(axis=1)
)

print("\nNLI pairs:", len(nli_array))

print("\nProbability sums:")
print("Minimum:", probability_sums.min())
print("Mean:", probability_sums.mean())
print("Maximum:", probability_sums.max())

print(
    "\nAll approximately 1:",
    np.allclose(
        probability_sums,
        1.0,
        atol=1e-5
    )
)

print("\nMean probabilities:")
print(
    "Entailment:",
    round(nli_array[:, 0].mean(), 4)
)
print(
    "Neutral:",
    round(nli_array[:, 1].mean(), 4)
)
print(
    "Contradiction:",
    round(nli_array[:, 2].mean(), 4)
)

Missing values:
entailment       0
neutral          0
contradiction    0
dtype: int64

NLI pairs: 21807

Probability sums:
Minimum: 0.9999998612329364
Mean: 1.0000000006312986
Maximum: 1.0000001423759386

All approximately 1: True

Mean probabilities:
Entailment: 0.8582
Neutral: 0.1024
Contradiction: 0.0394


In [40]:
# ============================================================
# STEP 45 — DOMINANT NLI CLASS PER CANDIDATE SENTENCE
# ============================================================

nli_probability_cols = [
    "entailment",
    "neutral",
    "contradiction"
]

nli_pairs_df["dominant_nli"] = (
    nli_pairs_df[nli_probability_cols]
    .idxmax(axis=1)
)

print("Dominant NLI class counts:")
print(
    nli_pairs_df["dominant_nli"]
    .value_counts()
)

print("\nDominant NLI class proportions:")
print(
    nli_pairs_df["dominant_nli"]
    .value_counts(normalize=True)
    .round(4)
)

Dominant NLI class counts:
dominant_nli
entailment       19311
neutral           1811
contradiction      685
Name: count, dtype: int64

Dominant NLI class proportions:
dominant_nli
entailment       0.8855
neutral          0.0830
contradiction    0.0314
Name: proportion, dtype: float64


In [42]:
# ============================================================
# STEP 46 — SUMMARY-LEVEL NLI AGGREGATION
# ============================================================

summary_nli_df = (
    nli_pairs_df
    .groupby("row_pos")
    .agg(
        nli_mean_entailment=(
            "entailment",
            "mean"
        ),

        nli_mean_contradiction=(
            "contradiction",
            "mean"
        ),

        nli_mean_neutral=(
            "neutral",
            "mean"
        ),

        nli_max_contradiction=(
            "contradiction",
            "max"
        ),

        nli_min_entailment=(
            "entailment",
            "min"
        ),

        nli_sentence_count=(
            "hypothesis",
            "count"
        )
    )
    .reset_index()
)


# Supported-sentence ratio
supported_ratio = (
    nli_pairs_df
    .assign(
        supported=lambda x:
        x["dominant_nli"].eq("entailment")
    )
    .groupby("row_pos")["supported"]
    .mean()
    .rename(
        "nli_supported_sentence_ratio"
    )
    .reset_index()
)


# Contradicted-sentence ratio
contradicted_ratio = (
    nli_pairs_df
    .assign(
        contradicted=lambda x:
        x["dominant_nli"].eq("contradiction")
    )
    .groupby("row_pos")["contradicted"]
    .mean()
    .rename(
        "nli_contradicted_sentence_ratio"
    )
    .reset_index()
)


summary_nli_df = (
    summary_nli_df
    .merge(
        supported_ratio,
        on="row_pos",
        how="left"
    )
    .merge(
        contradicted_ratio,
        on="row_pos",
        how="left"
    )
)


print(
    "Summary-level NLI shape:",
    summary_nli_df.shape
)

display(
    summary_nli_df.head()
)

Summary-level NLI shape: (6000, 9)


,row_pos,nli_mean_entailment,nli_mean_contradiction,nli_mean_neutral,nli_max_contradiction,nli_min_entailment,nli_sentence_count,nli_supported_sentence_ratio,nli_contradicted_sentence_ratio
0,0,0.976210,0.004757,0.019033,0.011940,0.959834,4,1.000000,0.000000
1,1,0.738404,0.093036,0.168560,0.269759,0.259537,3,0.666667,0.000000
2,2,0.744027,0.168527,0.087446,0.483574,0.264186,3,0.666667,0.333333
3,3,0.747471,0.244418,0.008111,0.973593,0.015813,4,0.750000,0.250000
4,4,0.765155,0.174773,0.060072,0.691693,0.104192,4,0.750000,0.250000


In [44]:
# ============================================================
# STEP 47 — ATTACH NLI FEATURES TO MAIN FEATURE TABLE
# ============================================================

feature_df = (
    feature_df
    .reset_index(drop=True)
)

feature_df["row_pos"] = (
    feature_df.index
)


feature_df = feature_df.merge(
    summary_nli_df,
    on="row_pos",
    how="left",
    validate="one_to_one"
)


nli_feature_cols = [
    "nli_mean_entailment",
    "nli_mean_contradiction",
    "nli_mean_neutral",
    "nli_max_contradiction",
    "nli_min_entailment",
    "nli_sentence_count",
    "nli_supported_sentence_ratio",
    "nli_contradicted_sentence_ratio"
]


print(
    "feature_df shape:",
    feature_df.shape
)

print("\nMissing NLI values:")

print(
    feature_df[
        nli_feature_cols
    ].isna().sum()
)

feature_df shape: (6000, 35)

Missing NLI values:
nli_mean_entailment                0
nli_mean_contradiction             0
nli_mean_neutral                   0
nli_max_contradiction              0
nli_min_entailment                 0
nli_sentence_count                 0
nli_supported_sentence_ratio       0
nli_contradicted_sentence_ratio    0
dtype: int64


In [46]:
# ============================================================
# STEP 48 — NLI FEATURE STATISTICS
# ============================================================

nli_stats = (
    feature_df[
        nli_feature_cols
    ]
    .describe()
    .T
    .round(4)
)

display(nli_stats)

,count,mean,std,min,25%,50%,75%,max
nli_mean_entailment,6000.0,0.8637,0.1612,0.0008,0.7690,0.9484,0.9799,0.9982
nli_mean_contradiction,6000.0,0.0367,0.0773,0.0006,0.0034,0.0067,0.0212,0.7259
nli_mean_neutral,6000.0,0.0995,0.1345,0.0007,0.0138,0.0327,0.1421,0.9979
nli_max_contradiction,6000.0,0.1157,0.2448,0.0007,0.0056,0.0134,0.0573,0.9992
nli_min_entailment,6000.0,0.6498,0.3866,0.0001,0.2122,0.8866,0.9642,0.9976
nli_sentence_count,6000.0,3.6345,0.9932,1.0000,3.0000,4.0000,4.0000,10.0000
nli_supported_sentence_ratio,6000.0,0.8924,0.1804,0.0000,0.7500,1.0000,1.0000,1.0000
nli_contradicted_sentence_ratio,6000.0,0.0285,0.0905,0.0000,0.0000,0.0000,0.0000,0.7500


In [48]:
# ============================================================
# NLI FEATURES vs HUMAN ACU — SANITY CHECK
# ============================================================

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau
)

check_features = [
    "nli_mean_entailment",
    "nli_mean_contradiction",
    "nli_mean_neutral",
    "nli_max_contradiction",
    "nli_min_entailment",
    "nli_supported_sentence_ratio",
    "nli_contradicted_sentence_ratio"
]

results = []

y = feature_df[
    "human_content_coverage"
].astype(float)

for col in check_features:

    x = feature_df[col].astype(float)

    results.append({
        "Feature": col,
        "Pearson": pearsonr(
            x,
            y
        ).statistic,
        "Spearman": spearmanr(
            x,
            y
        ).statistic,
        "Kendall": kendalltau(
            x,
            y
        ).statistic
    })


nli_correlation_df = (
    pd.DataFrame(results)
    .round(4)
)

display(
    nli_correlation_df
)

,Feature,Pearson,Spearman,Kendall
0,nli_mean_entailment,0.0685,0.0340,0.0232
1,nli_mean_contradiction,-0.0406,0.0151,0.0100
2,nli_mean_neutral,-0.0587,-0.0259,-0.0178
3,nli_max_contradiction,-0.0212,0.0224,0.0148
4,nli_min_entailment,0.0470,0.0230,0.0158
5,nli_supported_sentence_ratio,0.0670,0.0512,0.0400
6,nli_contradicted_sentence_ratio,-0.0337,-0.0164,-0.0133


In [50]:
# ============================================================
# STEP 49 — FINAL FEATURE EXTRACTION CHECKPOINT
# ============================================================

feature_df.to_parquet(
    "ROSE_ALL_FEATURES_COMPLETE.parquet",
    index=False
)

summary_nli_df.to_parquet(
    "ROSE_SUMMARY_NLI_FEATURES.parquet",
    index=False
)

nli_pairs_df.to_parquet(
    "ROSE_NLI_SENTENCE_LEVEL.parquet",
    index=False
)

print("✓ Main feature table saved")
print("✓ Summary NLI features saved")
print("✓ Sentence-level NLI results saved")

print(
    "\nFinal feature_df shape:",
    feature_df.shape
)

✓ Main feature table saved
✓ Summary NLI features saved
✓ Sentence-level NLI results saved

Final feature_df shape: (6000, 35)


In [52]:
# ============================================================
# STEP 50 — FREEZE FEATURE EXTRACTION + RECREATE SPLIT
# ============================================================

import numpy as np
import pandas as pd

# Save final feature extraction checkpoint
feature_df.to_parquet(
    "ROSE_ALL_FEATURES_COMPLETE.parquet",
    index=False
)

print("✓ Feature extraction checkpoint saved")


# ------------------------------------------------------------
# Recreate development / holdout masks
# ------------------------------------------------------------

development_docs = np.asarray(
    development_docs,
    dtype=str
)

holdout_docs = np.asarray(
    holdout_docs,
    dtype=str
)

doc_ids = feature_df[
    "doc_id"
].astype(str)

development_mask = doc_ids.isin(
    development_docs
)

holdout_mask = doc_ids.isin(
    holdout_docs
)


development_feature_df = (
    feature_df[
        development_mask
    ]
    .copy()
    .reset_index(drop=True)
)

holdout_feature_df = (
    feature_df[
        holdout_mask
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "\nDevelopment summaries:",
    len(development_feature_df)
)

print(
    "Development documents:",
    development_feature_df[
        "doc_id"
    ].nunique()
)

print(
    "\nHoldout summaries:",
    len(holdout_feature_df)
)

print(
    "Holdout documents:",
    holdout_feature_df[
        "doc_id"
    ].nunique()
)


# ------------------------------------------------------------
# Leakage check
# ------------------------------------------------------------

dev_doc_set = set(
    development_feature_df[
        "doc_id"
    ].astype(str)
)

holdout_doc_set = set(
    holdout_feature_df[
        "doc_id"
    ].astype(str)
)

overlap = (
    dev_doc_set &
    holdout_doc_set
)

print(
    "\nDocument overlap:",
    len(overlap)
)

assert len(overlap) == 0
assert len(development_feature_df) == 4800
assert len(holdout_feature_df) == 1200

print("\n✓ Split verified — no document leakage")

✓ Feature extraction checkpoint saved

Development summaries: 4800
Development documents: 400

Holdout summaries: 1200
Holdout documents: 100

Document overlap: 0

✓ Split verified — no document leakage


## 7. Feature Scaling and Fixed-Weight Manual Hybrid Metric

This section organises the lexical, semantic, NLI-derived factuality, and structural features used in the hybrid evaluation framework.

The raw features are scaled using parameters fitted only on the development partition and are subsequently aggregated into four quality-group scores. A fixed-weight Manual Hybrid metric is then constructed by assigning equal weight to the lexical, semantic, factuality, and structural quality dimensions.

In [54]:
# ============================================================
# STEP 51 — FEATURE INVENTORY
# ============================================================

lexical_features = [
    "rougeL"
]

semantic_features = [
    "bertScoreF1",
    "moverScore"
]

factuality_features = [
    "nli_mean_entailment",
    "nli_mean_contradiction",
    "nli_mean_neutral",
    "nli_max_contradiction",
    "nli_min_entailment",
    "nli_supported_sentence_ratio",
    "nli_contradicted_sentence_ratio"
]

structural_features = [
    "length_adequacy",
    "sentence_count_ratio",
    "redundancy_quality",
    "average_sentence_length"
]


all_model_features = (
    lexical_features
    + semantic_features
    + factuality_features
    + structural_features
)


print("Lexical:", lexical_features)
print("Semantic:", semantic_features)
print("Factuality:", factuality_features)
print("Structural:", structural_features)

print(
    "\nTotal raw modelling features:",
    len(all_model_features)
)


missing = [
    col
    for col in all_model_features
    if col not in feature_df.columns
]

print(
    "\nMissing feature columns:",
    missing
)

assert len(missing) == 0

print("\n✓ All required raw features available")

Lexical: ['rougeL']
Semantic: ['bertScoreF1', 'moverScore']
Factuality: ['nli_mean_entailment', 'nli_mean_contradiction', 'nli_mean_neutral', 'nli_max_contradiction', 'nli_min_entailment', 'nli_supported_sentence_ratio', 'nli_contradicted_sentence_ratio']
Structural: ['length_adequacy', 'sentence_count_ratio', 'redundancy_quality', 'average_sentence_length']

Total raw modelling features: 14

Missing feature columns: []

✓ All required raw features available


In [56]:
# ============================================================
# STEP 52 — DEVELOPMENT-ONLY MINMAX SCALING
# ============================================================

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib


# All 14 raw modelling features
scale_features = (
    lexical_features
    + semantic_features
    + factuality_features
    + structural_features
)

print("Number of features:", len(scale_features))
print(scale_features)


# ------------------------------------------------------------
# Fit scaler ONLY on development data
# ------------------------------------------------------------

feature_scaler = MinMaxScaler(
    feature_range=(0, 1),
    clip=True
)

development_scaled_values = (
    feature_scaler.fit_transform(
        development_feature_df[
            scale_features
        ].astype(float)
    )
)


# ------------------------------------------------------------
# Apply same frozen scaler to holdout
# ------------------------------------------------------------

holdout_scaled_values = (
    feature_scaler.transform(
        holdout_feature_df[
            scale_features
        ].astype(float)
    )
)


# ------------------------------------------------------------
# Add scaled columns
# ------------------------------------------------------------

scaled_feature_names = [
    f"{col}_scaled"
    for col in scale_features
]

development_feature_df[
    scaled_feature_names
] = development_scaled_values

holdout_feature_df[
    scaled_feature_names
] = holdout_scaled_values


print(
    "\nDevelopment:",
    development_feature_df.shape
)

print(
    "Holdout:",
    holdout_feature_df.shape
)

print("\n✓ Scaling completed")
print("✓ Scaler fitted on development only")
print("✓ Holdout transformed using frozen scaler")

Number of features: 14
['rougeL', 'bertScoreF1', 'moverScore', 'nli_mean_entailment', 'nli_mean_contradiction', 'nli_mean_neutral', 'nli_max_contradiction', 'nli_min_entailment', 'nli_supported_sentence_ratio', 'nli_contradicted_sentence_ratio', 'length_adequacy', 'sentence_count_ratio', 'redundancy_quality', 'average_sentence_length']

Development: (4800, 49)
Holdout: (1200, 49)

✓ Scaling completed
✓ Scaler fitted on development only
✓ Holdout transformed using frozen scaler


In [58]:
# ============================================================
# STEP 52B — VALIDATE SCALING
# ============================================================

print("Development scaled range:")

print(
    development_feature_df[
        scaled_feature_names
    ]
    .agg(["min", "max"])
    .round(4)
)


print("\nHoldout scaled range:")

print(
    holdout_feature_df[
        scaled_feature_names
    ]
    .agg(["min", "max"])
    .round(4)
)


print("\nMissing values:")

print(
    "Development:",
    development_feature_df[
        scaled_feature_names
    ].isna().sum().sum()
)

print(
    "Holdout:",
    holdout_feature_df[
        scaled_feature_names
    ].isna().sum().sum()
)

Development scaled range:
     rougeL_scaled  bertScoreF1_scaled  moverScore_scaled  \
min            0.0                 0.0                0.0   
max            1.0                 1.0                1.0   

     nli_mean_entailment_scaled  nli_mean_contradiction_scaled  \
min                         0.0                            0.0   
max                         1.0                            1.0   

     nli_mean_neutral_scaled  nli_max_contradiction_scaled  \
min                      0.0                           0.0   
max                      1.0                           1.0   

     nli_min_entailment_scaled  nli_supported_sentence_ratio_scaled  \
min                        0.0                                  0.0   
max                        1.0                                  1.0   

     nli_contradicted_sentence_ratio_scaled  length_adequacy_scaled  \
min                                     0.0                     0.0   
max                                     1.0     

In [60]:
joblib.dump(
    feature_scaler,
    "ROSE_feature_minmax_scaler.joblib"
)

print("✓ Development-fitted scaler saved")

✓ Development-fitted scaler saved


In [62]:
# ============================================================
# STEP 53 — CONSTRUCT FOUR QUALITY GROUP SCORES
# ============================================================

def create_group_scores(df):

    df = df.copy()

    # --------------------------------------------------------
    # 1. LEXICAL
    # Higher ROUGE-L = better
    # --------------------------------------------------------

    df["lexical_group_score"] = (
        df["rougeL_scaled"]
    )


    # --------------------------------------------------------
    # 2. SEMANTIC
    # Higher BERTScore and MoverScore = better
    # --------------------------------------------------------

    df["semantic_group_score"] = (
        df[
            [
                "bertScoreF1_scaled",
                "moverScore_scaled"
            ]
        ]
        .mean(axis=1)
    )


    # --------------------------------------------------------
    # 3. FACTUALITY
    #
    # Higher entailment/support = better
    # Lower contradiction/neutral = better
    # --------------------------------------------------------

    df[
        "quality_mean_contradiction"
    ] = (
        1.0
        -
        df[
            "nli_mean_contradiction_scaled"
        ]
    )

    df[
        "quality_mean_neutral"
    ] = (
        1.0
        -
        df[
            "nli_mean_neutral_scaled"
        ]
    )

    df[
        "quality_max_contradiction"
    ] = (
        1.0
        -
        df[
            "nli_max_contradiction_scaled"
        ]
    )

    df[
        "quality_contradicted_ratio"
    ] = (
        1.0
        -
        df[
            "nli_contradicted_sentence_ratio_scaled"
        ]
    )


    factuality_quality_features = [

        "nli_mean_entailment_scaled",

        "quality_mean_contradiction",

        "quality_mean_neutral",

        "quality_max_contradiction",

        "nli_min_entailment_scaled",

        "nli_supported_sentence_ratio_scaled",

        "quality_contradicted_ratio"
    ]


    df["factuality_group_score"] = (
        df[
            factuality_quality_features
        ]
        .mean(axis=1)
    )


    # --------------------------------------------------------
    # 4. STRUCTURAL QUALITY
    #
    # These three already have a clear higher-is-better
    # interpretation.
    # --------------------------------------------------------

    structural_quality_features = [

        "length_adequacy_scaled",

        "sentence_count_ratio_scaled",

        "redundancy_quality_scaled"
    ]


    df["structural_group_score"] = (
        df[
            structural_quality_features
        ]
        .mean(axis=1)
    )


    return df


development_feature_df = (
    create_group_scores(
        development_feature_df
    )
)

holdout_feature_df = (
    create_group_scores(
        holdout_feature_df
    )
)

print("✓ Four quality groups created")

✓ Four quality groups created


In [64]:
# ============================================================
# STEP 54 — GROUP SCORE VALIDATION
# ============================================================

group_score_cols = [

    "lexical_group_score",

    "semantic_group_score",

    "factuality_group_score",

    "structural_group_score"
]


print("DEVELOPMENT GROUP SCORES")

display(
    development_feature_df[
        group_score_cols
    ]
    .describe()
    .T
    .round(4)
)


print("\nHOLDOUT GROUP SCORES")

display(
    holdout_feature_df[
        group_score_cols
    ]
    .describe()
    .T
    .round(4)
)


print("\nMissing group scores:")

print(
    "Development:",
    development_feature_df[
        group_score_cols
    ].isna().sum().sum()
)

print(
    "Holdout:",
    holdout_feature_df[
        group_score_cols
    ].isna().sum().sum()
)

DEVELOPMENT GROUP SCORES


,count,mean,std,min,25%,50%,75%,max
lexical_group_score,4800.0,0.2962,0.1295,0.0000,0.2061,0.2754,0.3619,1.0000
semantic_group_score,4800.0,0.3829,0.1023,0.0563,0.3169,0.3784,0.4406,0.9896
factuality_group_score,4800.0,0.8722,0.1599,0.1853,0.7598,0.9645,0.9882,0.9998
structural_group_score,4800.0,0.7836,0.1139,0.2422,0.7104,0.7909,0.8661,1.0000



HOLDOUT GROUP SCORES


,count,mean,std,min,25%,50%,75%,max
lexical_group_score,1200.0,0.2900,0.1170,0.0000,0.2091,0.2727,0.3619,0.7798
semantic_group_score,1200.0,0.3793,0.0919,0.1012,0.3180,0.3787,0.4366,0.7403
factuality_group_score,1200.0,0.8734,0.1606,0.2737,0.7553,0.9694,0.9889,0.9997
structural_group_score,1200.0,0.7759,0.1162,0.2671,0.6985,0.7819,0.8602,1.0000



Missing group scores:
Development: 0
Holdout: 0


In [66]:
# ============================================================
# DEVELOPMENT-ONLY GROUP CORRELATIONS
# ============================================================

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau
)

group_results = []

y_dev = development_feature_df[
    "human_content_coverage"
].astype(float)


for col in group_score_cols:

    x = development_feature_df[
        col
    ].astype(float)

    group_results.append({

        "Group": col,

        "Pearson":
            pearsonr(
                x,
                y_dev
            ).statistic,

        "Spearman":
            spearmanr(
                x,
                y_dev
            ).statistic,

        "Kendall":
            kendalltau(
                x,
                y_dev
            ).statistic
    })


group_correlation_df = (
    pd.DataFrame(
        group_results
    )
    .round(4)
)

display(
    group_correlation_df
)

,Group,Pearson,Spearman,Kendall
0,lexical_group_score,0.5726,0.5642,0.4012
1,semantic_group_score,0.6296,0.6242,0.4491
2,factuality_group_score,0.0420,0.0153,0.0107
3,structural_group_score,-0.0211,-0.0342,-0.0235


In [68]:
development_feature_df.to_parquet(
    "ROSE_DEVELOPMENT_SCALED_FEATURES.parquet",
    index=False
)

holdout_feature_df.to_parquet(
    "ROSE_HOLDOUT_SCALED_FEATURES.parquet",
    index=False
)

print("✓ Development checkpoint saved")
print("✓ Untouched holdout checkpoint saved")

✓ Development checkpoint saved
✓ Untouched holdout checkpoint saved


In [70]:
# ============================================================
# STEP 55 — DEVELOPMENT-ONLY GROUP CORRELATIONS
# ============================================================

import pandas as pd
from scipy.stats import pearsonr, spearmanr, kendalltau


group_score_cols = [
    "lexical_group_score",
    "semantic_group_score",
    "factuality_group_score",
    "structural_group_score"
]

y_dev = (
    development_feature_df[
        "human_content_coverage"
    ]
    .astype(float)
)

group_results = []

for col in group_score_cols:

    x = (
        development_feature_df[col]
        .astype(float)
    )

    group_results.append({
        "Group": col,

        "Pearson":
            pearsonr(
                x,
                y_dev
            ).statistic,

        "Spearman":
            spearmanr(
                x,
                y_dev
            ).statistic,

        "Kendall":
            kendalltau(
                x,
                y_dev
            ).statistic
    })


group_correlation_df = (
    pd.DataFrame(group_results)
    .round(4)
)

display(group_correlation_df)

,Group,Pearson,Spearman,Kendall
0,lexical_group_score,0.5726,0.5642,0.4012
1,semantic_group_score,0.6296,0.6242,0.4491
2,factuality_group_score,0.0420,0.0153,0.0107
3,structural_group_score,-0.0211,-0.0342,-0.0235


In [72]:
# ============================================================
# STEP 56 — MANUAL HYBRID BASELINE
# Equal fixed weight for each quality dimension
# ============================================================

MANUAL_WEIGHTS = {
    "lexical_group_score": 0.25,
    "semantic_group_score": 0.25,
    "factuality_group_score": 0.25,
    "structural_group_score": 0.25
}

print("Manual Hybrid weights:")
print(MANUAL_WEIGHTS)

print(
    "\nWeight sum:",
    sum(MANUAL_WEIGHTS.values())
)


def add_manual_hybrid(df):

    df = df.copy()

    df["manual_hybrid_raw"] = (
        0.25 * df["lexical_group_score"]
        +
        0.25 * df["semantic_group_score"]
        +
        0.25 * df["factuality_group_score"]
        +
        0.25 * df["structural_group_score"]
    )

    return df


development_feature_df = add_manual_hybrid(
    development_feature_df
)

holdout_feature_df = add_manual_hybrid(
    holdout_feature_df
)

print("\n✓ Manual Hybrid created")

Manual Hybrid weights:
{'lexical_group_score': 0.25, 'semantic_group_score': 0.25, 'factuality_group_score': 0.25, 'structural_group_score': 0.25}

Weight sum: 1.0

✓ Manual Hybrid created


In [74]:
# ============================================================
# STEP 56B — MANUAL HYBRID SANITY CHECK
# ============================================================

print("Development Manual Hybrid:")

print(
    development_feature_df[
        "manual_hybrid_raw"
    ]
    .describe()
    .round(4)
)


print("\nHoldout Manual Hybrid:")

print(
    holdout_feature_df[
        "manual_hybrid_raw"
    ]
    .describe()
    .round(4)
)


print("\nMissing values:")

print(
    "Development:",
    development_feature_df[
        "manual_hybrid_raw"
    ].isna().sum()
)

print(
    "Holdout:",
    holdout_feature_df[
        "manual_hybrid_raw"
    ].isna().sum()
)

Development Manual Hybrid:
count    4800.0000
mean        0.5837
std         0.0790
min         0.2509
25%         0.5363
50%         0.5804
75%         0.6296
max         0.9740
Name: manual_hybrid_raw, dtype: float64

Holdout Manual Hybrid:
count    1200.0000
mean        0.5797
std         0.0762
min         0.3394
25%         0.5313
50%         0.5804
75%         0.6336
max         0.8556
Name: manual_hybrid_raw, dtype: float64

Missing values:
Development: 0
Holdout: 0


In [76]:
# ============================================================
# DEVELOPMENT-ONLY MANUAL HYBRID CORRELATION
# ============================================================

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau
)

manual_x = (
    development_feature_df[
        "manual_hybrid_raw"
    ].astype(float)
)

manual_y = (
    development_feature_df[
        "human_content_coverage"
    ].astype(float)
)


manual_pearson = pearsonr(
    manual_x,
    manual_y
).statistic

manual_spearman = spearmanr(
    manual_x,
    manual_y
).statistic

manual_kendall = kendalltau(
    manual_x,
    manual_y
).statistic


print("Manual Hybrid — Development")

print(
    "Pearson:",
    round(manual_pearson, 4)
)

print(
    "Spearman:",
    round(manual_spearman, 4)
)

print(
    "Kendall:",
    round(manual_kendall, 4)
)

Manual Hybrid — Development
Pearson: 0.452
Spearman: 0.4225
Kendall: 0.2955


In [78]:
development_feature_df.to_parquet(
    "ROSE_DEVELOPMENT_WITH_MANUAL_HYBRID.parquet",
    index=False
)

holdout_feature_df.to_parquet(
    "ROSE_HOLDOUT_WITH_MANUAL_HYBRID.parquet",
    index=False
)

print("✓ Manual Hybrid checkpoints saved")

✓ Manual Hybrid checkpoints saved


## 8. Learned Hybrid Model Comparison

This section compares candidate regression models for the Learned Hybrid evaluation metric using only the development partition.

Model selection is performed with document-grouped cross-validation so that summaries from the same source document remain within the same fold. Candidate models and hyperparameter settings are evaluated before freezing the final Learned Hybrid configuration for holdout testing.

In [80]:
# ============================================================
# STEP 57 — NESTED GROUPED CROSS-VALIDATION SETUP
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GroupKFold,
    GridSearchCV
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.ensemble import (
    RandomForestRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau
)


# ------------------------------------------------------------
# Modelling matrix
# ------------------------------------------------------------

model_feature_cols = [
    f"{col}_scaled"
    for col in all_model_features
]

X_dev = (
    development_feature_df[
        model_feature_cols
    ]
    .astype(float)
    .reset_index(drop=True)
)

y_dev = (
    development_feature_df[
        "human_content_coverage"
    ]
    .astype(float)
    .reset_index(drop=True)
)

groups_dev = (
    development_feature_df[
        "doc_id"
    ]
    .astype(str)
    .reset_index(drop=True)
)


print("X shape:", X_dev.shape)
print("y shape:", y_dev.shape)
print(
    "Unique documents:",
    groups_dev.nunique()
)

assert X_dev.shape == (4800, 14)
assert groups_dev.nunique() == 400

print("\n✓ Development modelling data ready")

X shape: (4800, 14)
y shape: (4800,)
Unique documents: 400

✓ Development modelling data ready


In [82]:
# ============================================================
# STEP 58 — MODEL SEARCH SPACES
# ============================================================

model_searches = {

    "Linear Regression": {
        "model": LinearRegression(),
        "params": {}
    },

    "Ridge": {
        "model": Ridge(),
        "params": {
            "alpha": [
                0.01,
                0.1,
                1.0,
                10.0,
                100.0
            ]
        }
    },

    "Lasso": {
        "model": Lasso(
            max_iter=20000,
            random_state=42
        ),
        "params": {
            "alpha": [
                0.0001,
                0.001,
                0.01,
                0.1
            ]
        }
    },

    "Elastic Net": {
        "model": ElasticNet(
            max_iter=20000,
            random_state=42
        ),
        "params": {
            "alpha": [
                0.0001,
                0.001,
                0.01,
                0.1
            ],

            "l1_ratio": [
                0.25,
                0.50,
                0.75
            ]
        }
    },

    "Random Forest": {
        "model": RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        ),
        "params": {
            "n_estimators": [300],

            "max_depth": [
                None,
                12
            ],

            "min_samples_leaf": [
                1,
                3
            ],

            "max_features": [
                "sqrt",
                0.7
            ]
        }
    }
}

print(
    "Models:",
    list(model_searches.keys())
)

Models: ['Linear Regression', 'Ridge', 'Lasso', 'Elastic Net', 'Random Forest']


In [102]:
# ============================================================
# CORRECTED NESTED CV — NO PREPROCESSING LEAKAGE
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import (
    GroupKFold,
    GridSearchCV
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau
)


# ------------------------------------------------------------
# RAW features — important
# ------------------------------------------------------------

X_dev_raw = (
    development_feature_df[
        all_model_features
    ]
    .astype(float)
    .reset_index(drop=True)
)

y_dev = (
    development_feature_df[
        "human_content_coverage"
    ]
    .astype(float)
    .reset_index(drop=True)
)

groups_dev = (
    development_feature_df[
        "doc_id"
    ]
    .astype(str)
    .reset_index(drop=True)
)


print("X shape:", X_dev_raw.shape)
print("y shape:", y_dev.shape)
print("Documents:", groups_dev.nunique())

assert X_dev_raw.shape == (4800, 14)
assert groups_dev.nunique() == 400

X shape: (4800, 14)
y shape: (4800,)
Documents: 400


In [104]:
# ============================================================
# MODELS — SCALER INSIDE EACH PIPELINE
# ============================================================

model_searches = {

    "Linear Regression": {
        "pipeline": Pipeline([
            (
                "scaler",
                MinMaxScaler(
                    feature_range=(0, 1),
                    clip=True
                )
            ),
            (
                "model",
                LinearRegression()
            )
        ]),
        "params": {}
    },

    "Ridge": {
        "pipeline": Pipeline([
            (
                "scaler",
                MinMaxScaler(
                    feature_range=(0, 1),
                    clip=True
                )
            ),
            (
                "model",
                Ridge()
            )
        ]),
        "params": {
            "model__alpha": [
                0.01,
                0.1,
                1.0,
                10.0,
                100.0
            ]
        }
    },

    "Lasso": {
        "pipeline": Pipeline([
            (
                "scaler",
                MinMaxScaler(
                    feature_range=(0, 1),
                    clip=True
                )
            ),
            (
                "model",
                Lasso(
                    max_iter=20000,
                    random_state=42
                )
            )
        ]),
        "params": {
            "model__alpha": [
                0.0001,
                0.001,
                0.01,
                0.1
            ]
        }
    },

    "Elastic Net": {
        "pipeline": Pipeline([
            (
                "scaler",
                MinMaxScaler(
                    feature_range=(0, 1),
                    clip=True
                )
            ),
            (
                "model",
                ElasticNet(
                    max_iter=20000,
                    random_state=42
                )
            )
        ]),
        "params": {
            "model__alpha": [
                0.0001,
                0.001,
                0.01,
                0.1
            ],

            "model__l1_ratio": [
                0.25,
                0.50,
                0.75
            ]
        }
    },

    "Random Forest": {
        "pipeline": Pipeline([
            (
                "scaler",
                MinMaxScaler(
                    feature_range=(0, 1),
                    clip=True
                )
            ),
            (
                "model",
                RandomForestRegressor(
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]),
        "params": {
            "model__n_estimators": [300],

            "model__max_depth": [
                None,
                12
            ],

            "model__min_samples_leaf": [
                1,
                3
            ],

            "model__max_features": [
                "sqrt",
                0.7
            ]
        }
    }
}

In [106]:
# ============================================================
# CORRECTED NESTED GROUPED CROSS-VALIDATION
# ============================================================

outer_cv = GroupKFold(
    n_splits=5
)

inner_cv = GroupKFold(
    n_splits=4
)

corrected_model_results = []

corrected_oof_predictions = {}

corrected_best_params = {}


for model_name, specification in model_searches.items():

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    oof_pred = np.full(
        len(X_dev_raw),
        np.nan
    )

    fold_parameters = []


    for fold_number, (
        train_idx,
        valid_idx
    ) in enumerate(

        outer_cv.split(
            X_dev_raw,
            y_dev,
            groups_dev
        ),

        start=1
    ):

        print(
            f"\nOuter fold {fold_number}/5"
        )

        X_train = X_dev_raw.iloc[
            train_idx
        ]

        y_train = y_dev.iloc[
            train_idx
        ]

        groups_train = groups_dev.iloc[
            train_idx
        ]

        X_valid = X_dev_raw.iloc[
            valid_idx
        ]


        search = GridSearchCV(

            estimator=
                specification["pipeline"],

            param_grid=
                specification["params"],

            scoring=
                "neg_mean_absolute_error",

            cv=inner_cv,

            n_jobs=-1,

            refit=True
        )


        search.fit(
            X_train,
            y_train,
            groups=groups_train
        )


        predictions = (
            search.best_estimator_
            .predict(X_valid)
        )


        # Keep target domain valid
        predictions = np.clip(
            predictions,
            0.0,
            1.0
        )


        oof_pred[
            valid_idx
        ] = predictions


        fold_parameters.append(
            search.best_params_
        )


        print(
            "Best params:",
            search.best_params_
        )


    assert not np.isnan(
        oof_pred
    ).any()


    result = {

        "Model":
            model_name,

        "Pearson":
            pearsonr(
                y_dev,
                oof_pred
            ).statistic,

        "Spearman":
            spearmanr(
                y_dev,
                oof_pred
            ).statistic,

        "Kendall":
            kendalltau(
                y_dev,
                oof_pred
            ).statistic,

        "MAE":
            mean_absolute_error(
                y_dev,
                oof_pred
            ),

        "RMSE":
            np.sqrt(
                mean_squared_error(
                    y_dev,
                    oof_pred
                )
            )
    }


    corrected_model_results.append(
        result
    )

    corrected_oof_predictions[
        model_name
    ] = oof_pred

    corrected_best_params[
        model_name
    ] = fold_parameters


    print("\nResults:")

    for key, value in result.items():

        if key != "Model":

            print(
                key,
                round(value, 4)
            )


Linear Regression

Outer fold 1/5
Best params: {}

Outer fold 2/5
Best params: {}

Outer fold 3/5
Best params: {}

Outer fold 4/5
Best params: {}

Outer fold 5/5
Best params: {}

Results:
Pearson 0.673
Spearman 0.6716
Kendall 0.4897
MAE 0.1378
RMSE 0.1751

Ridge

Outer fold 1/5
Best params: {'model__alpha': 1.0}

Outer fold 2/5
Best params: {'model__alpha': 1.0}

Outer fold 3/5
Best params: {'model__alpha': 1.0}

Outer fold 4/5
Best params: {'model__alpha': 1.0}

Outer fold 5/5
Best params: {'model__alpha': 1.0}

Results:
Pearson 0.6851
Spearman 0.6765
Kendall 0.4931
MAE 0.1368
RMSE 0.1724

Lasso

Outer fold 1/5
Best params: {'model__alpha': 0.0001}

Outer fold 2/5
Best params: {'model__alpha': 0.0001}

Outer fold 3/5
Best params: {'model__alpha': 0.0001}

Outer fold 4/5
Best params: {'model__alpha': 0.0001}

Outer fold 5/5
Best params: {'model__alpha': 0.0001}

Results:
Pearson 0.6852
Spearman 0.6767
Kendall 0.4934
MAE 0.1367
RMSE 0.1724

Elastic Net

Outer fold 1/5
Best params: {'mo

In [108]:
# ============================================================
# CORRECTED MODEL COMPARISON
# ============================================================

corrected_model_comparison_df = (
    pd.DataFrame(
        corrected_model_results
    )
    .sort_values(
        [
            "Pearson",
            "MAE"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


display(
    corrected_model_comparison_df
    .round(4)
)


print("\nRidge alpha selections:")

for fold, params in enumerate(
    corrected_best_params["Ridge"],
    start=1
):

    print(
        f"Fold {fold}:",
        params
    )

,Model,Pearson,Spearman,Kendall,MAE,RMSE
0,Random Forest,0.6853,0.6716,0.4903,0.1378,0.1724
1,Lasso,0.6852,0.6767,0.4934,0.1367,0.1724
2,Elastic Net,0.6852,0.6767,0.4933,0.1368,0.1724
3,Ridge,0.6851,0.6765,0.4931,0.1368,0.1724
4,Linear Regression,0.6730,0.6716,0.4897,0.1378,0.1751



Ridge alpha selections:
Fold 1: {'model__alpha': 1.0}
Fold 2: {'model__alpha': 1.0}
Fold 3: {'model__alpha': 1.0}
Fold 4: {'model__alpha': 1.0}
Fold 5: {'model__alpha': 1.0}


## 9. Final Frozen Learned Hybrid Model

Following development-only model comparison, the final Learned Hybrid configuration is frozen before evaluation on the locked holdout partition.

The final model uses a leakage-safe pipeline in which MinMax feature scaling and Ridge Regression are fitted using only the development data. The frozen pipeline is subsequently applied to the untouched holdout summaries without further model selection or tuning.

In [110]:
# ============================================================
# STEP 63 — FINAL LEAKAGE-SAFE LEARNED HYBRID
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge
import joblib


final_learned_hybrid = Pipeline([
    
    (
        "scaler",
        MinMaxScaler(
            feature_range=(0, 1),
            clip=True
        )
    ),
    
    (
        "model",
        Ridge(
            alpha=1.0
        )
    )
])


final_learned_hybrid.fit(
    X_dev_raw,
    y_dev
)


print("✓ Final Learned Hybrid fitted")

print(
    "Final Ridge alpha:",
    final_learned_hybrid[
        "model"
    ].alpha
)

print(
    "Number of features:",
    len(all_model_features)
)

✓ Final Learned Hybrid fitted
Final Ridge alpha: 1.0
Number of features: 14


In [112]:
joblib.dump(
    final_learned_hybrid,
    "ROSE_FINAL_LEARNED_HYBRID_PIPELINE.joblib"
)

print("✓ Final model pipeline saved")

✓ Final model pipeline saved


In [114]:
# ============================================================
# STEP 64 — FINAL RIDGE COEFFICIENTS
# ============================================================

final_ridge = (
    final_learned_hybrid[
        "model"
    ]
)

ridge_coefficients_df = pd.DataFrame({
    
    "Feature":
        all_model_features,
    
    "Coefficient":
        final_ridge.coef_
})


ridge_coefficients_df[
    "Absolute_Coefficient"
] = (
    ridge_coefficients_df[
        "Coefficient"
    ].abs()
)


ridge_coefficients_df = (
    ridge_coefficients_df
    .sort_values(
        "Absolute_Coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "Intercept:",
    round(
        float(
            final_ridge.intercept_
        ),
        6
    )
)

display(
    ridge_coefficients_df.round(6)
)

Intercept: -0.106941


,Feature,Coefficient,Absolute_Coefficient
0,bertScoreF1,0.839712,0.839712
1,moverScore,0.348279,0.348279
2,average_sentence_length,0.296204,0.296204
3,length_adequacy,-0.270745,0.270745
4,rougeL,0.236657,0.236657
5,nli_mean_entailment,0.092490,0.092490
6,nli_mean_neutral,-0.071075,0.071075
7,nli_contradicted_sentence_ratio,-0.062953,0.062953
8,nli_min_entailment,-0.050336,0.050336
9,redundancy_quality,0.044007,0.044007


In [116]:
# ============================================================
# STEP 65 — DEVELOPMENT-ONLY BASELINE CALIBRATION
# ============================================================

from sklearn.linear_model import LinearRegression


baseline_columns = {
    
    "ROUGE-L":
        "rougeL",
    
    "BERTScore F1":
        "bertScoreF1",
    
    "MoverScore":
        "moverScore",
    
    "Manual Hybrid":
        "manual_hybrid_raw"
}


baseline_calibrators = {}


for method_name, column_name in (
    baseline_columns.items()
):
    
    X_cal = (
        development_feature_df[
            [column_name]
        ]
        .astype(float)
    )
    
    y_cal = (
        development_feature_df[
            "human_content_coverage"
        ]
        .astype(float)
    )
    
    
    calibrator = LinearRegression()
    
    calibrator.fit(
        X_cal,
        y_cal
    )
    
    
    baseline_calibrators[
        method_name
    ] = calibrator
    
    
    print(
        "\n",
        method_name
    )
    
    print(
        "Intercept:",
        round(
            float(
                calibrator.intercept_
            ),
            6
        )
    )
    
    print(
        "Slope:",
        round(
            float(
                calibrator.coef_[0]
            ),
            6
        )
    )


 ROUGE-L
Intercept: 0.068413
Slope: 1.097801

 BERTScore F1
Intercept: 0.042517
Slope: 1.088746

 MoverScore
Intercept: -1.824891
Slope: 3.789167

 Manual Hybrid
Intercept: -0.38134
Slope: 1.354038


In [118]:
joblib.dump(
    baseline_calibrators,
    "ROSE_BASELINE_CALIBRATORS.joblib"
)

print(
    "\n✓ Baseline calibrators frozen"
)


✓ Baseline calibrators frozen


## 10. Locked Holdout Evaluation

This section applies the frozen Learned Hybrid model and the calibrated baseline metrics to the untouched RoSE holdout partition.

Performance is evaluated against the ACU-based human content-coverage scores using Pearson correlation, Spearman correlation, Kendall correlation, mean absolute error, root mean squared error, and Concordance Correlation Coefficient. The final holdout predictions are then checked and saved for subsequent analyses.

In [120]:
# ============================================================
# STEP 66 — FINAL HOLDOUT PREDICTIONS
# FIRST TRUE TEST OF THE FROZEN MODEL
# ============================================================

X_holdout_raw = (
    holdout_feature_df[
        all_model_features
    ]
    .astype(float)
)


# ------------------------------------------------------------
# Learned Hybrid
# ------------------------------------------------------------

learned_prediction = (
    final_learned_hybrid.predict(
        X_holdout_raw
    )
)

learned_prediction = np.clip(
    learned_prediction,
    0.0,
    1.0
)


holdout_feature_df[
    "learned_hybrid_prediction"
] = learned_prediction


# ------------------------------------------------------------
# Calibrated baseline predictions
# ------------------------------------------------------------

for method_name, column_name in (
    baseline_columns.items()
):
    
    calibrator = (
        baseline_calibrators[
            method_name
        ]
    )
    
    prediction = calibrator.predict(
        holdout_feature_df[
            [column_name]
        ].astype(float)
    )
    
    prediction = np.clip(
        prediction,
        0.0,
        1.0
    )
    
    
    safe_name = (
        method_name
        .lower()
        .replace("-", "")
        .replace(" ", "_")
    )
    
    
    holdout_feature_df[
        f"{safe_name}_calibrated_prediction"
    ] = prediction


print(
    "✓ Final holdout predictions generated"
)

print(
    "\nHoldout summaries:",
    len(
        holdout_feature_df
    )
)

print(
    "\nLearned prediction range:"
)

print(
    holdout_feature_df[
        "learned_hybrid_prediction"
    ].min(),
    "to",
    holdout_feature_df[
        "learned_hybrid_prediction"
    ].max()
)

✓ Final holdout predictions generated

Holdout summaries: 1200

Learned prediction range:
0.0 to 0.9424640085039041


In [122]:
holdout_feature_df.to_parquet(
    "ROSE_FINAL_HOLDOUT_PREDICTIONS.parquet",
    index=False
)

print(
    "✓ Final holdout prediction checkpoint saved"
)

✓ Final holdout prediction checkpoint saved


In [124]:
# ============================================================
# STEP 67 — FINAL ROSE HOLDOUT EVALUATION
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)


# ------------------------------------------------------------
# Concordance Correlation Coefficient
# ------------------------------------------------------------

def concordance_correlation_coefficient(y_true, y_pred):

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)

    var_true = np.var(
        y_true,
        ddof=1
    )

    var_pred = np.var(
        y_pred,
        ddof=1
    )

    covariance = np.cov(
        y_true,
        y_pred,
        ddof=1
    )[0, 1]

    ccc = (
        2.0 * covariance
        /
        (
            var_true
            +
            var_pred
            +
            (mean_true - mean_pred) ** 2
        )
    )

    return ccc


# ------------------------------------------------------------
# Human scores
# ------------------------------------------------------------

y_holdout = (
    holdout_feature_df[
        "human_content_coverage"
    ]
    .astype(float)
    .to_numpy()
)


# ------------------------------------------------------------
# Method definitions
#
# rank_col = original metric used for correlations
# pred_col = calibrated/final prediction used for errors
# ------------------------------------------------------------

final_methods = {

    "Learned Hybrid": {
        "rank_col":
            "learned_hybrid_prediction",

        "pred_col":
            "learned_hybrid_prediction"
    },

    "ROUGE-L": {
        "rank_col":
            "rougeL",

        "pred_col":
            "rougel_calibrated_prediction"
    },

    "BERTScore F1": {
        "rank_col":
            "bertScoreF1",

        "pred_col":
            "bertscore_f1_calibrated_prediction"
    },

    "MoverScore": {
        "rank_col":
            "moverScore",

        "pred_col":
            "moverscore_calibrated_prediction"
    },

    "Manual Hybrid": {
        "rank_col":
            "manual_hybrid_raw",

        "pred_col":
            "manual_hybrid_calibrated_prediction"
    }
}


# ------------------------------------------------------------
# Evaluate all methods
# ------------------------------------------------------------

final_results = []


for method_name, columns in final_methods.items():

    rank_scores = (
        holdout_feature_df[
            columns["rank_col"]
        ]
        .astype(float)
        .to_numpy()
    )

    predictions = (
        holdout_feature_df[
            columns["pred_col"]
        ]
        .astype(float)
        .to_numpy()
    )


    pearson_value = pearsonr(
        rank_scores,
        y_holdout
    ).statistic

    spearman_value = spearmanr(
        rank_scores,
        y_holdout
    ).statistic

    kendall_value = kendalltau(
        rank_scores,
        y_holdout
    ).statistic

    mae_value = mean_absolute_error(
        y_holdout,
        predictions
    )

    rmse_value = np.sqrt(
        mean_squared_error(
            y_holdout,
            predictions
        )
    )

    ccc_value = (
        concordance_correlation_coefficient(
            y_holdout,
            predictions
        )
    )


    final_results.append({

        "Method":
            method_name,

        "Pearson":
            pearson_value,

        "Spearman":
            spearman_value,

        "Kendall":
            kendall_value,

        "MAE":
            mae_value,

        "RMSE":
            rmse_value,

        "CCC":
            ccc_value
    })


final_rose_results_df = (
    pd.DataFrame(
        final_results
    )
    .sort_values(
        "Pearson",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    final_rose_results_df
    .round(4)
)

,Method,Pearson,Spearman,Kendall,MAE,RMSE,CCC
0,Learned Hybrid,0.7087,0.7029,0.5227,0.1337,0.1712,0.6313
1,BERTScore F1,0.6223,0.6080,0.4406,0.1487,0.1889,0.5287
2,MoverScore,0.5738,0.5941,0.4260,0.1539,0.1970,0.4655
3,ROUGE-L,0.5653,0.5675,0.4076,0.1588,0.1985,0.4569
4,Manual Hybrid,0.4610,0.4626,0.3229,0.1723,0.2131,0.3337


In [126]:
# ============================================================
# STEP 68 — FINAL PREDICTION SANITY CHECK
# ============================================================

prediction_cols = [
    "learned_hybrid_prediction",
    "rougel_calibrated_prediction",
    "bertscore_f1_calibrated_prediction",
    "moverscore_calibrated_prediction",
    "manual_hybrid_calibrated_prediction"
]


print("Missing final predictions:")

print(
    holdout_feature_df[
        prediction_cols
    ]
    .isna()
    .sum()
)


print("\nPrediction ranges:")

for col in prediction_cols:

    print(
        col,
        "→",
        round(
            holdout_feature_df[col].min(),
            4
        ),
        "to",
        round(
            holdout_feature_df[col].max(),
            4
        )
    )

Missing final predictions:
learned_hybrid_prediction              0
rougel_calibrated_prediction           0
bertscore_f1_calibrated_prediction     0
moverscore_calibrated_prediction       0
manual_hybrid_calibrated_prediction    0
dtype: int64

Prediction ranges:
learned_hybrid_prediction → 0.0 to 0.9425
rougel_calibrated_prediction → 0.092 to 0.9153
bertscore_f1_calibrated_prediction → 0.0 to 0.8598
moverscore_calibrated_prediction → 0.0713 to 0.9404
manual_hybrid_calibrated_prediction → 0.0782 to 0.7772


In [128]:
# ============================================================
# STEP 69 — SAVE FINAL ROSE RESULTS
# ============================================================

final_rose_results_df.to_csv(
    "ROSE_FINAL_MAIN_RESULTS.csv",
    index=False
)

holdout_feature_df.to_parquet(
    "ROSE_FINAL_HOLDOUT_EVALUATED.parquet",
    index=False
)

print("✓ Final RoSE results saved")
print("✓ Final evaluated holdout saved")

✓ Final RoSE results saved
✓ Final evaluated holdout saved


## 11. Within-Document Pairwise Ranking Evaluation

This section evaluates whether the automatic evaluation methods correctly reproduce human preferences between pairs of candidate summaries generated for the same source document.

Candidate-summary pairs are formed within each holdout document using the ACU-based human content-coverage scores as the preference target. Pairwise ranking accuracy is then calculated for the Learned Hybrid metric, ROUGE-L, BERTScore F1, MoverScore, and the Manual Hybrid metric.

In [130]:
# ============================================================
# STEP 70 — PAIRWISE RANKING ACCURACY ON FINAL ROSE HOLDOUT
# ============================================================

import numpy as np
import pandas as pd
from itertools import combinations


pairwise_methods = {
    
    "Learned Hybrid":
        "learned_hybrid_prediction",
    
    "BERTScore F1":
        "bertScoreF1",
    
    "MoverScore":
        "moverScore",
    
    "ROUGE-L":
        "rougeL",
    
    "Manual Hybrid":
        "manual_hybrid_raw"
}


pairwise_rows = []


for doc_id, group in (
    holdout_feature_df
    .groupby("doc_id")
):

    group = group.reset_index(drop=True)

    for i, j in combinations(
        range(len(group)),
        2
    ):

        human_i = float(
            group.loc[
                i,
                "human_content_coverage"
            ]
        )

        human_j = float(
            group.loc[
                j,
                "human_content_coverage"
            ]
        )


        # ----------------------------------------------
        # Exclude human ties
        # ----------------------------------------------

        if np.isclose(
            human_i,
            human_j,
            atol=1e-12
        ):
            continue


        human_direction = np.sign(
            human_i - human_j
        )


        row = {
            "doc_id": doc_id,
            "i": i,
            "j": j
        }


        for method_name, column_name in (
            pairwise_methods.items()
        ):

            metric_i = float(
                group.loc[
                    i,
                    column_name
                ]
            )

            metric_j = float(
                group.loc[
                    j,
                    column_name
                ]
            )


            metric_difference = (
                metric_i - metric_j
            )


            # Metric tie = half credit
            if np.isclose(
                metric_difference,
                0.0,
                atol=1e-12
            ):

                credit = 0.5

            else:

                metric_direction = np.sign(
                    metric_difference
                )

                credit = float(
                    metric_direction
                    ==
                    human_direction
                )


            row[
                method_name
            ] = credit


        pairwise_rows.append(
            row
        )


pairwise_df = pd.DataFrame(
    pairwise_rows
)


print(
    "Holdout documents:",
    holdout_feature_df[
        "doc_id"
    ].nunique()
)

print(
    "Total summaries:",
    len(
        holdout_feature_df
    )
)

print(
    "Valid non-tied human pairs:",
    len(
        pairwise_df
    )
)

Holdout documents: 100
Total summaries: 1200
Valid non-tied human pairs: 4856


In [132]:
# ============================================================
# STEP 70B — PAIRWISE ACCURACY TABLE
# ============================================================

pairwise_results = []


for method_name in pairwise_methods.keys():

    scores = (
        pairwise_df[
            method_name
        ]
        .astype(float)
    )


    pairwise_results.append({

        "Method":
            method_name,

        "Pairwise_Accuracy":
            scores.mean(),

        "Full_Correct":
            int(
                (scores == 1.0).sum()
            ),

        "Metric_Ties":
            int(
                (scores == 0.5).sum()
            ),

        "Incorrect":
            int(
                (scores == 0.0).sum()
            ),

        "Valid_Pairs":
            len(scores)
    })


pairwise_results_df = (
    pd.DataFrame(
        pairwise_results
    )
    .sort_values(
        "Pairwise_Accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    pairwise_results_df.round(4)
)

,Method,Pairwise_Accuracy,Full_Correct,Metric_Ties,Incorrect,Valid_Pairs
0,Learned Hybrid,0.7184,3477,23,1356,4856
1,MoverScore,0.6972,3373,25,1458,4856
2,BERTScore F1,0.6770,3276,23,1557,4856
3,ROUGE-L,0.6493,3133,40,1683,4856
4,Manual Hybrid,0.6035,2919,23,1914,4856


In [134]:
pairwise_df.to_parquet(
    "ROSE_FINAL_PAIRWISE_PAIRS.parquet",
    index=False
)

pairwise_results_df.to_csv(
    "ROSE_FINAL_PAIRWISE_RESULTS.csv",
    index=False
)

print(
    "✓ Pairwise results saved"
)

✓ Pairwise results saved


## 12. Document-Level Bootstrap Confidence Intervals and Paired Significance Testing

This section quantifies uncertainty in the within-document pairwise ranking results using document-level bootstrap resampling.

A total of 5,000 bootstrap samples are drawn at the source-document level to estimate 95% confidence intervals for pairwise ranking accuracy. Paired bootstrap differences are then calculated between the Learned Hybrid metric and each baseline, followed by Holm correction for multiple comparisons.

In [136]:
# ============================================================
# STEP 71 — DOCUMENT-LEVEL BOOTSTRAP CIs
# ============================================================

import numpy as np
import pandas as pd

N_BOOTSTRAP = 5000
RANDOM_STATE = 42

rng = np.random.default_rng(
    RANDOM_STATE
)

method_names = list(
    pairwise_methods.keys()
)

unique_holdout_docs = (
    pairwise_df["doc_id"]
    .astype(str)
    .unique()
)

print(
    "Bootstrap documents:",
    len(unique_holdout_docs)
)

assert len(unique_holdout_docs) == 100


# ------------------------------------------------------------
# Store each document's pairwise scores
# ------------------------------------------------------------

document_pairwise_scores = {}

for doc_id in unique_holdout_docs:

    doc_rows = (
        pairwise_df[
            pairwise_df["doc_id"].astype(str)
            == str(doc_id)
        ]
    )

    document_pairwise_scores[
        str(doc_id)
    ] = {
        method:
            doc_rows[method]
            .astype(float)
            .to_numpy()

        for method in method_names
    }


# ------------------------------------------------------------
# Bootstrap
# ------------------------------------------------------------

bootstrap_scores = {
    method: []
    for method in method_names
}


for bootstrap_iteration in range(
    N_BOOTSTRAP
):

    sampled_docs = rng.choice(
        unique_holdout_docs,
        size=len(unique_holdout_docs),
        replace=True
    )

    for method in method_names:

        sampled_pair_scores = []

        for doc_id in sampled_docs:

            sampled_pair_scores.append(
                document_pairwise_scores[
                    str(doc_id)
                ][method]
            )

        sampled_pair_scores = np.concatenate(
            sampled_pair_scores
        )

        bootstrap_scores[
            method
        ].append(
            sampled_pair_scores.mean()
        )


# ------------------------------------------------------------
# Confidence intervals
# ------------------------------------------------------------

bootstrap_ci_results = []

for method in method_names:

    values = np.asarray(
        bootstrap_scores[method]
    )

    bootstrap_ci_results.append({

        "Method":
            method,

        "Observed_Accuracy":
            pairwise_df[
                method
            ].mean(),

        "Bootstrap_Mean":
            values.mean(),

        "CI_Lower_95":
            np.percentile(
                values,
                2.5
            ),

        "CI_Upper_95":
            np.percentile(
                values,
                97.5
            )
    })


bootstrap_ci_df = (
    pd.DataFrame(
        bootstrap_ci_results
    )
    .sort_values(
        "Observed_Accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    bootstrap_ci_df.round(4)
)

Bootstrap documents: 100


,Method,Observed_Accuracy,Bootstrap_Mean,CI_Lower_95,CI_Upper_95
0,Learned Hybrid,0.7184,0.7184,0.6918,0.7434
1,MoverScore,0.6972,0.6972,0.6684,0.7262
2,BERTScore F1,0.6770,0.6769,0.6493,0.7042
3,ROUGE-L,0.6493,0.6492,0.6186,0.6793
4,Manual Hybrid,0.6035,0.6032,0.5724,0.6353


In [138]:
# ============================================================
# STEP 72 — PAIRED DOCUMENT-BOOTSTRAP DIFFERENCES
# ============================================================

baseline_methods = [
    "MoverScore",
    "BERTScore F1",
    "ROUGE-L",
    "Manual Hybrid"
]


learned_bootstrap = np.asarray(
    bootstrap_scores[
        "Learned Hybrid"
    ]
)


paired_results = []


for baseline in baseline_methods:

    baseline_bootstrap = np.asarray(
        bootstrap_scores[
            baseline
        ]
    )

    differences = (
        learned_bootstrap
        -
        baseline_bootstrap
    )


    observed_difference = (
        pairwise_df[
            "Learned Hybrid"
        ].mean()
        -
        pairwise_df[
            baseline
        ].mean()
    )


    ci_lower = np.percentile(
        differences,
        2.5
    )

    ci_upper = np.percentile(
        differences,
        97.5
    )


    # --------------------------------------------------------
    # Two-sided bootstrap p-value
    # --------------------------------------------------------

    proportion_le_zero = (
        np.mean(
            differences <= 0
        )
    )

    proportion_ge_zero = (
        np.mean(
            differences >= 0
        )
    )

    p_value = min(
        1.0,
        2.0
        *
        min(
            proportion_le_zero,
            proportion_ge_zero
        )
    )


    paired_results.append({

        "Comparison":
            f"Learned vs {baseline}",

        "Difference":
            observed_difference,

        "CI_Lower_95":
            ci_lower,

        "CI_Upper_95":
            ci_upper,

        "Bootstrap_p":
            p_value
    })


paired_bootstrap_df = pd.DataFrame(
    paired_results
)

display(
    paired_bootstrap_df.round(4)
)

,Comparison,Difference,CI_Lower_95,CI_Upper_95,Bootstrap_p
0,Learned vs MoverScore,0.0212,-0.0079,0.0524,0.1664
1,Learned vs BERTScore F1,0.0414,0.0181,0.0656,0.0004
2,Learned vs ROUGE-L,0.0691,0.0404,0.0995,0.0000
3,Learned vs Manual Hybrid,0.1149,0.0779,0.1529,0.0000


In [140]:
# ============================================================
# STEP 73 — HOLM MULTIPLE-COMPARISON CORRECTION
# ============================================================

from statsmodels.stats.multitest import (
    multipletests
)


raw_p_values = (
    paired_bootstrap_df[
        "Bootstrap_p"
    ].to_numpy()
)


reject, corrected_p, _, _ = multipletests(

    raw_p_values,

    alpha=0.05,

    method="holm"
)


paired_bootstrap_df[
    "Holm_p"
] = corrected_p


paired_bootstrap_df[
    "Significant_0.05"
] = reject


display(
    paired_bootstrap_df.round(4)
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


,Comparison,Difference,CI_Lower_95,CI_Upper_95,Bootstrap_p,Holm_p,Significant_0.05
0,Learned vs MoverScore,0.0212,-0.0079,0.0524,0.1664,0.1664,False
1,Learned vs BERTScore F1,0.0414,0.0181,0.0656,0.0004,0.0008,True
2,Learned vs ROUGE-L,0.0691,0.0404,0.0995,0.0000,0.0000,True
3,Learned vs Manual Hybrid,0.1149,0.0779,0.1529,0.0000,0.0000,True


In [142]:
bootstrap_ci_df.to_csv(
    "ROSE_PAIRWISE_BOOTSTRAP_CI.csv",
    index=False
)

paired_bootstrap_df.to_csv(
    "ROSE_PAIRWISE_PAIRED_SIGNIFICANCE.csv",
    index=False
)

print("✓ Bootstrap confidence intervals saved")
print("✓ Paired significance results saved")

✓ Bootstrap confidence intervals saved
✓ Paired significance results saved


## 13. Calibration and Agreement Analysis

This section evaluates prediction agreement and calibration on the locked RoSE holdout partition.

For each evaluation method, the analysis reports mean absolute error, root mean squared error, Concordance Correlation Coefficient, mean prediction bias, calibration intercept and slope, and a binned calibration-gap measure. These analyses complement the correlation-based and pairwise-ranking evaluations by examining how closely predicted scores agree with the human content-coverage scale.

In [144]:
# ============================================================
# STEP 75 — FINAL CALIBRATION / AGREEMENT ANALYSIS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from sklearn.linear_model import LinearRegression


# ------------------------------------------------------------
# CCC
# ------------------------------------------------------------

def concordance_correlation_coefficient(y_true, y_pred):

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)

    var_true = np.var(
        y_true,
        ddof=1
    )

    var_pred = np.var(
        y_pred,
        ddof=1
    )

    covariance = np.cov(
        y_true,
        y_pred,
        ddof=1
    )[0, 1]

    return (
        2 * covariance
        /
        (
            var_true
            +
            var_pred
            +
            (mean_true - mean_pred) ** 2
        )
    )


# ------------------------------------------------------------
# Mean Absolute Binned Calibration Gap
# ------------------------------------------------------------

def mean_absolute_binned_gap(
    y_true,
    y_pred,
    n_bins=5
):

    temp = pd.DataFrame({
        "human": y_true,
        "prediction": y_pred
    })

    temp["bin"] = pd.qcut(
        temp["prediction"],
        q=n_bins,
        duplicates="drop"
    )

    binned = (
        temp
        .groupby(
            "bin",
            observed=True
        )
        .agg(
            mean_prediction=(
                "prediction",
                "mean"
            ),
            mean_human=(
                "human",
                "mean"
            ),
            count=(
                "human",
                "size"
            )
        )
        .reset_index(drop=True)
    )

    binned["absolute_gap"] = (
        binned["mean_prediction"]
        -
        binned["mean_human"]
    ).abs()

    return (
        binned["absolute_gap"].mean(),
        binned
    )


# ------------------------------------------------------------
# Human scores
# ------------------------------------------------------------

y_true = (
    holdout_feature_df[
        "human_content_coverage"
    ]
    .astype(float)
    .to_numpy()
)


calibration_methods = {

    "Learned Hybrid":
        "learned_hybrid_prediction",

    "BERTScore F1":
        "bertscore_f1_calibrated_prediction",

    "MoverScore":
        "moverscore_calibrated_prediction",

    "ROUGE-L":
        "rougel_calibrated_prediction",

    "Manual Hybrid":
        "manual_hybrid_calibrated_prediction"
}


calibration_results = []

calibration_bins = {}


for method_name, prediction_col in (
    calibration_methods.items()
):

    y_pred = (
        holdout_feature_df[
            prediction_col
        ]
        .astype(float)
        .to_numpy()
    )


    # --------------------------------------------------------
    # Errors
    # --------------------------------------------------------

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    ccc = (
        concordance_correlation_coefficient(
            y_true,
            y_pred
        )
    )


    # prediction - human
    mean_bias = np.mean(
        y_pred - y_true
    )


    # --------------------------------------------------------
    # Calibration regression:
    #
    # human = intercept + slope * prediction
    #
    # Ideal:
    # intercept = 0
    # slope = 1
    # --------------------------------------------------------

    calibration_model = LinearRegression()

    calibration_model.fit(
        y_pred.reshape(-1, 1),
        y_true
    )

    intercept = float(
        calibration_model.intercept_
    )

    slope = float(
        calibration_model.coef_[0]
    )


    # --------------------------------------------------------
    # Binned calibration gap
    # --------------------------------------------------------

    binned_gap, bin_df = (
        mean_absolute_binned_gap(
            y_true,
            y_pred,
            n_bins=5
        )
    )


    calibration_bins[
        method_name
    ] = bin_df


    calibration_results.append({

        "Method":
            method_name,

        "MAE":
            mae,

        "RMSE":
            rmse,

        "CCC":
            ccc,

        "Mean_Bias":
            mean_bias,

        "Calibration_Intercept":
            intercept,

        "Calibration_Slope":
            slope,

        "Mean_Abs_Binned_Gap":
            binned_gap
    })


calibration_results_df = (
    pd.DataFrame(
        calibration_results
    )
    .sort_values(
        "MAE"
    )
    .reset_index(drop=True)
)


display(
    calibration_results_df
    .round(4)
)

,Method,MAE,RMSE,CCC,Mean_Bias,Calibration_Intercept,Calibration_Slope,Mean_Abs_Binned_Gap
0,Learned Hybrid,0.1337,0.1712,0.6313,0.0178,-0.0773,1.1468,0.0262
1,BERTScore F1,0.1487,0.1889,0.5287,0.0196,-0.0636,1.1084,0.0201
2,MoverScore,0.1539,0.1970,0.4655,0.0139,-0.0605,1.1161,0.0267
3,ROUGE-L,0.1588,0.1985,0.4569,0.0154,-0.0578,1.1053,0.0182
4,Manual Hybrid,0.1723,0.2131,0.3337,0.0164,-0.0448,1.0703,0.0178


In [146]:
# ============================================================
# STEP 76 — LEARNED HYBRID CALIBRATION BINS
# ============================================================

learned_bins_df = (
    calibration_bins[
        "Learned Hybrid"
    ]
    .copy()
)

display(
    learned_bins_df.round(4)
)

,mean_prediction,mean_human,count,absolute_gap
0,0.2003,0.1544,240,0.0459
1,0.3278,0.2875,240,0.0403
2,0.4027,0.3788,240,0.0240
3,0.4807,0.4868,240,0.0061
4,0.6134,0.6282,240,0.0149


In [148]:
# ============================================================
# STEP 77 — BLAND–ALTMAN AGREEMENT
# LEARNED HYBRID
# ============================================================

learned_pred = (
    holdout_feature_df[
        "learned_hybrid_prediction"
    ]
    .astype(float)
    .to_numpy()
)


differences = (
    learned_pred - y_true
)

mean_bias = np.mean(
    differences
)

difference_sd = np.std(
    differences,
    ddof=1
)

lower_limit = (
    mean_bias
    -
    1.96 * difference_sd
)

upper_limit = (
    mean_bias
    +
    1.96 * difference_sd
)


print("Learned Hybrid Bland–Altman")

print(
    "Mean bias:",
    round(
        mean_bias,
        4
    )
)

print(
    "Lower 95% agreement limit:",
    round(
        lower_limit,
        4
    )
)

print(
    "Upper 95% agreement limit:",
    round(
        upper_limit,
        4
    )
)

Learned Hybrid Bland–Altman
Mean bias: 0.0178
Lower 95% agreement limit: -0.316
Upper 95% agreement limit: 0.3517


In [150]:
# ============================================================
# STEP 78 — SAVE FINAL CALIBRATION RESULTS
# ============================================================

calibration_results_df.to_csv(
    "ROSE_FINAL_CALIBRATION_RESULTS.csv",
    index=False
)

learned_bins_df.to_csv(
    "ROSE_LEARNED_HYBRID_CALIBRATION_BINS.csv",
    index=False
)

paired_bootstrap_df.to_csv(
    "ROSE_FINAL_PAIRWISE_SIGNIFICANCE.csv",
    index=False
)

bootstrap_ci_df.to_csv(
    "ROSE_FINAL_PAIRWISE_BOOTSTRAP_CI.csv",
    index=False
)

print("✓ Calibration results saved")
print("✓ Calibration bins saved")
print("✓ Pairwise significance results saved")
print("✓ Bootstrap confidence intervals saved")

✓ Calibration results saved
✓ Calibration bins saved
✓ Pairwise significance results saved
✓ Bootstrap confidence intervals saved


In [152]:
# ============================================================
# STEP 79 — FINAL ROSE MASTER RESULTS TABLE
# ============================================================

rose_master_results_df = (
    final_rose_results_df
    .merge(
        pairwise_results_df[
            [
                "Method",
                "Pairwise_Accuracy"
            ]
        ],
        on="Method",
        how="left"
    )
)

rose_master_results_df = (
    rose_master_results_df[
        [
            "Method",
            "Pearson",
            "Spearman",
            "Kendall",
            "Pairwise_Accuracy",
            "MAE",
            "RMSE",
            "CCC"
        ]
    ]
    .sort_values(
        "Pearson",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    rose_master_results_df.round(4)
)

rose_master_results_df.to_csv(
    "ROSE_FINAL_MASTER_RESULTS.csv",
    index=False
)

print("✓ Final RoSE master results saved")

,Method,Pearson,Spearman,Kendall,Pairwise_Accuracy,MAE,RMSE,CCC
0,Learned Hybrid,0.7087,0.7029,0.5227,0.7184,0.1337,0.1712,0.6313
1,BERTScore F1,0.6223,0.6080,0.4406,0.6770,0.1487,0.1889,0.5287
2,MoverScore,0.5738,0.5941,0.4260,0.6972,0.1539,0.1970,0.4655
3,ROUGE-L,0.5653,0.5675,0.4076,0.6493,0.1588,0.1985,0.4569
4,Manual Hybrid,0.4610,0.4626,0.3229,0.6035,0.1723,0.2131,0.3337


✓ Final RoSE master results saved


## 14. Feature-Family Ablation Analysis

This section evaluates the contribution of the lexical, semantic, NLI-derived factuality, and structural feature families to the Learned Hybrid metric.

The completed RoSE feature dataset and the original development/holdout partition are recovered, after which a set of predefined ablation configurations is evaluated. These experiments assess the predictive value of individual feature families and combinations while preserving the original document-grouped experimental design.

In [160]:
import os
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# STEP 1: Locate and load the completed RoSE feature dataset
# ---------------------------------------------------------

possible_files = [
    "ROSE_ALL_FEATURES_COMPLETE.parquet",
    "ROSE_feature_df_before_NLI_PORTABLE.parquet",
    "ROSE_FINAL_HOLDOUT_EVALUATED.parquet",
]

print("Checking available files:\n")

for f in possible_files:
    print(f"{f}: {'FOUND' if os.path.exists(f) else 'not found'}")

# Prefer the complete pre-modelling feature table
feature_file = "ROSE_ALL_FEATURES_COMPLETE.parquet"

if not os.path.exists(feature_file):
    raise FileNotFoundError(
        "ROSE_ALL_FEATURES_COMPLETE.parquet was not found. "
        "Please show me the file names in your notebook folder."
    )

rose_ablation_df = pd.read_parquet(feature_file)

print("\nLoaded:", feature_file)
print("Shape:", rose_ablation_df.shape)

print("\nColumns:")
for i, col in enumerate(rose_ablation_df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nFirst 3 rows:")
display(rose_ablation_df.head(3))

print("\nBasic document information:")
if "doc_id" in rose_ablation_df.columns:
    print("Unique documents:", rose_ablation_df["doc_id"].nunique())

if "human_content_coverage" in rose_ablation_df.columns:
    print(
        "\nHuman target summary:\n",
        rose_ablation_df["human_content_coverage"].describe()
    )

Checking available files:

ROSE_ALL_FEATURES_COMPLETE.parquet: FOUND
ROSE_feature_df_before_NLI_PORTABLE.parquet: FOUND
ROSE_FINAL_HOLDOUT_EVALUATED.parquet: FOUND

Loaded: ROSE_ALL_FEATURES_COMPLETE.parquet
Shape: (6000, 35)

Columns:
01. summary_id
02. doc_id
03. model
04. source
05. reference_summary
06. candidate_summary
07. human_content_coverage
08. source_clean
09. reference_clean
10. candidate_clean
11. candidate_word_count
12. reference_word_count
13. length_adequacy
14. candidate_sentence_count
15. reference_sentence_count
16. sentence_count_ratio
17. repeated_bigram_rate
18. redundancy_quality
19. average_sentence_length
20. rougeL_precision
21. rougeL_recall
22. rougeL
23. bertScore_precision
24. bertScore_recall
25. bertScoreF1
26. moverScore
27. row_pos
28. nli_mean_entailment
29. nli_mean_contradiction
30. nli_mean_neutral
31. nli_max_contradiction
32. nli_min_entailment
33. nli_sentence_count
34. nli_supported_sentence_ratio
35. nli_contradicted_sentence_ratio

First 3 

,summary_id,doc_id,model,source,reference_summary,candidate_summary,human_content_coverage,source_clean,reference_clean,candidate_clean,...,moverScore,row_pos,nli_mean_entailment,nli_mean_contradiction,nli_mean_neutral,nli_max_contradiction,nli_min_entailment,nli_sentence_count,nli_supported_sentence_ratio,nli_contradicted_sentence_ratio
0,0,000571afe702684d90c1d222ce70b1e1375c1016,bart,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...,Juan Arango bites Jesus Zavela in a moment of ...,1.0,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...,Juan Arango bites Jesus Zavela in a moment of ...,...,0.580591,0,0.976210,0.004757,0.019033,0.011940,0.959834,4,1.000000,0.000000
1,1,000571afe702684d90c1d222ce70b1e1375c1016,gold,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...,Juan Arango bit Jesus Zavala's shoulder in Clu...,0.3,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...,Juan Arango bit Jesus Zavala's shoulder in Clu...,...,0.592242,1,0.738404,0.093036,0.168560,0.269759,0.259537,3,0.666667,0.000000
2,2,000571afe702684d90c1d222ce70b1e1375c1016,pegasus,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...,Club Tijuana lost 4-3 to Monterrey in the Mexi...,0.3,Club Tijuana star Juan Arango conjured memorie...,Juan Arango escaped punishment from the refere...,Club Tijuana lost 4-3 to Monterrey in the Mexi...,...,0.583281,2,0.744027,0.168527,0.087446,0.483574,0.264186,3,0.666667,0.333333



Basic document information:
Unique documents: 500

Human target summary:
 count    6000.000000
mean        0.404678
std         0.237396
min         0.000000
25%         0.230769
50%         0.400000
75%         0.571429
max         1.000000
Name: human_content_coverage, dtype: float64


In [162]:
import pandas as pd
import numpy as np
import os

# ============================================================
# STEP 2: Recover the EXACT original RoSE dev/holdout split
# ============================================================

# We already loaded the complete feature table in Step 1:
# rose_ablation_df

holdout_file = "ROSE_FINAL_HOLDOUT_EVALUATED.parquet"

if not os.path.exists(holdout_file):
    raise FileNotFoundError(
        f"{holdout_file} was not found."
    )

# Load the frozen final holdout from the original experiment
original_holdout_df = pd.read_parquet(holdout_file)

print("=" * 70)
print("ORIGINAL FINAL HOLDOUT FILE")
print("=" * 70)

print("Shape:", original_holdout_df.shape)

print("\nColumns:")
for i, col in enumerate(original_holdout_df.columns, start=1):
    print(f"{i:02d}. {col}")


# ------------------------------------------------------------
# 1. Basic required-column checks
# ------------------------------------------------------------

if "doc_id" not in original_holdout_df.columns:
    raise ValueError(
        "The final holdout file does not contain doc_id."
    )

if "doc_id" not in rose_ablation_df.columns:
    raise ValueError(
        "The full feature dataframe does not contain doc_id."
    )


# ------------------------------------------------------------
# 2. Recover exact holdout document IDs
# ------------------------------------------------------------

holdout_doc_ids = original_holdout_df["doc_id"].drop_duplicates().tolist()

print("\n" + "=" * 70)
print("RECOVERED HOLDOUT DOCUMENTS")
print("=" * 70)

print("Unique holdout documents:", len(holdout_doc_ids))


# ------------------------------------------------------------
# 3. Reconstruct development and holdout from master dataframe
# ------------------------------------------------------------

rose_holdout_df = rose_ablation_df[
    rose_ablation_df["doc_id"].isin(holdout_doc_ids)
].copy()

rose_dev_df = rose_ablation_df[
    ~rose_ablation_df["doc_id"].isin(holdout_doc_ids)
].copy()


# ------------------------------------------------------------
# 4. Verify expected sizes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RECONSTRUCTED PARTITIONS")
print("=" * 70)

print(
    f"Development: {len(rose_dev_df):,} summaries | "
    f"{rose_dev_df['doc_id'].nunique()} documents"
)

print(
    f"Holdout:     {len(rose_holdout_df):,} summaries | "
    f"{rose_holdout_df['doc_id'].nunique()} documents"
)

print(
    f"Total:       {len(rose_dev_df) + len(rose_holdout_df):,} summaries"
)


# ------------------------------------------------------------
# 5. Check document overlap
# ------------------------------------------------------------

dev_docs = set(rose_dev_df["doc_id"])
holdout_docs = set(rose_holdout_df["doc_id"])

overlap = dev_docs.intersection(holdout_docs)

print("\nDocument overlap:", len(overlap))


# ------------------------------------------------------------
# 6. Check that all 500 documents are accounted for
# ------------------------------------------------------------

all_reconstructed_docs = dev_docs.union(holdout_docs)

print(
    "Total reconstructed unique documents:",
    len(all_reconstructed_docs)
)


# ------------------------------------------------------------
# 7. Verify number of summaries per document
# ------------------------------------------------------------

dev_counts = rose_dev_df.groupby("doc_id").size()
holdout_counts = rose_holdout_df.groupby("doc_id").size()

print("\n" + "=" * 70)
print("SUMMARIES PER DOCUMENT")
print("=" * 70)

print("\nDevelopment distribution:")
print(dev_counts.value_counts().sort_index())

print("\nHoldout distribution:")
print(holdout_counts.value_counts().sort_index())


# ------------------------------------------------------------
# 8. Strong identity check against the saved final holdout
# ------------------------------------------------------------

if (
    "summary_id" in original_holdout_df.columns
    and "summary_id" in rose_holdout_df.columns
):
    saved_ids = set(original_holdout_df["summary_id"])
    reconstructed_ids = set(rose_holdout_df["summary_id"])

    missing_from_reconstruction = saved_ids - reconstructed_ids
    extra_in_reconstruction = reconstructed_ids - saved_ids

    print("\n" + "=" * 70)
    print("HOLDOUT IDENTITY CHECK")
    print("=" * 70)

    print(
        "Saved final holdout summary IDs:",
        len(saved_ids)
    )

    print(
        "Reconstructed holdout summary IDs:",
        len(reconstructed_ids)
    )

    print(
        "Missing from reconstructed holdout:",
        len(missing_from_reconstruction)
    )

    print(
        "Extra in reconstructed holdout:",
        len(extra_in_reconstruction)
    )


# ------------------------------------------------------------
# 9. Human target distributions
# ------------------------------------------------------------

target = "human_content_coverage"

print("\n" + "=" * 70)
print("HUMAN TARGET DISTRIBUTION")
print("=" * 70)

print("\nDevelopment:")
print(rose_dev_df[target].describe())

print("\nHoldout:")
print(rose_holdout_df[target].describe())


# ------------------------------------------------------------
# 10. Assertions: stop immediately if anything is wrong
# ------------------------------------------------------------

assert rose_dev_df["doc_id"].nunique() == 400, \
    "Expected 400 development documents."

assert rose_holdout_df["doc_id"].nunique() == 100, \
    "Expected 100 holdout documents."

assert len(rose_dev_df) == 4800, \
    "Expected 4,800 development summaries."

assert len(rose_holdout_df) == 1200, \
    "Expected 1,200 holdout summaries."

assert len(overlap) == 0, \
    "Development and holdout document overlap detected."

assert len(all_reconstructed_docs) == 500, \
    "Expected 500 total unique documents."

assert (dev_counts == 12).all(), \
    "Not every development document has 12 summaries."

assert (holdout_counts == 12).all(), \
    "Not every holdout document has 12 summaries."


# If summary_id exists in both:
if (
    "summary_id" in original_holdout_df.columns
    and "summary_id" in rose_holdout_df.columns
):
    assert set(original_holdout_df["summary_id"]) == \
           set(rose_holdout_df["summary_id"]), \
        "Reconstructed holdout does not exactly match final holdout."


print("\n" + "=" * 70)
print("✅ STEP 2 PASSED")
print("Exact original RoSE development/holdout split recovered.")
print("=" * 70)


# ------------------------------------------------------------
# 11. Save a permanent split manifest for reproducibility
# ------------------------------------------------------------

split_manifest = (
    rose_ablation_df[["doc_id"]]
    .drop_duplicates()
    .copy()
)

split_manifest["partition"] = np.where(
    split_manifest["doc_id"].isin(holdout_doc_ids),
    "holdout",
    "development"
)

split_manifest = split_manifest.sort_values(
    ["partition", "doc_id"]
).reset_index(drop=True)

split_manifest.to_csv(
    "ROSE_ABLATION_SPLIT_MANIFEST.csv",
    index=False
)

print(
    "\nSaved reproducibility manifest:",
    "ROSE_ABLATION_SPLIT_MANIFEST.csv"
)

ORIGINAL FINAL HOLDOUT FILE
Shape: (1200, 63)

Columns:
01. summary_id
02. doc_id
03. model
04. source
05. reference_summary
06. candidate_summary
07. human_content_coverage
08. source_clean
09. reference_clean
10. candidate_clean
11. candidate_word_count
12. reference_word_count
13. length_adequacy
14. candidate_sentence_count
15. reference_sentence_count
16. sentence_count_ratio
17. repeated_bigram_rate
18. redundancy_quality
19. average_sentence_length
20. rougeL_precision
21. rougeL_recall
22. rougeL
23. bertScore_precision
24. bertScore_recall
25. bertScoreF1
26. moverScore
27. row_pos
28. nli_mean_entailment
29. nli_mean_contradiction
30. nli_mean_neutral
31. nli_max_contradiction
32. nli_min_entailment
33. nli_sentence_count
34. nli_supported_sentence_ratio
35. nli_contradicted_sentence_ratio
36. rougeL_scaled
37. bertScoreF1_scaled
38. moverScore_scaled
39. nli_mean_entailment_scaled
40. nli_mean_contradiction_scaled
41. nli_mean_neutral_scaled
42. nli_max_contradiction_scaled


In [164]:
import numpy as np
import pandas as pd

# ============================================================
# STEP 3: Lock and validate the exact 14 raw modelling features
# ============================================================

TARGET = "human_content_coverage"
GROUP_COLUMN = "doc_id"

# ------------------------------------------------------------
# 1. Define the feature families EXACTLY
# ------------------------------------------------------------

LEXICAL_FEATURES = [
    "rougeL"
]

SEMANTIC_FEATURES = [
    "bertScoreF1",
    "moverScore"
]

FACTUALITY_FEATURES = [
    "nli_mean_entailment",
    "nli_mean_contradiction",
    "nli_mean_neutral",
    "nli_max_contradiction",
    "nli_min_entailment",
    "nli_supported_sentence_ratio",
    "nli_contradicted_sentence_ratio"
]

STRUCTURAL_FEATURES = [
    "length_adequacy",
    "sentence_count_ratio",
    "redundancy_quality",
    "average_sentence_length"
]

ALL_FEATURES = (
    LEXICAL_FEATURES
    + SEMANTIC_FEATURES
    + FACTUALITY_FEATURES
    + STRUCTURAL_FEATURES
)

print("=" * 75)
print("FROZEN FEATURE INVENTORY")
print("=" * 75)

print(f"\nLexical features:     {len(LEXICAL_FEATURES)}")
for f in LEXICAL_FEATURES:
    print("  -", f)

print(f"\nSemantic features:    {len(SEMANTIC_FEATURES)}")
for f in SEMANTIC_FEATURES:
    print("  -", f)

print(f"\nFactuality features:  {len(FACTUALITY_FEATURES)}")
for f in FACTUALITY_FEATURES:
    print("  -", f)

print(f"\nStructural features:  {len(STRUCTURAL_FEATURES)}")
for f in STRUCTURAL_FEATURES:
    print("  -", f)

print("\nTOTAL FEATURES:", len(ALL_FEATURES))


# ------------------------------------------------------------
# 2. Check for duplicate feature names
# ------------------------------------------------------------

duplicate_features = [
    f for f in ALL_FEATURES
    if ALL_FEATURES.count(f) > 1
]

duplicate_features = sorted(set(duplicate_features))

print("\n" + "=" * 75)
print("DUPLICATE FEATURE CHECK")
print("=" * 75)

print("Duplicate feature names:", duplicate_features)

assert len(duplicate_features) == 0, \
    "Duplicate feature names detected."


# ------------------------------------------------------------
# 3. Check that every feature exists
# ------------------------------------------------------------

missing_dev = [
    f for f in ALL_FEATURES
    if f not in rose_dev_df.columns
]

missing_holdout = [
    f for f in ALL_FEATURES
    if f not in rose_holdout_df.columns
]

print("\n" + "=" * 75)
print("COLUMN AVAILABILITY")
print("=" * 75)

print("Missing from development:", missing_dev)
print("Missing from holdout:", missing_holdout)

assert len(missing_dev) == 0, \
    "One or more modelling features are missing from development data."

assert len(missing_holdout) == 0, \
    "One or more modelling features are missing from holdout data."


# ------------------------------------------------------------
# 4. Confirm target and group variables exist
# ------------------------------------------------------------

assert TARGET in rose_dev_df.columns, \
    f"Target column {TARGET} not found."

assert GROUP_COLUMN in rose_dev_df.columns, \
    f"Grouping column {GROUP_COLUMN} not found."


# ------------------------------------------------------------
# 5. Create DEVELOPMENT-ONLY modelling matrices
# ------------------------------------------------------------

X_dev = rose_dev_df[ALL_FEATURES].copy()
y_dev = rose_dev_df[TARGET].astype(float).copy()
groups_dev = rose_dev_df[GROUP_COLUMN].copy()

print("\n" + "=" * 75)
print("DEVELOPMENT DESIGN MATRIX")
print("=" * 75)

print("X_dev shape:", X_dev.shape)
print("y_dev shape:", y_dev.shape)
print("Unique development documents:", groups_dev.nunique())


# ------------------------------------------------------------
# 6. Check feature dtypes
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FEATURE DATA TYPES")
print("=" * 75)

dtype_table = pd.DataFrame({
    "feature": ALL_FEATURES,
    "dtype": [str(X_dev[f].dtype) for f in ALL_FEATURES]
})

display(dtype_table)

non_numeric_features = [
    f for f in ALL_FEATURES
    if not pd.api.types.is_numeric_dtype(X_dev[f])
]

print("\nNon-numeric modelling features:", non_numeric_features)

assert len(non_numeric_features) == 0, \
    "One or more modelling features are not numeric."


# ------------------------------------------------------------
# 7. Missing-value check
# ------------------------------------------------------------

missing_table = pd.DataFrame({
    "feature": ALL_FEATURES,
    "missing_count": [
        X_dev[f].isna().sum()
        for f in ALL_FEATURES
    ],
    "missing_percent": [
        100 * X_dev[f].isna().mean()
        for f in ALL_FEATURES
    ]
})

print("\n" + "=" * 75)
print("MISSING VALUES")
print("=" * 75)

display(missing_table)

total_missing = int(X_dev.isna().sum().sum())

print("Total missing feature values:", total_missing)

assert total_missing == 0, \
    "Missing values detected in the modelling features."

assert y_dev.isna().sum() == 0, \
    "Missing values detected in the human target."


# ------------------------------------------------------------
# 8. Infinite-value check
# ------------------------------------------------------------

X_numeric = X_dev.astype(float)

finite_mask = np.isfinite(X_numeric.to_numpy())

number_nonfinite = int((~finite_mask).sum())

print("\n" + "=" * 75)
print("INFINITE / NON-FINITE VALUE CHECK")
print("=" * 75)

print("Number of non-finite feature values:", number_nonfinite)

assert number_nonfinite == 0, \
    "Infinite or non-finite feature values detected."

assert np.isfinite(y_dev.to_numpy()).all(), \
    "Non-finite human target values detected."


# ------------------------------------------------------------
# 9. Constant / zero-variance feature check
# ------------------------------------------------------------

feature_variation = pd.DataFrame({
    "feature": ALL_FEATURES,
    "unique_values": [
        X_dev[f].nunique(dropna=False)
        for f in ALL_FEATURES
    ],
    "mean": [
        X_dev[f].mean()
        for f in ALL_FEATURES
    ],
    "std": [
        X_dev[f].std()
        for f in ALL_FEATURES
    ],
    "min": [
        X_dev[f].min()
        for f in ALL_FEATURES
    ],
    "max": [
        X_dev[f].max()
        for f in ALL_FEATURES
    ]
})

print("\n" + "=" * 75)
print("FEATURE VARIATION")
print("=" * 75)

display(feature_variation.round(6))

constant_features = feature_variation.loc[
    feature_variation["unique_values"] <= 1,
    "feature"
].tolist()

print("\nConstant features:", constant_features)

assert len(constant_features) == 0, \
    "One or more modelling features have zero variance."


# ------------------------------------------------------------
# 10. Check expected bounded features
# ------------------------------------------------------------

bounded_01_features = [
    "rougeL",
    "nli_mean_entailment",
    "nli_mean_contradiction",
    "nli_mean_neutral",
    "nli_max_contradiction",
    "nli_min_entailment",
    "nli_supported_sentence_ratio",
    "nli_contradicted_sentence_ratio",
    "length_adequacy",
    "sentence_count_ratio",
    "redundancy_quality"
]

range_checks = []

for f in bounded_01_features:
    range_checks.append({
        "feature": f,
        "min": X_dev[f].min(),
        "max": X_dev[f].max(),
        "within_0_1": (
            X_dev[f].min() >= -1e-12
            and X_dev[f].max() <= 1 + 1e-12
        )
    })

range_check_df = pd.DataFrame(range_checks)

print("\n" + "=" * 75)
print("EXPECTED [0,1] RANGE CHECK")
print("=" * 75)

display(range_check_df.round(6))

assert range_check_df["within_0_1"].all(), \
    "A feature expected in [0,1] is outside its expected range."


# ------------------------------------------------------------
# 11. Inspect semantic and sentence-length ranges separately
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SPECIAL FEATURE RANGES")
print("=" * 75)

for f in [
    "bertScoreF1",
    "moverScore",
    "average_sentence_length"
]:
    print(
        f"{f:30s} "
        f"min={X_dev[f].min():.6f} | "
        f"max={X_dev[f].max():.6f} | "
        f"mean={X_dev[f].mean():.6f}"
    )


# ------------------------------------------------------------
# 12. Confirm excluded related columns are NOT in model
# ------------------------------------------------------------

EXCLUDED_DIAGNOSTIC_COLUMNS = [
    "rougeL_precision",
    "rougeL_recall",
    "bertScore_precision",
    "bertScore_recall",
    "candidate_word_count",
    "reference_word_count",
    "candidate_sentence_count",
    "reference_sentence_count",
    "repeated_bigram_rate",
    "nli_sentence_count"
]

accidentally_included = [
    f for f in EXCLUDED_DIAGNOSTIC_COLUMNS
    if f in ALL_FEATURES
]

print("\n" + "=" * 75)
print("EXCLUDED-COLUMN CHECK")
print("=" * 75)

print(
    "Diagnostic/additional columns accidentally included:",
    accidentally_included
)

assert len(accidentally_included) == 0, \
    "An excluded diagnostic feature was accidentally included."


# ------------------------------------------------------------
# 13. Target sanity check
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("TARGET SANITY CHECK")
print("=" * 75)

print(y_dev.describe())

print(
    "\nTarget within [0,1]:",
    bool(
        (y_dev.min() >= 0)
        and
        (y_dev.max() <= 1)
    )
)

assert y_dev.min() >= 0 and y_dev.max() <= 1, \
    "Human target is outside expected [0,1] range."


# ------------------------------------------------------------
# 14. Save feature manifest
# ------------------------------------------------------------

feature_manifest_rows = []

for category, features in {
    "Lexical": LEXICAL_FEATURES,
    "Semantic": SEMANTIC_FEATURES,
    "Factuality": FACTUALITY_FEATURES,
    "Structural": STRUCTURAL_FEATURES
}.items():

    for f in features:
        feature_manifest_rows.append({
            "category": category,
            "feature": f
        })

feature_manifest = pd.DataFrame(feature_manifest_rows)

feature_manifest.to_csv(
    "ROSE_ABLATION_FEATURE_MANIFEST.csv",
    index=False
)

print("\nSaved:")
print("ROSE_ABLATION_FEATURE_MANIFEST.csv")


# ------------------------------------------------------------
# 15. Final assertions
# ------------------------------------------------------------

assert len(ALL_FEATURES) == 14, \
    "Expected exactly 14 modelling features."

assert X_dev.shape == (4800, 14), \
    "Expected development design matrix shape (4800, 14)."

assert groups_dev.nunique() == 400, \
    "Expected 400 development documents."

print("\n" + "=" * 75)
print("✅ STEP 3 PASSED")
print("The exact 14-feature development design matrix is valid.")
print("No model has been fitted yet.")
print("=" * 75)

FROZEN FEATURE INVENTORY

Lexical features:     1
  - rougeL

Semantic features:    2
  - bertScoreF1
  - moverScore

Factuality features:  7
  - nli_mean_entailment
  - nli_mean_contradiction
  - nli_mean_neutral
  - nli_max_contradiction
  - nli_min_entailment
  - nli_supported_sentence_ratio
  - nli_contradicted_sentence_ratio

Structural features:  4
  - length_adequacy
  - sentence_count_ratio
  - redundancy_quality
  - average_sentence_length

TOTAL FEATURES: 14

DUPLICATE FEATURE CHECK
Duplicate feature names: []

COLUMN AVAILABILITY
Missing from development: []
Missing from holdout: []

DEVELOPMENT DESIGN MATRIX
X_dev shape: (4800, 14)
y_dev shape: (4800,)
Unique development documents: 400

FEATURE DATA TYPES


,feature,dtype
0,rougeL,float64
1,bertScoreF1,float64
2,moverScore,float64
3,nli_mean_entailment,float64
4,nli_mean_contradiction,float64
5,nli_mean_neutral,float64
6,nli_max_contradiction,float64
7,nli_min_entailment,float64
8,nli_supported_sentence_ratio,float64
9,nli_contradicted_sentence_ratio,float64



Non-numeric modelling features: []

MISSING VALUES


,feature,missing_count,missing_percent
0,rougeL,0,0.0
1,bertScoreF1,0,0.0
2,moverScore,0,0.0
3,nli_mean_entailment,0,0.0
4,nli_mean_contradiction,0,0.0
5,nli_mean_neutral,0,0.0
6,nli_max_contradiction,0,0.0
7,nli_min_entailment,0,0.0
8,nli_supported_sentence_ratio,0,0.0
9,nli_contradicted_sentence_ratio,0,0.0


Total missing feature values: 0

INFINITE / NON-FINITE VALUE CHECK
Number of non-finite feature values: 0

FEATURE VARIATION


,feature,unique_values,mean,std,min,max
0,rougeL,2344,0.310301,0.123467,0.027778,0.981481
1,bertScoreF1,4564,0.336667,0.138185,-0.200838,0.946296
2,moverScore,4568,0.589563,0.035972,0.471747,0.868020
3,nli_mean_entailment,4570,0.863295,0.160639,0.000773,0.998226
4,nli_mean_contradiction,4571,0.036781,0.077957,0.000633,0.725900
5,nli_mean_neutral,4571,0.099924,0.133698,0.000765,0.997675
6,nli_max_contradiction,3891,0.115346,0.244740,0.000948,0.999179
7,nli_min_entailment,3923,0.649228,0.385403,0.000093,0.997563
8,nli_supported_sentence_ratio,19,0.892375,0.179944,0.000000,1.000000
9,nli_contradicted_sentence_ratio,13,0.028906,0.091479,0.000000,0.750000



Constant features: []

EXPECTED [0,1] RANGE CHECK


,feature,min,max,within_0_1
0,rougeL,0.027778,0.981481,True
1,nli_mean_entailment,0.000773,0.998226,True
2,nli_mean_contradiction,0.000633,0.725900,True
3,nli_mean_neutral,0.000765,0.997675,True
4,nli_max_contradiction,0.000948,0.999179,True
5,nli_min_entailment,0.000093,0.997563,True
6,nli_supported_sentence_ratio,0.000000,1.000000,True
7,nli_contradicted_sentence_ratio,0.000000,0.750000,True
8,length_adequacy,0.113636,1.000000,True
9,sentence_count_ratio,0.200000,1.000000,True



SPECIAL FEATURE RANGES
bertScoreF1                    min=-0.200838 | max=0.946296 | mean=0.336667
moverScore                     min=0.471747 | max=0.868020 | mean=0.589563
average_sentence_length        min=4.666667 | max=59.000000 | mean=17.195760

EXCLUDED-COLUMN CHECK
Diagnostic/additional columns accidentally included: []

TARGET SANITY CHECK
count    4800.000000
mean        0.409063
std         0.236702
min         0.000000
25%         0.235294
50%         0.400000
75%         0.571429
max         1.000000
Name: human_content_coverage, dtype: float64

Target within [0,1]: True

Saved:
ROSE_ABLATION_FEATURE_MANIFEST.csv

✅ STEP 3 PASSED
The exact 14-feature development design matrix is valid.
No model has been fitted yet.


In [166]:
# ============================================================
# STEP 4: Define and freeze the ablation configurations
# ============================================================

# ------------------------------------------------------------
# 1. Define ablation sets
# ------------------------------------------------------------

ABLATION_SETS = {

    # --------------------------------------------------------
    # Main reference model
    # --------------------------------------------------------
    "Full": ALL_FEATURES.copy(),

    # --------------------------------------------------------
    # Individual feature families
    # --------------------------------------------------------
    "Lexical only": (
        LEXICAL_FEATURES.copy()
    ),

    "Semantic only": (
        SEMANTIC_FEATURES.copy()
    ),

    "Factuality only": (
        FACTUALITY_FEATURES.copy()
    ),

    "Structural only": (
        STRUCTURAL_FEATURES.copy()
    ),

    # --------------------------------------------------------
    # Progressive combinations
    # --------------------------------------------------------
    "Lexical + Semantic": (
        LEXICAL_FEATURES
        + SEMANTIC_FEATURES
    ),

    "Lexical + Semantic + Factuality": (
        LEXICAL_FEATURES
        + SEMANTIC_FEATURES
        + FACTUALITY_FEATURES
    ),

    # --------------------------------------------------------
    # Leave-one-family-out ablations
    # --------------------------------------------------------
    "Full - NLI": (
        LEXICAL_FEATURES
        + SEMANTIC_FEATURES
        + STRUCTURAL_FEATURES
    ),

    "Full - Structural": (
        LEXICAL_FEATURES
        + SEMANTIC_FEATURES
        + FACTUALITY_FEATURES
    ),

    # --------------------------------------------------------
    # Remove all reference-similarity metrics
    # --------------------------------------------------------
    "Full - Reference Similarity": (
        FACTUALITY_FEATURES
        + STRUCTURAL_FEATURES
    )
}


# ------------------------------------------------------------
# 2. Expected number of features for each configuration
# ------------------------------------------------------------

EXPECTED_COUNTS = {
    "Full": 14,
    "Lexical only": 1,
    "Semantic only": 2,
    "Factuality only": 7,
    "Structural only": 4,
    "Lexical + Semantic": 3,
    "Lexical + Semantic + Factuality": 10,
    "Full - NLI": 7,
    "Full - Structural": 10,
    "Full - Reference Similarity": 11
}


# ------------------------------------------------------------
# 3. Helper function: feature-family membership
# ------------------------------------------------------------

def has_any(features, family):
    return any(f in features for f in family)


# ------------------------------------------------------------
# 4. Build an overview table
# ------------------------------------------------------------

ablation_overview = []

for name, features in ABLATION_SETS.items():

    ablation_overview.append({
        "ablation": name,
        "n_features": len(features),

        "lexical": has_any(
            features,
            LEXICAL_FEATURES
        ),

        "semantic": has_any(
            features,
            SEMANTIC_FEATURES
        ),

        "factuality": has_any(
            features,
            FACTUALITY_FEATURES
        ),

        "structural": has_any(
            features,
            STRUCTURAL_FEATURES
        ),

        "features": ", ".join(features)
    })

ablation_overview_df = pd.DataFrame(ablation_overview)


print("=" * 90)
print("FROZEN ABLATION DESIGN")
print("=" * 90)

display(
    ablation_overview_df[
        [
            "ablation",
            "n_features",
            "lexical",
            "semantic",
            "factuality",
            "structural"
        ]
    ]
)


# ------------------------------------------------------------
# 5. Print exact feature lists
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("EXACT FEATURES IN EACH ABLATION")
print("=" * 90)

for name, features in ABLATION_SETS.items():

    print(f"\n{name}")
    print("-" * len(name))

    for f in features:
        print("  -", f)


# ------------------------------------------------------------
# 6. Validate expected feature counts
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FEATURE COUNT CHECK")
print("=" * 90)

for name, expected in EXPECTED_COUNTS.items():

    actual = len(ABLATION_SETS[name])

    print(
        f"{name:35s} "
        f"expected={expected:2d} | "
        f"actual={actual:2d}"
    )

    assert actual == expected, \
        f"Feature count mismatch for {name}"


# ------------------------------------------------------------
# 7. Check for duplicates within every ablation
# ------------------------------------------------------------

for name, features in ABLATION_SETS.items():

    assert len(features) == len(set(features)), \
        f"Duplicate feature detected inside '{name}'"


# ------------------------------------------------------------
# 8. Check that all ablation features belong to the frozen 14
# ------------------------------------------------------------

for name, features in ABLATION_SETS.items():

    invalid = [
        f for f in features
        if f not in ALL_FEATURES
    ]

    assert len(invalid) == 0, \
        f"Unexpected features in '{name}': {invalid}"


# ------------------------------------------------------------
# 9. Check the leave-one-family-out sets mathematically
# ------------------------------------------------------------

assert set(ABLATION_SETS["Full - NLI"]) == \
       set(ALL_FEATURES) - set(FACTUALITY_FEATURES)

assert set(ABLATION_SETS["Full - Structural"]) == \
       set(ALL_FEATURES) - set(STRUCTURAL_FEATURES)

reference_similarity_features = (
    LEXICAL_FEATURES
    + SEMANTIC_FEATURES
)

assert set(
    ABLATION_SETS["Full - Reference Similarity"]
) == set(ALL_FEATURES) - set(reference_similarity_features)


# ------------------------------------------------------------
# 10. Check the progressive model definition
# ------------------------------------------------------------

assert set(
    ABLATION_SETS["Lexical + Semantic + Factuality"]
) == set(
    LEXICAL_FEATURES
    + SEMANTIC_FEATURES
    + FACTUALITY_FEATURES
)


# ------------------------------------------------------------
# 11. Save the ablation design permanently
# ------------------------------------------------------------

ablation_overview_df.to_csv(
    "ROSE_ABLATION_DESIGN.csv",
    index=False
)

print("\nSaved:")
print("ROSE_ABLATION_DESIGN.csv")


# ------------------------------------------------------------
# 12. Print the three primary comparisons
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("PRIMARY ABLATION COMPARISONS")
print("=" * 90)

print("""
1. Full vs Lexical + Semantic
   → Do factuality + structure add value beyond reference similarity?

2. Full vs Full - NLI
   → Does the NLI factuality family contribute?

3. Full vs Full - Structural
   → Does the structural family contribute?
""")


# ------------------------------------------------------------
# 13. Final validation
# ------------------------------------------------------------

assert len(ABLATION_SETS) == 10, \
    "Expected exactly 10 pre-defined ablation configurations."

assert ABLATION_SETS["Full"] == ALL_FEATURES, \
    "The Full model does not exactly match the frozen 14-feature model."

print("=" * 90)
print("✅ STEP 4 PASSED")
print("10 ablation configurations are now frozen.")
print("No ablation performance has been examined yet.")
print("=" * 90)

FROZEN ABLATION DESIGN


,ablation,n_features,lexical,semantic,factuality,structural
0,Full,14,True,True,True,True
1,Lexical only,1,True,False,False,False
2,Semantic only,2,False,True,False,False
3,Factuality only,7,False,False,True,False
4,Structural only,4,False,False,False,True
5,Lexical + Semantic,3,True,True,False,False
6,Lexical + Semantic + Factuality,10,True,True,True,False
7,Full - NLI,7,True,True,False,True
8,Full - Structural,10,True,True,True,False
9,Full - Reference Similarity,11,False,False,True,True



EXACT FEATURES IN EACH ABLATION

Full
----
  - rougeL
  - bertScoreF1
  - moverScore
  - nli_mean_entailment
  - nli_mean_contradiction
  - nli_mean_neutral
  - nli_max_contradiction
  - nli_min_entailment
  - nli_supported_sentence_ratio
  - nli_contradicted_sentence_ratio
  - length_adequacy
  - sentence_count_ratio
  - redundancy_quality
  - average_sentence_length

Lexical only
------------
  - rougeL

Semantic only
-------------
  - bertScoreF1
  - moverScore

Factuality only
---------------
  - nli_mean_entailment
  - nli_mean_contradiction
  - nli_mean_neutral
  - nli_max_contradiction
  - nli_min_entailment
  - nli_supported_sentence_ratio
  - nli_contradicted_sentence_ratio

Structural only
---------------
  - length_adequacy
  - sentence_count_ratio
  - redundancy_quality
  - average_sentence_length

Lexical + Semantic
------------------
  - rougeL
  - bertScoreF1
  - moverScore

Lexical + Semantic + Factuality
-------------------------------
  - rougeL
  - bertScoreF1
  - m

In [168]:
from sklearn.model_selection import GroupKFold
import numpy as np
import pandas as pd

# ============================================================
# STEP 5A: Create ONE frozen set of grouped CV folds
# ============================================================

N_SPLITS = 5

gkf = GroupKFold(n_splits=N_SPLITS)

# Work positionally to avoid issues with original dataframe indices
dev_work_df = rose_dev_df.reset_index(drop=False).copy()

# Preserve original master dataframe index for reproducibility
if "index" in dev_work_df.columns:
    dev_work_df = dev_work_df.rename(
        columns={"index": "master_row_index"}
    )

X_placeholder = dev_work_df[ALL_FEATURES]
y_oof = dev_work_df[TARGET].astype(float).to_numpy()
groups_oof = dev_work_df[GROUP_COLUMN].to_numpy()

# Generate the folds ONCE
cv_splits = list(
    gkf.split(
        X_placeholder,
        y_oof,
        groups=groups_oof
    )
)

print("=" * 80)
print("FROZEN GROUPED CROSS-VALIDATION FOLDS")
print("=" * 80)

fold_assignment = np.full(
    len(dev_work_df),
    fill_value=-1,
    dtype=int
)

fold_rows = []

for fold_no, (train_idx, val_idx) in enumerate(
    cv_splits,
    start=1
):

    train_docs = set(
        groups_oof[train_idx]
    )

    val_docs = set(
        groups_oof[val_idx]
    )

    overlap = train_docs.intersection(val_docs)

    # Store fold assignment
    fold_assignment[val_idx] = fold_no

    fold_rows.append({
        "fold": fold_no,
        "train_rows": len(train_idx),
        "validation_rows": len(val_idx),
        "train_documents": len(train_docs),
        "validation_documents": len(val_docs),
        "document_overlap": len(overlap)
    })

fold_summary_df = pd.DataFrame(fold_rows)

display(fold_summary_df)


# ------------------------------------------------------------
# Strong validation checks
# ------------------------------------------------------------

assert len(cv_splits) == 5

assert (fold_assignment > 0).all(), \
    "Some development rows were never assigned to validation."

assert set(fold_assignment) == {1, 2, 3, 4, 5}, \
    "Unexpected fold labels."

# Each document must occur in only one validation fold
fold_check_df = pd.DataFrame({
    "doc_id": groups_oof,
    "fold": fold_assignment
})

fold_counts_per_doc = (
    fold_check_df
    .groupby("doc_id")["fold"]
    .nunique()
)

assert (fold_counts_per_doc == 1).all(), \
    "A source document appears in multiple validation folds."


# Check no train/validation document overlap
assert (
    fold_summary_df["document_overlap"] == 0
).all(), \
    "Document leakage detected within grouped CV."


print("\nValidation-fold row counts:")
print(
    pd.Series(fold_assignment)
    .value_counts()
    .sort_index()
)

print("\nValidation-fold document counts:")
print(
    fold_check_df
    .drop_duplicates(["doc_id", "fold"])
    ["fold"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# Save fold assignments permanently
# ------------------------------------------------------------

dev_work_df["ablation_cv_fold"] = fold_assignment

fold_manifest_columns = [
    "summary_id",
    "doc_id",
    "ablation_cv_fold"
]

if "master_row_index" in dev_work_df.columns:
    fold_manifest_columns.insert(
        0,
        "master_row_index"
    )

dev_work_df[
    fold_manifest_columns
].to_csv(
    "ROSE_ABLATION_CV_FOLD_MANIFEST.csv",
    index=False
)

print("\nSaved:")
print("ROSE_ABLATION_CV_FOLD_MANIFEST.csv")

print("\n" + "=" * 80)
print("✅ STEP 5A PASSED")
print("The same 5 document-grouped folds are frozen for every ablation.")
print("=" * 80)

FROZEN GROUPED CROSS-VALIDATION FOLDS


,fold,train_rows,validation_rows,train_documents,validation_documents,document_overlap
0,1,3840,960,320,80,0
1,2,3840,960,320,80,0
2,3,3840,960,320,80,0
3,4,3840,960,320,80,0
4,5,3840,960,320,80,0



Validation-fold row counts:
1    960
2    960
3    960
4    960
5    960
Name: count, dtype: int64

Validation-fold document counts:
fold
1    80
2    80
3    80
4    80
5    80
Name: count, dtype: int64

Saved:
ROSE_ABLATION_CV_FOLD_MANIFEST.csv

✅ STEP 5A PASSED
The same 5 document-grouped folds are frozen for every ablation.


In [170]:
# ============================================================
# STEP 5B: Identify unique ablation feature sets
# ============================================================

unique_feature_sets = {}
ablation_aliases = {}

for name, features in ABLATION_SETS.items():

    # Tuple preserves exact feature ordering
    feature_key = tuple(features)

    if feature_key not in unique_feature_sets:

        unique_feature_sets[feature_key] = name
        ablation_aliases[name] = name

    else:

        canonical_name = unique_feature_sets[feature_key]
        ablation_aliases[name] = canonical_name


print("=" * 80)
print("UNIQUE ABLATION CONFIGURATIONS")
print("=" * 80)

print("Conceptual ablation labels:", len(ABLATION_SETS))
print("Unique feature sets to fit:", len(unique_feature_sets))

print("\nAlias mapping:")

for name, canonical in ablation_aliases.items():

    if name == canonical:
        print(f"  {name:35s} -> FIT")
    else:
        print(f"  {name:35s} -> same as '{canonical}'")


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(ABLATION_SETS) == 10, \
    "Expected 10 conceptual ablation labels."

assert len(unique_feature_sets) == 9, \
    "Expected exactly 9 unique feature sets."


# Explicit duplicate check
assert (
    tuple(ABLATION_SETS["Lexical + Semantic + Factuality"])
    ==
    tuple(ABLATION_SETS["Full - Structural"])
), "Expected these two configurations to be identical."


print("\n" + "=" * 80)
print("✅ STEP 5B PASSED")
print("Duplicate ablation configuration identified correctly.")
print("Only 9 unique models will need to be fitted.")
print("=" * 80)

UNIQUE ABLATION CONFIGURATIONS
Conceptual ablation labels: 10
Unique feature sets to fit: 9

Alias mapping:
  Full                                -> FIT
  Lexical only                        -> FIT
  Semantic only                       -> FIT
  Factuality only                     -> FIT
  Structural only                     -> FIT
  Lexical + Semantic                  -> FIT
  Lexical + Semantic + Factuality     -> FIT
  Full - NLI                          -> FIT
  Full - Structural                   -> same as 'Lexical + Semantic + Factuality'
  Full - Reference Similarity         -> FIT

✅ STEP 5B PASSED
Duplicate ablation configuration identified correctly.
Only 9 unique models will need to be fitted.


In [172]:
# ============================================================
# STEP 5C: Generate leakage-safe OOF predictions
# for all 9 unique ablation feature sets
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge


# ------------------------------------------------------------
# 1. Freeze modelling specification
# ------------------------------------------------------------

RIDGE_ALPHA = 1.0

print("=" * 90)
print("ABLATION MODELLING SPECIFICATION")
print("=" * 90)

print("Model: Ridge Regression")
print("Alpha:", RIDGE_ALPHA)
print("Scaling: MinMaxScaler(clip=True)")
print("Cross-validation: 5-fold GroupKFold")
print("Grouping variable: doc_id")
print("Development summaries:", len(dev_work_df))
print("Development documents:", dev_work_df["doc_id"].nunique())
print("Unique feature configurations:", len(unique_feature_sets))


# ------------------------------------------------------------
# 2. Storage objects
# ------------------------------------------------------------

# One 4,800-length OOF prediction vector for each
# unique ablation configuration
oof_predictions_unique = {}

# Store fold-level modelling information
fold_prediction_records = []


# ------------------------------------------------------------
# 3. Fit each unique ablation configuration
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("GENERATING OUT-OF-FOLD PREDICTIONS")
print("=" * 90)


for feature_key, canonical_name in unique_feature_sets.items():

    features = list(feature_key)

    print(
        f"\n>>> {canonical_name}"
        f" | {len(features)} feature(s)"
    )

    # Empty prediction vector.
    # Every row should be filled exactly once.
    oof_pred = np.full(
        shape=len(dev_work_df),
        fill_value=np.nan,
        dtype=float
    )


    # --------------------------------------------------------
    # Use EXACTLY the same frozen folds for every ablation
    # --------------------------------------------------------

    for fold_no, (train_idx, val_idx) in enumerate(
        cv_splits,
        start=1
    ):

        # --------------------------------------------
        # Training and validation features
        # --------------------------------------------

        X_train = (
            dev_work_df
            .iloc[train_idx][features]
            .astype(float)
        )

        X_val = (
            dev_work_df
            .iloc[val_idx][features]
            .astype(float)
        )

        y_train_fold = y_oof[train_idx]


        # --------------------------------------------
        # Confirm document separation again
        # --------------------------------------------

        train_docs = set(
            dev_work_df
            .iloc[train_idx]["doc_id"]
        )

        val_docs = set(
            dev_work_df
            .iloc[val_idx]["doc_id"]
        )

        assert len(
            train_docs.intersection(val_docs)
        ) == 0, (
            f"Document leakage detected in "
            f"{canonical_name}, fold {fold_no}"
        )


        # --------------------------------------------
        # Fixed modelling pipeline
        # --------------------------------------------

        pipeline = Pipeline([
            (
                "scaler",
                MinMaxScaler(
                    clip=True
                )
            ),
            (
                "ridge",
                Ridge(
                    alpha=RIDGE_ALPHA
                )
            )
        ])


        # --------------------------------------------
        # Fit ONLY on training documents
        # --------------------------------------------

        pipeline.fit(
            X_train,
            y_train_fold
        )


        # --------------------------------------------
        # Predict unseen validation documents
        # --------------------------------------------

        fold_pred = pipeline.predict(
            X_val
        )


        # Keep predictions on the human target scale [0,1]
        # using the same clipping convention as the
        # original Learned Hybrid experiment.
        fold_pred = np.clip(
            fold_pred,
            0.0,
            1.0
        )


        # Store predictions in their original
        # development-row positions
        oof_pred[val_idx] = fold_pred


        # --------------------------------------------
        # Store diagnostic information
        # --------------------------------------------

        fold_prediction_records.append({
            "ablation": canonical_name,
            "fold": fold_no,
            "n_features": len(features),

            "train_rows": len(train_idx),
            "validation_rows": len(val_idx),

            "train_documents": len(train_docs),
            "validation_documents": len(val_docs),

            "prediction_min": float(
                np.min(fold_pred)
            ),

            "prediction_mean": float(
                np.mean(fold_pred)
            ),

            "prediction_max": float(
                np.max(fold_pred)
            )
        })


    # --------------------------------------------------------
    # 4. Validate complete OOF vector
    # --------------------------------------------------------

    assert len(oof_pred) == 4800, (
        f"Unexpected OOF length for {canonical_name}"
    )

    assert not np.isnan(oof_pred).any(), (
        f"Missing OOF predictions for {canonical_name}"
    )

    assert np.isfinite(oof_pred).all(), (
        f"Non-finite OOF predictions for {canonical_name}"
    )

    assert oof_pred.min() >= 0.0, (
        f"Prediction below 0 for {canonical_name}"
    )

    assert oof_pred.max() <= 1.0, (
        f"Prediction above 1 for {canonical_name}"
    )


    # Freeze the completed OOF predictions
    oof_predictions_unique[
        canonical_name
    ] = oof_pred


    print(
        "    COMPLETE"
        f" | min={oof_pred.min():.4f}"
        f" | mean={oof_pred.mean():.4f}"
        f" | max={oof_pred.max():.4f}"
    )


# ------------------------------------------------------------
# 5. Fold-level diagnostic table
# ------------------------------------------------------------

oof_fit_summary_df = pd.DataFrame(
    fold_prediction_records
)

print("\n" + "=" * 90)
print("FOLD-LEVEL DIAGNOSTIC SUMMARY")
print("=" * 90)

display(
    oof_fit_summary_df
)


# ------------------------------------------------------------
# 6. Final strong checks
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("OOF COMPLETION CHECK")
print("=" * 90)

print(
    "Unique ablation prediction vectors:",
    len(oof_predictions_unique)
)

for name, pred in oof_predictions_unique.items():

    print(
        f"{name:35s}"
        f" rows={len(pred)}"
        f" | missing={np.isnan(pred).sum()}"
        f" | finite={np.isfinite(pred).all()}"
    )

    assert len(pred) == 4800
    assert np.isnan(pred).sum() == 0
    assert np.isfinite(pred).all()


assert len(oof_predictions_unique) == 9, (
    "Expected OOF predictions for 9 unique feature sets."
)


# ------------------------------------------------------------
# 7. Save fold diagnostics
# ------------------------------------------------------------

oof_fit_summary_df.to_csv(
    "ROSE_ABLATION_OOF_FOLD_SUMMARY.csv",
    index=False
)

print("\nSaved:")
print("ROSE_ABLATION_OOF_FOLD_SUMMARY.csv")


print("\n" + "=" * 90)
print("✅ STEP 5C PASSED")
print("Leakage-safe OOF predictions generated for all 9 unique models.")
print("No holdout data were used.")
print("Performance has not yet been calculated.")
print("=" * 90)

ABLATION MODELLING SPECIFICATION
Model: Ridge Regression
Alpha: 1.0
Scaling: MinMaxScaler(clip=True)
Cross-validation: 5-fold GroupKFold
Grouping variable: doc_id
Development summaries: 4800
Development documents: 400
Unique feature configurations: 9

GENERATING OUT-OF-FOLD PREDICTIONS

>>> Full | 14 feature(s)
    COMPLETE | min=0.0000 | mean=0.4092 | max=1.0000

>>> Lexical only | 1 feature(s)
    COMPLETE | min=0.1185 | mean=0.4090 | max=1.0000

>>> Semantic only | 2 feature(s)
    COMPLETE | min=0.0000 | mean=0.4092 | max=1.0000

>>> Factuality only | 7 feature(s)
    COMPLETE | min=0.1663 | mean=0.4090 | max=0.4707

>>> Structural only | 4 feature(s)
    COMPLETE | min=0.2534 | mean=0.4090 | max=0.6339

>>> Lexical + Semantic | 3 feature(s)
    COMPLETE | min=0.0000 | mean=0.4092 | max=1.0000

>>> Lexical + Semantic + Factuality | 10 feature(s)
    COMPLETE | min=0.0000 | mean=0.4092 | max=1.0000

>>> Full - NLI | 7 feature(s)
    COMPLETE | min=0.0000 | mean=0.4092 | max=1.0000



,ablation,fold,n_features,train_rows,validation_rows,train_documents,validation_documents,prediction_min,prediction_mean,prediction_max
0,Full,1,14,3840,960,320,80,0.000000,0.423600,1.000000
1,Full,2,14,3840,960,320,80,0.000000,0.392497,1.000000
2,Full,3,14,3840,960,320,80,0.000000,0.438062,1.000000
3,Full,4,14,3840,960,320,80,0.000000,0.391321,0.829261
4,Full,5,14,3840,960,320,80,0.000000,0.400707,0.931777
5,Lexical only,1,1,3840,960,320,80,0.167504,0.413792,1.000000
6,Lexical only,2,1,3840,960,320,80,0.118479,0.405069,0.994081
7,Lexical only,3,1,3840,960,320,80,0.174490,0.440357,1.000000
8,Lexical only,4,1,3840,960,320,80,0.157762,0.394971,0.859607
9,Lexical only,5,1,3840,960,320,80,0.120087,0.390602,0.970327



OOF COMPLETION CHECK
Unique ablation prediction vectors: 9
Full                                rows=4800 | missing=0 | finite=True
Lexical only                        rows=4800 | missing=0 | finite=True
Semantic only                       rows=4800 | missing=0 | finite=True
Factuality only                     rows=4800 | missing=0 | finite=True
Structural only                     rows=4800 | missing=0 | finite=True
Lexical + Semantic                  rows=4800 | missing=0 | finite=True
Lexical + Semantic + Factuality     rows=4800 | missing=0 | finite=True
Full - NLI                          rows=4800 | missing=0 | finite=True
Full - Reference Similarity         rows=4800 | missing=0 | finite=True

Saved:
ROSE_ABLATION_OOF_FOLD_SUMMARY.csv

✅ STEP 5C PASSED
Leakage-safe OOF predictions generated for all 9 unique models.
No holdout data were used.
Performance has not yet been calculated.


In [174]:
# ============================================================
# STEP 5D: Build and save complete OOF ablation prediction table
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Helper for clean prediction-column names
# ------------------------------------------------------------

def safe_column_name(name):
    return (
        name.lower()
        .replace(" + ", "_plus_")
        .replace(" - ", "_minus_")
        .replace(" ", "_")
        .replace("-", "_")
    )


# ------------------------------------------------------------
# 2. Start with identifiers + human target
# ------------------------------------------------------------

oof_ablation_df = pd.DataFrame({
    "summary_id": dev_work_df["summary_id"].values,
    "doc_id": dev_work_df["doc_id"].values,
    "model": dev_work_df["model"].values,
    "human_content_coverage": y_oof,
    "cv_fold": fold_assignment
})


# ------------------------------------------------------------
# 3. Add prediction for every CONCEPTUAL ablation label
# ------------------------------------------------------------

for ablation_name in ABLATION_SETS.keys():

    canonical_name = ablation_aliases[
        ablation_name
    ]

    predictions = oof_predictions_unique[
        canonical_name
    ]

    column_name = (
        "pred_"
        + safe_column_name(ablation_name)
    )

    oof_ablation_df[
        column_name
    ] = predictions


# ------------------------------------------------------------
# 4. Identify prediction columns
# ------------------------------------------------------------

prediction_columns = [
    c for c in oof_ablation_df.columns
    if c.startswith("pred_")
]


print("=" * 90)
print("COMPLETE OOF ABLATION PREDICTION TABLE")
print("=" * 90)

print("Shape:", oof_ablation_df.shape)
print("Prediction columns:", len(prediction_columns))

print("\nColumns:")

for c in oof_ablation_df.columns:
    print("  -", c)


# ------------------------------------------------------------
# 5. Preview
# ------------------------------------------------------------

print("\nPreview:")

display(
    oof_ablation_df.head()
)


# ------------------------------------------------------------
# 6. Missing / finite prediction checks
# ------------------------------------------------------------

missing_predictions = (
    oof_ablation_df[
        prediction_columns
    ]
    .isna()
    .sum()
)

print("\n" + "=" * 90)
print("PREDICTION COMPLETENESS")
print("=" * 90)

print(missing_predictions)

assert (
    missing_predictions == 0
).all(), "Missing OOF predictions detected."


finite_check = np.isfinite(
    oof_ablation_df[
        prediction_columns
    ].to_numpy()
).all()

print("\nAll predictions finite:", finite_check)

assert finite_check


# ------------------------------------------------------------
# 7. Verify duplicate conceptual configurations
# ------------------------------------------------------------

col_lsf = (
    "pred_lexical_plus_semantic_plus_factuality"
)

col_no_struct = (
    "pred_full_minus_structural"
)

exact_same = np.array_equal(
    oof_ablation_df[col_lsf].to_numpy(),
    oof_ablation_df[col_no_struct].to_numpy()
)

print("\n" + "=" * 90)
print("DUPLICATE-CONFIGURATION CHECK")
print("=" * 90)

print(
    "Lexical + Semantic + Factuality "
    "identical to Full - Structural:",
    exact_same
)

assert exact_same, (
    "Identical feature sets unexpectedly produced "
    "different predictions."
)


# ------------------------------------------------------------
# 8. Check each development row appears once
# ------------------------------------------------------------

print("\nUnique summaries:", oof_ablation_df["summary_id"].nunique())
print("Total rows:", len(oof_ablation_df))
print("Unique documents:", oof_ablation_df["doc_id"].nunique())

assert len(oof_ablation_df) == 4800
assert oof_ablation_df["summary_id"].nunique() == 4800
assert oof_ablation_df["doc_id"].nunique() == 400


# ------------------------------------------------------------
# 9. Fold check
# ------------------------------------------------------------

print("\nFold distribution:")

print(
    oof_ablation_df["cv_fold"]
    .value_counts()
    .sort_index()
)

assert (
    oof_ablation_df["cv_fold"]
    .value_counts()
    .sort_index()
    .eq(960)
).all()


# ------------------------------------------------------------
# 10. Save permanent checkpoint
# ------------------------------------------------------------

oof_ablation_df.to_parquet(
    "ROSE_ABLATION_OOF_PREDICTIONS.parquet",
    index=False
)

# CSV version is optional but useful for easy inspection
oof_ablation_df.to_csv(
    "ROSE_ABLATION_OOF_PREDICTIONS.csv",
    index=False
)


print("\nSaved:")
print("ROSE_ABLATION_OOF_PREDICTIONS.parquet")
print("ROSE_ABLATION_OOF_PREDICTIONS.csv")


print("\n" + "=" * 90)
print("✅ STEP 5D PASSED")
print("All development-only OOF ablation predictions are now frozen.")
print("No holdout data were used.")
print("=" * 90)

COMPLETE OOF ABLATION PREDICTION TABLE
Shape: (4800, 15)
Prediction columns: 10

Columns:
  - summary_id
  - doc_id
  - model
  - human_content_coverage
  - cv_fold
  - pred_full
  - pred_lexical_only
  - pred_semantic_only
  - pred_factuality_only
  - pred_structural_only
  - pred_lexical_plus_semantic
  - pred_lexical_plus_semantic_plus_factuality
  - pred_full_minus_nli
  - pred_full_minus_structural
  - pred_full_minus_reference_similarity

Preview:


,summary_id,doc_id,model,human_content_coverage,cv_fold,pred_full,pred_lexical_only,pred_semantic_only,pred_factuality_only,pred_structural_only,pred_lexical_plus_semantic,pred_lexical_plus_semantic_plus_factuality,pred_full_minus_nli,pred_full_minus_structural,pred_full_minus_reference_similarity
0,12,000642916e3a6c33411c617cf2f3c134a206fba8,bart,0.363636,5,0.260429,0.327674,0.283410,0.400348,0.377461,0.286765,0.280651,0.256823,0.280651,0.373302
1,13,000642916e3a6c33411c617cf2f3c134a206fba8,gold,0.636364,5,0.320237,0.413888,0.398611,0.398367,0.363230,0.394377,0.364057,0.331042,0.364057,0.367250
2,14,000642916e3a6c33411c617cf2f3c134a206fba8,pegasus,0.363636,5,0.487664,0.522233,0.437629,0.406497,0.426707,0.464725,0.466456,0.497994,0.466456,0.411080
3,15,000642916e3a6c33411c617cf2f3c134a206fba8,brio,0.454545,5,0.417357,0.522028,0.454687,0.409738,0.377779,0.476869,0.479469,0.422311,0.479469,0.370045
4,16,000642916e3a6c33411c617cf2f3c134a206fba8,gsum,0.363636,5,0.300070,0.380658,0.351295,0.417769,0.375726,0.351496,0.356607,0.297969,0.356607,0.379726



PREDICTION COMPLETENESS
pred_full                                     0
pred_lexical_only                             0
pred_semantic_only                            0
pred_factuality_only                          0
pred_structural_only                          0
pred_lexical_plus_semantic                    0
pred_lexical_plus_semantic_plus_factuality    0
pred_full_minus_nli                           0
pred_full_minus_structural                    0
pred_full_minus_reference_similarity          0
dtype: int64

All predictions finite: True

DUPLICATE-CONFIGURATION CHECK
Lexical + Semantic + Factuality identical to Full - Structural: True

Unique summaries: 4800
Total rows: 4800
Unique documents: 400

Fold distribution:
cv_fold
1    960
2    960
3    960
4    960
5    960
Name: count, dtype: int64

Saved:
ROSE_ABLATION_OOF_PREDICTIONS.parquet
ROSE_ABLATION_OOF_PREDICTIONS.csv

✅ STEP 5D PASSED
All development-only OOF ablation predictions are now frozen.
No holdout data were used.


In [176]:
# ============================================================
# STEP 6A: Evaluate OOF ablation predictions
# Pearson / Spearman / Kendall / MAE / RMSE / CCC
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr, kendalltau
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ------------------------------------------------------------
# 1. Lin's Concordance Correlation Coefficient
# ------------------------------------------------------------

def concordance_correlation_coefficient(y_true, y_pred):
    """
    Lin's Concordance Correlation Coefficient (CCC).

    Uses population variance/covariance (ddof=0).

    CCC = 2 * cov(x,y) /
          [var(x) + var(y) + (mean_x - mean_y)^2]
    """

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)

    var_true = np.var(y_true, ddof=0)
    var_pred = np.var(y_pred, ddof=0)

    covariance = np.mean(
        (y_true - mean_true) *
        (y_pred - mean_pred)
    )

    denominator = (
        var_true
        + var_pred
        + (mean_true - mean_pred) ** 2
    )

    if denominator == 0:
        return np.nan

    return (
        2 * covariance
        / denominator
    )


# ------------------------------------------------------------
# 2. Define the 9 UNIQUE ablation models to report
# ------------------------------------------------------------

REPORTING_MODELS = {

    "Full": "pred_full",

    "Lexical only":
        "pred_lexical_only",

    "Semantic only":
        "pred_semantic_only",

    "Factuality only":
        "pred_factuality_only",

    "Structural only":
        "pred_structural_only",

    "Lexical + Semantic":
        "pred_lexical_plus_semantic",

    "Full - NLI":
        "pred_full_minus_nli",

    # Same predictions as
    # Lexical + Semantic + Factuality
    "Full - Structural":
        "pred_full_minus_structural",

    "Full - Reference Similarity":
        "pred_full_minus_reference_similarity"
}


# ------------------------------------------------------------
# 3. Verify every prediction column exists
# ------------------------------------------------------------

missing_columns = [
    col
    for col in REPORTING_MODELS.values()
    if col not in oof_ablation_df.columns
]

assert len(missing_columns) == 0, (
    f"Missing prediction columns: {missing_columns}"
)


# ------------------------------------------------------------
# 4. Human target
# ------------------------------------------------------------

y_true = (
    oof_ablation_df[
        "human_content_coverage"
    ]
    .astype(float)
    .to_numpy()
)


# ------------------------------------------------------------
# 5. Evaluate every model
# ------------------------------------------------------------

results = []

for model_name, pred_column in REPORTING_MODELS.items():

    y_pred = (
        oof_ablation_df[
            pred_column
        ]
        .astype(float)
        .to_numpy()
    )

    # Correlations
    pearson = pearsonr(
        y_true,
        y_pred
    ).statistic

    spearman = spearmanr(
        y_true,
        y_pred
    ).statistic

    kendall = kendalltau(
        y_true,
        y_pred
    ).statistic

    # Prediction errors
    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    # Agreement
    ccc = (
        concordance_correlation_coefficient(
            y_true,
            y_pred
        )
    )

    # Useful diagnostics
    prediction_mean = np.mean(y_pred)

    clipped_low = np.sum(
        np.isclose(
            y_pred,
            0.0,
            atol=1e-12
        )
    )

    clipped_high = np.sum(
        np.isclose(
            y_pred,
            1.0,
            atol=1e-12
        )
    )

    results.append({
        "Model": model_name,
        "Pearson": pearson,
        "Spearman": spearman,
        "Kendall": kendall,
        "MAE": mae,
        "RMSE": rmse,
        "CCC": ccc,
        "Prediction_Mean": prediction_mean,
        "Clipped_at_0": clipped_low,
        "Clipped_at_1": clipped_high
    })


ablation_results_df = pd.DataFrame(
    results
)


# ------------------------------------------------------------
# 6. Sort with Full first, then strongest Pearson
# ------------------------------------------------------------

full_row = ablation_results_df[
    ablation_results_df["Model"] == "Full"
]

other_rows = (
    ablation_results_df[
        ablation_results_df["Model"] != "Full"
    ]
    .sort_values(
        "Pearson",
        ascending=False
    )
)

ablation_results_df = pd.concat(
    [full_row, other_rows],
    ignore_index=True
)


# ------------------------------------------------------------
# 7. Display main thesis-style table
# ------------------------------------------------------------

print("=" * 100)
print("RoSE DEVELOPMENT ABLATION RESULTS")
print("5-FOLD DOCUMENT-GROUPED OUT-OF-FOLD PREDICTIONS")
print("=" * 100)

display(
    ablation_results_df[
        [
            "Model",
            "Pearson",
            "Spearman",
            "Kendall",
            "MAE",
            "RMSE",
            "CCC"
        ]
    ].round(4)
)


# ------------------------------------------------------------
# 8. Display prediction diagnostics separately
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("PREDICTION RANGE / CLIPPING DIAGNOSTICS")
print("=" * 100)

display(
    ablation_results_df[
        [
            "Model",
            "Prediction_Mean",
            "Clipped_at_0",
            "Clipped_at_1"
        ]
    ].round(4)
)


# ------------------------------------------------------------
# 9. Save results
# ------------------------------------------------------------

ablation_results_df.to_csv(
    "ROSE_ABLATION_MAIN_RESULTS.csv",
    index=False
)

print("\nSaved:")
print("ROSE_ABLATION_MAIN_RESULTS.csv")


# ------------------------------------------------------------
# 10. Strong completion checks
# ------------------------------------------------------------

assert len(ablation_results_df) == 9

assert (
    ablation_results_df[
        ["Pearson", "Spearman", "Kendall",
         "MAE", "RMSE", "CCC"]
    ]
    .notna()
    .all()
    .all()
)

print("\n" + "=" * 100)
print("✅ STEP 6A PASSED")
print("Main OOF ablation metrics calculated successfully.")
print("=" * 100)

RoSE DEVELOPMENT ABLATION RESULTS
5-FOLD DOCUMENT-GROUPED OUT-OF-FOLD PREDICTIONS


,Model,Pearson,Spearman,Kendall,MAE,RMSE,CCC
0,Full,0.6851,0.6765,0.4931,0.1368,0.1724,0.6351
1,Full - NLI,0.6861,0.6784,0.4946,0.1365,0.1722,0.6359
2,Lexical + Semantic,0.6392,0.6308,0.4534,0.1444,0.1820,0.5763
3,Full - Structural,0.6375,0.6279,0.4512,0.1446,0.1824,0.5751
4,Semantic only,0.6356,0.6264,0.4507,0.1448,0.1828,0.5706
5,Lexical only,0.5716,0.5622,0.3996,0.1557,0.1942,0.4874
6,Full - Reference Similarity,0.1847,0.1867,0.1259,0.1894,0.2328,0.0787
7,Structural only,0.1839,0.1881,0.1271,0.1891,0.2327,0.0744
8,Factuality only,-0.0154,-0.0517,-0.0352,0.1936,0.2377,-0.0024



PREDICTION RANGE / CLIPPING DIAGNOSTICS


,Model,Prediction_Mean,Clipped_at_0,Clipped_at_1
0,Full,0.4092,23,18
1,Full - NLI,0.4092,21,18
2,Lexical + Semantic,0.4092,14,16
3,Full - Structural,0.4092,15,16
4,Semantic only,0.4092,21,14
5,Lexical only,0.4090,0,14
6,Full - Reference Similarity,0.4090,0,0
7,Structural only,0.4090,0,0
8,Factuality only,0.4090,0,0



Saved:
ROSE_ABLATION_MAIN_RESULTS.csv

✅ STEP 6A PASSED
Main OOF ablation metrics calculated successfully.


In [178]:
# ============================================================
# STEP 6B: Compare every ablation directly with FULL model
# ============================================================

metric_columns = [
    "Pearson",
    "Spearman",
    "Kendall",
    "MAE",
    "RMSE",
    "CCC"
]

full_metrics = (
    ablation_results_df
    .set_index("Model")
    .loc["Full"]
)


comparison_rows = []

for _, row in ablation_results_df.iterrows():

    model_name = row["Model"]

    if model_name == "Full":
        continue

    comparison_rows.append({

        "Model": model_name,

        # Positive values mean Full is better
        # for correlations / CCC
        "Full_minus_Model_Pearson":
            full_metrics["Pearson"]
            - row["Pearson"],

        "Full_minus_Model_Spearman":
            full_metrics["Spearman"]
            - row["Spearman"],

        "Full_minus_Model_Kendall":
            full_metrics["Kendall"]
            - row["Kendall"],

        # For errors, positive means the ablated
        # model has MORE error than Full
        "Model_minus_Full_MAE":
            row["MAE"]
            - full_metrics["MAE"],

        "Model_minus_Full_RMSE":
            row["RMSE"]
            - full_metrics["RMSE"],

        "Full_minus_Model_CCC":
            full_metrics["CCC"]
            - row["CCC"]
    })


ablation_delta_df = pd.DataFrame(
    comparison_rows
)


print("=" * 105)
print("CHANGE RELATIVE TO FULL 14-FEATURE MODEL")
print("=" * 105)

display(
    ablation_delta_df.round(4)
)


# ------------------------------------------------------------
# Primary scientific comparisons
# ------------------------------------------------------------

primary_models = [
    "Lexical + Semantic",
    "Full - NLI",
    "Full - Structural",
    "Full - Reference Similarity"
]

print("\n" + "=" * 105)
print("PRIMARY ABLATION COMPARISONS")
print("=" * 105)

display(
    ablation_delta_df[
        ablation_delta_df["Model"].isin(
            primary_models
        )
    ].round(4)
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

ablation_delta_df.to_csv(
    "ROSE_ABLATION_DELTAS_VS_FULL.csv",
    index=False
)

print("\nSaved:")
print("ROSE_ABLATION_DELTAS_VS_FULL.csv")


print("\n" + "=" * 105)
print("✅ STEP 6B PASSED")
print("=" * 105)

CHANGE RELATIVE TO FULL 14-FEATURE MODEL


,Model,Full_minus_Model_Pearson,Full_minus_Model_Spearman,Full_minus_Model_Kendall,Model_minus_Full_MAE,Model_minus_Full_RMSE,Full_minus_Model_CCC
0,Full - NLI,-0.0010,-0.0019,-0.0015,-0.0003,-0.0002,-0.0007
1,Lexical + Semantic,0.0459,0.0457,0.0397,0.0075,0.0096,0.0588
2,Full - Structural,0.0475,0.0485,0.0419,0.0078,0.0099,0.0600
3,Semantic only,0.0495,0.0501,0.0424,0.0079,0.0103,0.0645
4,Lexical only,0.1134,0.1142,0.0935,0.0189,0.0218,0.1477
5,Full - Reference Similarity,0.5004,0.4898,0.3671,0.0526,0.0604,0.5564
6,Structural only,0.5012,0.4883,0.3660,0.0523,0.0603,0.5607
7,Factuality only,0.7005,0.7282,0.5283,0.0568,0.0652,0.6375



PRIMARY ABLATION COMPARISONS


,Model,Full_minus_Model_Pearson,Full_minus_Model_Spearman,Full_minus_Model_Kendall,Model_minus_Full_MAE,Model_minus_Full_RMSE,Full_minus_Model_CCC
0,Full - NLI,-0.0010,-0.0019,-0.0015,-0.0003,-0.0002,-0.0007
1,Lexical + Semantic,0.0459,0.0457,0.0397,0.0075,0.0096,0.0588
2,Full - Structural,0.0475,0.0485,0.0419,0.0078,0.0099,0.0600
5,Full - Reference Similarity,0.5004,0.4898,0.3671,0.0526,0.0604,0.5564



Saved:
ROSE_ABLATION_DELTAS_VS_FULL.csv

✅ STEP 6B PASSED


In [180]:
# ============================================================
# STEP 7A: Paired DOCUMENT-level bootstrap for ablation deltas
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr, kendalltau
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ------------------------------------------------------------
# 1. Bootstrap specification
# ------------------------------------------------------------

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


# ------------------------------------------------------------
# 2. Primary ablation comparisons
# ------------------------------------------------------------

PRIMARY_BOOTSTRAP_MODELS = {
    "Full - NLI":
        "pred_full_minus_nli",

    "Lexical + Semantic":
        "pred_lexical_plus_semantic",

    "Full - Structural":
        "pred_full_minus_structural",

    "Full - Reference Similarity":
        "pred_full_minus_reference_similarity"
}


FULL_COLUMN = "pred_full"
TARGET_COLUMN = "human_content_coverage"


# ------------------------------------------------------------
# 3. Prepare document clusters
# ------------------------------------------------------------

doc_ids_array = (
    oof_ablation_df["doc_id"]
    .to_numpy()
)

unique_docs = np.unique(
    doc_ids_array
)

n_documents = len(unique_docs)

print("=" * 95)
print("DOCUMENT-LEVEL BOOTSTRAP SPECIFICATION")
print("=" * 95)

print("Development documents:", n_documents)
print("Summaries:", len(oof_ablation_df))
print("Bootstrap replicates:", N_BOOTSTRAP)
print("Random seed:", BOOTSTRAP_SEED)
print("Resampling unit: source document")
print("Sampling: with replacement")


assert n_documents == 400


# ------------------------------------------------------------
# 4. Map every document to its summary-row positions
# ------------------------------------------------------------

doc_to_indices = {
    doc_id: np.flatnonzero(
        doc_ids_array == doc_id
    )
    for doc_id in unique_docs
}


# Every development document should have 12 summaries
assert all(
    len(indices) == 12
    for indices in doc_to_indices.values()
)


# ------------------------------------------------------------
# 5. Store arrays once for speed
# ------------------------------------------------------------

y_all = (
    oof_ablation_df[TARGET_COLUMN]
    .astype(float)
    .to_numpy()
)

full_all = (
    oof_ablation_df[FULL_COLUMN]
    .astype(float)
    .to_numpy()
)

ablation_arrays = {
    model_name:
        oof_ablation_df[pred_col]
        .astype(float)
        .to_numpy()

    for model_name, pred_col
    in PRIMARY_BOOTSTRAP_MODELS.items()
}


# ------------------------------------------------------------
# 6. Metric helper
# ------------------------------------------------------------

def calculate_ablation_metrics(y_true, y_pred):

    pearson = pearsonr(
        y_true,
        y_pred
    ).statistic

    spearman = spearmanr(
        y_true,
        y_pred
    ).statistic

    kendall = kendalltau(
        y_true,
        y_pred
    ).statistic

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    ccc = concordance_correlation_coefficient(
        y_true,
        y_pred
    )

    return {
        "Pearson": pearson,
        "Spearman": spearman,
        "Kendall": kendall,
        "MAE": mae,
        "RMSE": rmse,
        "CCC": ccc
    }


# ------------------------------------------------------------
# 7. Bootstrap
# ------------------------------------------------------------

bootstrap_records = []

print("\nRunning paired document bootstrap...")


for b in range(N_BOOTSTRAP):

    # --------------------------------------------------------
    # Sample 400 document IDs WITH replacement
    # --------------------------------------------------------

    sampled_docs = rng.choice(
        unique_docs,
        size=n_documents,
        replace=True
    )


    # --------------------------------------------------------
    # Preserve multiplicity:
    # if a document is sampled twice,
    # all 12 of its rows appear twice
    # --------------------------------------------------------

    sampled_indices = np.concatenate([
        doc_to_indices[doc_id]
        for doc_id in sampled_docs
    ])


    y_b = y_all[
        sampled_indices
    ]

    full_b = full_all[
        sampled_indices
    ]


    # Full metrics calculated once per resample
    full_metrics_b = (
        calculate_ablation_metrics(
            y_b,
            full_b
        )
    )


    # --------------------------------------------------------
    # Compare each ablation on EXACTLY same sampled documents
    # --------------------------------------------------------

    for model_name, ablation_all in ablation_arrays.items():

        ablated_b = ablation_all[
            sampled_indices
        ]

        ablated_metrics_b = (
            calculate_ablation_metrics(
                y_b,
                ablated_b
            )
        )


        # ----------------------------------------------------
        # Define every delta so:
        #
        # POSITIVE = FULL MODEL BETTER
        # ----------------------------------------------------

        bootstrap_records.append({

            "bootstrap": b + 1,
            "Model": model_name,

            "Delta_Pearson":
                full_metrics_b["Pearson"]
                - ablated_metrics_b["Pearson"],

            "Delta_Spearman":
                full_metrics_b["Spearman"]
                - ablated_metrics_b["Spearman"],

            "Delta_Kendall":
                full_metrics_b["Kendall"]
                - ablated_metrics_b["Kendall"],

            "Delta_CCC":
                full_metrics_b["CCC"]
                - ablated_metrics_b["CCC"],

            # Error direction reversed:
            # positive = ablated model has MORE error
            "Delta_MAE":
                ablated_metrics_b["MAE"]
                - full_metrics_b["MAE"],

            "Delta_RMSE":
                ablated_metrics_b["RMSE"]
                - full_metrics_b["RMSE"]
        })


    # Progress indicator
    if (b + 1) % 500 == 0:
        print(
            f"  completed {b + 1:,} / "
            f"{N_BOOTSTRAP:,}"
        )


bootstrap_ablation_df = pd.DataFrame(
    bootstrap_records
)


print("\nBootstrap rows:", len(bootstrap_ablation_df))

expected_rows = (
    N_BOOTSTRAP
    * len(PRIMARY_BOOTSTRAP_MODELS)
)

print("Expected rows:", expected_rows)

assert len(bootstrap_ablation_df) == expected_rows


print("\n" + "=" * 95)
print("✅ STEP 7A PASSED")
print("Paired document-bootstrap distributions generated.")
print("=" * 95)

DOCUMENT-LEVEL BOOTSTRAP SPECIFICATION
Development documents: 400
Summaries: 4800
Bootstrap replicates: 5000
Random seed: 42
Resampling unit: source document
Sampling: with replacement

Running paired document bootstrap...
  completed 500 / 5,000
  completed 1,000 / 5,000
  completed 1,500 / 5,000
  completed 2,000 / 5,000
  completed 2,500 / 5,000
  completed 3,000 / 5,000
  completed 3,500 / 5,000
  completed 4,000 / 5,000
  completed 4,500 / 5,000
  completed 5,000 / 5,000

Bootstrap rows: 20000
Expected rows: 20000

✅ STEP 7A PASSED
Paired document-bootstrap distributions generated.


In [182]:
# ============================================================
# STEP 7B: Summarise paired bootstrap ablation uncertainty
# ============================================================

DELTA_COLUMNS = [
    "Delta_Pearson",
    "Delta_Spearman",
    "Delta_Kendall",
    "Delta_MAE",
    "Delta_RMSE",
    "Delta_CCC"
]


# ------------------------------------------------------------
# Observed deltas from the original OOF sample
# ------------------------------------------------------------

observed_lookup = (
    ablation_delta_df
    .set_index("Model")
)


bootstrap_summary_rows = []


for model_name in PRIMARY_BOOTSTRAP_MODELS.keys():

    model_boot = (
        bootstrap_ablation_df[
            bootstrap_ablation_df["Model"]
            == model_name
        ]
    )

    row = {
        "Model": model_name
    }


    # --------------------------------------------------------
    # Match observed differences exactly
    # --------------------------------------------------------

    observed_row = observed_lookup.loc[
        model_name
    ]


    observed_mapping = {
        "Delta_Pearson":
            observed_row[
                "Full_minus_Model_Pearson"
            ],

        "Delta_Spearman":
            observed_row[
                "Full_minus_Model_Spearman"
            ],

        "Delta_Kendall":
            observed_row[
                "Full_minus_Model_Kendall"
            ],

        "Delta_MAE":
            observed_row[
                "Model_minus_Full_MAE"
            ],

        "Delta_RMSE":
            observed_row[
                "Model_minus_Full_RMSE"
            ],

        "Delta_CCC":
            observed_row[
                "Full_minus_Model_CCC"
            ]
    }


    for metric in DELTA_COLUMNS:

        values = (
            model_boot[metric]
            .to_numpy()
        )

        row[
            f"{metric}_Observed"
        ] = observed_mapping[metric]

        row[
            f"{metric}_BootstrapMean"
        ] = np.mean(values)

        row[
            f"{metric}_CI_Lower"
        ] = np.percentile(
            values,
            2.5
        )

        row[
            f"{metric}_CI_Upper"
        ] = np.percentile(
            values,
            97.5
        )

        # Descriptive CI flag:
        # True if percentile CI excludes zero
        lower = row[
            f"{metric}_CI_Lower"
        ]

        upper = row[
            f"{metric}_CI_Upper"
        ]

        row[
            f"{metric}_CI_Excludes_Zero"
        ] = (
            (lower > 0)
            or
            (upper < 0)
        )


    bootstrap_summary_rows.append(
        row
    )


ablation_bootstrap_summary_df = pd.DataFrame(
    bootstrap_summary_rows
)


print("=" * 120)
print("PAIRED DOCUMENT-BOOTSTRAP ABLATION RESULTS")
print("Positive delta = Full model performs better")
print("=" * 120)


# ------------------------------------------------------------
# Compact Pearson table first
# ------------------------------------------------------------

pearson_table = (
    ablation_bootstrap_summary_df[
        [
            "Model",
            "Delta_Pearson_Observed",
            "Delta_Pearson_BootstrapMean",
            "Delta_Pearson_CI_Lower",
            "Delta_Pearson_CI_Upper",
            "Delta_Pearson_CI_Excludes_Zero"
        ]
    ]
    .copy()
)

display(
    pearson_table.round(6)
)


# ------------------------------------------------------------
# Compact all-metric table
# ------------------------------------------------------------

compact_rows = []

for _, row in ablation_bootstrap_summary_df.iterrows():

    compact_rows.append({

        "Model":
            row["Model"],

        "Pearson Δ":
            row["Delta_Pearson_Observed"],

        "Pearson 95% CI":
            (
                f"[{row['Delta_Pearson_CI_Lower']:.4f}, "
                f"{row['Delta_Pearson_CI_Upper']:.4f}]"
            ),

        "Spearman Δ":
            row["Delta_Spearman_Observed"],

        "Spearman 95% CI":
            (
                f"[{row['Delta_Spearman_CI_Lower']:.4f}, "
                f"{row['Delta_Spearman_CI_Upper']:.4f}]"
            ),

        "Kendall Δ":
            row["Delta_Kendall_Observed"],

        "Kendall 95% CI":
            (
                f"[{row['Delta_Kendall_CI_Lower']:.4f}, "
                f"{row['Delta_Kendall_CI_Upper']:.4f}]"
            ),

        "MAE Δ":
            row["Delta_MAE_Observed"],

        "MAE 95% CI":
            (
                f"[{row['Delta_MAE_CI_Lower']:.4f}, "
                f"{row['Delta_MAE_CI_Upper']:.4f}]"
            ),

        "CCC Δ":
            row["Delta_CCC_Observed"],

        "CCC 95% CI":
            (
                f"[{row['Delta_CCC_CI_Lower']:.4f}, "
                f"{row['Delta_CCC_CI_Upper']:.4f}]"
            )
    })


ablation_bootstrap_compact_df = pd.DataFrame(
    compact_rows
)


print("\n" + "=" * 120)
print("COMPACT PRIMARY ABLATION TABLE")
print("=" * 120)

display(
    ablation_bootstrap_compact_df.round(6)
)


# ------------------------------------------------------------
# Save both raw and summary outputs
# ------------------------------------------------------------

bootstrap_ablation_df.to_parquet(
    "ROSE_ABLATION_BOOTSTRAP_RAW.parquet",
    index=False
)

ablation_bootstrap_summary_df.to_csv(
    "ROSE_ABLATION_BOOTSTRAP_SUMMARY.csv",
    index=False
)

ablation_bootstrap_compact_df.to_csv(
    "ROSE_ABLATION_BOOTSTRAP_COMPACT.csv",
    index=False
)


print("\nSaved:")
print("ROSE_ABLATION_BOOTSTRAP_RAW.parquet")
print("ROSE_ABLATION_BOOTSTRAP_SUMMARY.csv")
print("ROSE_ABLATION_BOOTSTRAP_COMPACT.csv")


print("\n" + "=" * 120)
print("✅ STEP 7B PASSED")
print("=" * 120)

PAIRED DOCUMENT-BOOTSTRAP ABLATION RESULTS
Positive delta = Full model performs better


,Model,Delta_Pearson_Observed,Delta_Pearson_BootstrapMean,Delta_Pearson_CI_Lower,Delta_Pearson_CI_Upper,Delta_Pearson_CI_Excludes_Zero
0,Full - NLI,-0.001034,-0.001031,-0.002841,0.000733,False
1,Lexical + Semantic,0.045867,0.046126,0.030573,0.062481,True
2,Full - Structural,0.047539,0.047795,0.032284,0.064166,True
3,Full - Reference Similarity,0.500401,0.499768,0.447450,0.553073,True



COMPACT PRIMARY ABLATION TABLE


,Model,Pearson Δ,Pearson 95% CI,Spearman Δ,Spearman 95% CI,Kendall Δ,Kendall 95% CI,MAE Δ,MAE 95% CI,CCC Δ,CCC 95% CI
0,Full - NLI,-0.001034,"[-0.0028, 0.0007]",-0.001882,"[-0.0042, 0.0004]",-0.001540,"[-0.0035, 0.0005]",-0.000304,"[-0.0007, 0.0001]",-0.000733,"[-0.0025, 0.0011]"
1,Lexical + Semantic,0.045867,"[0.0306, 0.0625]",0.045716,"[0.0282, 0.0639]",0.039732,"[0.0248, 0.0546]",0.007536,"[0.0049, 0.0101]",0.058805,"[0.0434, 0.0753]"
2,Full - Structural,0.047539,"[0.0323, 0.0642]",0.048549,"[0.0310, 0.0670]",0.041920,"[0.0273, 0.0568]",0.007777,"[0.0053, 0.0103]",0.059980,"[0.0446, 0.0763]"
3,Full - Reference Similarity,0.500401,"[0.4475, 0.5531]",0.489799,"[0.4380, 0.5412]",0.367137,"[0.3288, 0.4048]",0.052572,"[0.0450, 0.0603]",0.556377,"[0.5139, 0.5959]"



Saved:
ROSE_ABLATION_BOOTSTRAP_RAW.parquet
ROSE_ABLATION_BOOTSTRAP_SUMMARY.csv
ROSE_ABLATION_BOOTSTRAP_COMPACT.csv

✅ STEP 7B PASSED


In [184]:
# ============================================================
# STEP 8A: Direct 2x2 contribution analysis
#
# L = Lexical
# S = Semantic
# F = Factuality / NLI
# T = Structural
# ============================================================

import numpy as np
import pandas as pd

y_true = (
    oof_ablation_df["human_content_coverage"]
    .astype(float)
    .to_numpy()
)

PRED = {
    "LS":
        oof_ablation_df[
            "pred_lexical_plus_semantic"
        ].to_numpy(),

    "LSF":
        oof_ablation_df[
            "pred_full_minus_structural"
        ].to_numpy(),

    "LST":
        oof_ablation_df[
            "pred_full_minus_nli"
        ].to_numpy(),

    "LSFT":
        oof_ablation_df[
            "pred_full"
        ].to_numpy()
}


def get_metrics(y, pred):

    return {
        "Pearson":
            pearsonr(y, pred).statistic,

        "Spearman":
            spearmanr(y, pred).statistic,

        "Kendall":
            kendalltau(y, pred).statistic,

        "MAE":
            mean_absolute_error(y, pred),

        "RMSE":
            np.sqrt(
                mean_squared_error(y, pred)
            ),

        "CCC":
            concordance_correlation_coefficient(
                y,
                pred
            )
    }


# ------------------------------------------------------------
# Calculate model-level metrics
# ------------------------------------------------------------

factorial_metrics = {
    name: get_metrics(
        y_true,
        pred
    )
    for name, pred in PRED.items()
}


factorial_results_df = pd.DataFrame([
    {
        "Model": name,
        **metrics
    }
    for name, metrics
    in factorial_metrics.items()
])


model_labels = {
    "LS": "Lexical + Semantic",
    "LSF": "Lexical + Semantic + Factuality",
    "LST": "Lexical + Semantic + Structural",
    "LSFT": "Full: Lexical + Semantic + Factuality + Structural"
}

factorial_results_df[
    "Description"
] = factorial_results_df[
    "Model"
].map(model_labels)


factorial_results_df = factorial_results_df[
    [
        "Model",
        "Description",
        "Pearson",
        "Spearman",
        "Kendall",
        "MAE",
        "RMSE",
        "CCC"
    ]
]


print("=" * 110)
print("2x2 FEATURE-FAMILY CONFIGURATION RESULTS")
print("=" * 110)

display(
    factorial_results_df.round(6)
)


# ------------------------------------------------------------
# Define direct feature-family contrasts
#
# For correlation / CCC:
# positive = adding feature family improves performance
#
# For MAE/RMSE:
# positive = adding feature family reduces error
# ------------------------------------------------------------

CONTRASTS = {

    "NLI effect without Structural":
        ("LSF", "LS"),

    "NLI effect with Structural":
        ("LSFT", "LST"),

    "Structural effect without NLI":
        ("LST", "LS"),

    "Structural effect with NLI":
        ("LSFT", "LSF")
}


contrast_rows = []

for contrast_name, (with_model, without_model) \
        in CONTRASTS.items():

    m_with = factorial_metrics[with_model]
    m_without = factorial_metrics[without_model]

    contrast_rows.append({

        "Contrast": contrast_name,

        "Delta_Pearson":
            m_with["Pearson"]
            - m_without["Pearson"],

        "Delta_Spearman":
            m_with["Spearman"]
            - m_without["Spearman"],

        "Delta_Kendall":
            m_with["Kendall"]
            - m_without["Kendall"],

        # Positive = lower error after adding feature
        "Delta_MAE_Improvement":
            m_without["MAE"]
            - m_with["MAE"],

        "Delta_RMSE_Improvement":
            m_without["RMSE"]
            - m_with["RMSE"],

        "Delta_CCC":
            m_with["CCC"]
            - m_without["CCC"]
    })


factorial_contrasts_df = pd.DataFrame(
    contrast_rows
)


print("\n" + "=" * 110)
print("OBSERVED FEATURE-FAMILY CONTRIBUTIONS")
print("Positive value = adding that feature family improves performance")
print("=" * 110)

display(
    factorial_contrasts_df.round(6)
)


factorial_results_df.to_csv(
    "ROSE_ABLATION_FACTORIAL_MODELS.csv",
    index=False
)

factorial_contrasts_df.to_csv(
    "ROSE_ABLATION_FACTORIAL_CONTRASTS.csv",
    index=False
)

print("\nSaved:")
print("ROSE_ABLATION_FACTORIAL_MODELS.csv")
print("ROSE_ABLATION_FACTORIAL_CONTRASTS.csv")

print("\n✅ STEP 8A PASSED")

2x2 FEATURE-FAMILY CONFIGURATION RESULTS


,Model,Description,Pearson,Spearman,Kendall,MAE,RMSE,CCC
0,LS,Lexical + Semantic,0.639190,0.630760,0.453353,0.144354,0.182031,0.576321
1,LSF,Lexical + Semantic + Factuality,0.637517,0.627926,0.451166,0.144595,0.182353,0.575146
2,LST,Lexical + Semantic + Structural,0.686090,0.678357,0.494626,0.136514,0.172212,0.635859
3,LSFT,Full: Lexical + Semantic + Factuality + Struct...,0.685056,0.676475,0.493086,0.136818,0.172436,0.635126



OBSERVED FEATURE-FAMILY CONTRIBUTIONS
Positive value = adding that feature family improves performance


,Contrast,Delta_Pearson,Delta_Spearman,Delta_Kendall,Delta_MAE_Improvement,Delta_RMSE_Improvement,Delta_CCC
0,NLI effect without Structural,-0.001672,-0.002833,-0.002187,-0.000241,-0.000322,-0.001175
1,NLI effect with Structural,-0.001034,-0.001882,-0.001540,-0.000304,-0.000224,-0.000733
2,Structural effect without NLI,0.046901,0.047597,0.041272,0.007841,0.009820,0.059538
3,Structural effect with NLI,0.047539,0.048549,0.041920,0.007777,0.009917,0.059980



Saved:
ROSE_ABLATION_FACTORIAL_MODELS.csv
ROSE_ABLATION_FACTORIAL_CONTRASTS.csv

✅ STEP 8A PASSED


In [186]:
# ============================================================
# STEP 8B: Document-bootstrap direct feature contributions
# ============================================================

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

doc_ids = np.unique(
    oof_ablation_df["doc_id"]
)

doc_to_rows = {
    doc: np.flatnonzero(
        oof_ablation_df["doc_id"]
        .to_numpy() == doc
    )
    for doc in doc_ids
}

bootstrap_contrast_records = []


print("=" * 100)
print("BOOTSTRAPPING DIRECT FEATURE-FAMILY CONTRIBUTIONS")
print("=" * 100)


for b in range(N_BOOTSTRAP):

    sampled_docs = rng.choice(
        doc_ids,
        size=len(doc_ids),
        replace=True
    )

    idx = np.concatenate([
        doc_to_rows[d]
        for d in sampled_docs
    ])

    y_b = y_true[idx]


    # Calculate metrics for the 4 factorial models
    boot_metrics = {}

    for model_name, predictions in PRED.items():

        boot_metrics[
            model_name
        ] = get_metrics(
            y_b,
            predictions[idx]
        )


    for contrast_name, (
        with_model,
        without_model
    ) in CONTRASTS.items():

        m_with = boot_metrics[
            with_model
        ]

        m_without = boot_metrics[
            without_model
        ]

        bootstrap_contrast_records.append({

            "bootstrap":
                b + 1,

            "Contrast":
                contrast_name,

            "Delta_Pearson":
                m_with["Pearson"]
                - m_without["Pearson"],

            "Delta_Spearman":
                m_with["Spearman"]
                - m_without["Spearman"],

            "Delta_Kendall":
                m_with["Kendall"]
                - m_without["Kendall"],

            "Delta_MAE_Improvement":
                m_without["MAE"]
                - m_with["MAE"],

            "Delta_RMSE_Improvement":
                m_without["RMSE"]
                - m_with["RMSE"],

            "Delta_CCC":
                m_with["CCC"]
                - m_without["CCC"]
        })


    if (b + 1) % 500 == 0:

        print(
            f"  completed "
            f"{b + 1:,} / "
            f"{N_BOOTSTRAP:,}"
        )


factorial_bootstrap_df = pd.DataFrame(
    bootstrap_contrast_records
)

assert len(
    factorial_bootstrap_df
) == 20000


print("\nBootstrap rows:",
      len(factorial_bootstrap_df))

print("\n✅ Bootstrap generation complete.")

BOOTSTRAPPING DIRECT FEATURE-FAMILY CONTRIBUTIONS
  completed 500 / 5,000
  completed 1,000 / 5,000
  completed 1,500 / 5,000
  completed 2,000 / 5,000
  completed 2,500 / 5,000
  completed 3,000 / 5,000
  completed 3,500 / 5,000
  completed 4,000 / 5,000
  completed 4,500 / 5,000
  completed 5,000 / 5,000

Bootstrap rows: 20000

✅ Bootstrap generation complete.


In [188]:
# ============================================================
# STEP 8C: Summarise 95% bootstrap CIs
# ============================================================

observed_contrast_lookup = (
    factorial_contrasts_df
    .set_index("Contrast")
)

METRICS = [
    "Delta_Pearson",
    "Delta_Spearman",
    "Delta_Kendall",
    "Delta_MAE_Improvement",
    "Delta_RMSE_Improvement",
    "Delta_CCC"
]


summary_rows = []


for contrast in CONTRASTS.keys():

    subset = factorial_bootstrap_df[
        factorial_bootstrap_df[
            "Contrast"
        ] == contrast
    ]

    row = {
        "Contrast": contrast
    }


    for metric in METRICS:

        values = subset[
            metric
        ].to_numpy()

        observed = (
            observed_contrast_lookup
            .loc[contrast, metric]
        )

        low = np.percentile(
            values,
            2.5
        )

        high = np.percentile(
            values,
            97.5
        )

        row[
            f"{metric}_Observed"
        ] = observed

        row[
            f"{metric}_CI_Lower"
        ] = low

        row[
            f"{metric}_CI_Upper"
        ] = high

        row[
            f"{metric}_CI_Excludes_Zero"
        ] = (
            low > 0
            or
            high < 0
        )


    summary_rows.append(row)


factorial_bootstrap_summary_df = pd.DataFrame(
    summary_rows
)


# ------------------------------------------------------------
# Thesis-friendly compact output
# ------------------------------------------------------------

compact_rows = []

for _, r in factorial_bootstrap_summary_df.iterrows():

    compact_rows.append({

        "Contrast":
            r["Contrast"],

        "Pearson Δ":
            r[
                "Delta_Pearson_Observed"
            ],

        "Pearson 95% CI":
            (
                f"[{r['Delta_Pearson_CI_Lower']:.4f}, "
                f"{r['Delta_Pearson_CI_Upper']:.4f}]"
            ),

        "Spearman Δ":
            r[
                "Delta_Spearman_Observed"
            ],

        "Spearman 95% CI":
            (
                f"[{r['Delta_Spearman_CI_Lower']:.4f}, "
                f"{r['Delta_Spearman_CI_Upper']:.4f}]"
            ),

        "Kendall Δ":
            r[
                "Delta_Kendall_Observed"
            ],

        "Kendall 95% CI":
            (
                f"[{r['Delta_Kendall_CI_Lower']:.4f}, "
                f"{r['Delta_Kendall_CI_Upper']:.4f}]"
            ),

        "MAE improvement":
            r[
                "Delta_MAE_Improvement_Observed"
            ],

        "MAE 95% CI":
            (
                f"[{r['Delta_MAE_Improvement_CI_Lower']:.4f}, "
                f"{r['Delta_MAE_Improvement_CI_Upper']:.4f}]"
            ),

        "CCC Δ":
            r[
                "Delta_CCC_Observed"
            ],

        "CCC 95% CI":
            (
                f"[{r['Delta_CCC_CI_Lower']:.4f}, "
                f"{r['Delta_CCC_CI_Upper']:.4f}]"
            )
    })


factorial_bootstrap_compact_df = pd.DataFrame(
    compact_rows
)


print("=" * 120)
print("DIRECT FEATURE-FAMILY CONTRIBUTION WITH 95% DOCUMENT-BOOTSTRAP CI")
print("Positive = adding the feature family improves performance")
print("=" * 120)

display(
    factorial_bootstrap_compact_df.round(6)
)


factorial_bootstrap_df.to_parquet(
    "ROSE_ABLATION_FACTORIAL_BOOTSTRAP_RAW.parquet",
    index=False
)

factorial_bootstrap_summary_df.to_csv(
    "ROSE_ABLATION_FACTORIAL_BOOTSTRAP_SUMMARY.csv",
    index=False
)

factorial_bootstrap_compact_df.to_csv(
    "ROSE_ABLATION_FACTORIAL_BOOTSTRAP_COMPACT.csv",
    index=False
)


print("\nSaved:")
print("ROSE_ABLATION_FACTORIAL_BOOTSTRAP_RAW.parquet")
print("ROSE_ABLATION_FACTORIAL_BOOTSTRAP_SUMMARY.csv")
print("ROSE_ABLATION_FACTORIAL_BOOTSTRAP_COMPACT.csv")

print("\n✅ STEP 8 COMPLETE")

DIRECT FEATURE-FAMILY CONTRIBUTION WITH 95% DOCUMENT-BOOTSTRAP CI
Positive = adding the feature family improves performance


,Contrast,Pearson Δ,Pearson 95% CI,Spearman Δ,Spearman 95% CI,Kendall Δ,Kendall 95% CI,MAE improvement,MAE 95% CI,CCC Δ,CCC 95% CI
0,NLI effect without Structural,-0.001672,"[-0.0038, 0.0004]",-0.002833,"[-0.0054, -0.0003]",-0.002187,"[-0.0042, -0.0001]",-0.000241,"[-0.0006, 0.0002]",-0.001175,"[-0.0033, 0.0010]"
1,NLI effect with Structural,-0.001034,"[-0.0028, 0.0007]",-0.001882,"[-0.0042, 0.0004]",-0.001540,"[-0.0035, 0.0005]",-0.000304,"[-0.0007, 0.0001]",-0.000733,"[-0.0025, 0.0011]"
2,Structural effect without NLI,0.046901,"[0.0316, 0.0636]",0.047597,"[0.0300, 0.0659]",0.041272,"[0.0265, 0.0562]",0.007841,"[0.0053, 0.0104]",0.059538,"[0.0442, 0.0759]"
3,Structural effect with NLI,0.047539,"[0.0323, 0.0642]",0.048549,"[0.0310, 0.0670]",0.041920,"[0.0273, 0.0568]",0.007777,"[0.0053, 0.0103]",0.059980,"[0.0446, 0.0763]"



Saved:
ROSE_ABLATION_FACTORIAL_BOOTSTRAP_RAW.parquet
ROSE_ABLATION_FACTORIAL_BOOTSTRAP_SUMMARY.csv
ROSE_ABLATION_FACTORIAL_BOOTSTRAP_COMPACT.csv

✅ STEP 8 COMPLETE


## 15. Holdout Correlation Bootstrap Confidence Intervals and Significance Testing

This section quantifies uncertainty in the main RoSE holdout correlation results using document-level bootstrap resampling.

The frozen 100-document holdout partition is recovered and 5,000 document-level bootstrap samples are used to estimate 95% confidence intervals for Pearson, Spearman, and Kendall correlations. Paired bootstrap differences are then used to compare the Learned Hybrid metric with each baseline while preserving the same bootstrap replicates across methods.

In [190]:
# ============================================================
# STEP 9A: Recover and verify frozen RoSE final holdout
# for correlation bootstrap confidence intervals
# ============================================================

import os
import numpy as np
import pandas as pd

HOLDOUT_FILE = "ROSE_FINAL_HOLDOUT_EVALUATED.parquet"

assert os.path.exists(HOLDOUT_FILE), \
    f"Could not find {HOLDOUT_FILE}"

rose_corr_holdout_df = pd.read_parquet(
    HOLDOUT_FILE
)

print("=" * 90)
print("FROZEN RoSE FINAL HOLDOUT")
print("=" * 90)

print("Shape:", rose_corr_holdout_df.shape)
print(
    "Unique documents:",
    rose_corr_holdout_df["doc_id"].nunique()
)
print(
    "Unique summaries:",
    rose_corr_holdout_df["summary_id"].nunique()
)


# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

required_columns = [
    "summary_id",
    "doc_id",
    "human_content_coverage",
    "rougeL",
    "bertScoreF1",
    "moverScore",
    "manual_hybrid_raw",
    "learned_hybrid_prediction"
]

missing = [
    c for c in required_columns
    if c not in rose_corr_holdout_df.columns
]

print("\nMissing required columns:", missing)

assert len(missing) == 0


# ------------------------------------------------------------
# Confirm original holdout composition
# ------------------------------------------------------------

summaries_per_doc = (
    rose_corr_holdout_df
    .groupby("doc_id")
    .size()
)

print("\nSummaries per document:")
print(
    summaries_per_doc
    .value_counts()
    .sort_index()
)

assert len(rose_corr_holdout_df) == 1200
assert rose_corr_holdout_df["doc_id"].nunique() == 100
assert rose_corr_holdout_df["summary_id"].nunique() == 1200
assert (summaries_per_doc == 12).all()


# ------------------------------------------------------------
# Missing / finite checks
# ------------------------------------------------------------

evaluation_columns = [
    "human_content_coverage",
    "rougeL",
    "bertScoreF1",
    "moverScore",
    "manual_hybrid_raw",
    "learned_hybrid_prediction"
]

print("\n" + "=" * 90)
print("MISSING VALUE CHECK")
print("=" * 90)

print(
    rose_corr_holdout_df[
        evaluation_columns
    ].isna().sum()
)

assert (
    rose_corr_holdout_df[
        evaluation_columns
    ].isna().sum() == 0
).all()


finite = np.isfinite(
    rose_corr_holdout_df[
        evaluation_columns
    ].astype(float).to_numpy()
).all()

print("\nAll evaluation values finite:", finite)

assert finite


print("\n" + "=" * 90)
print("✅ STEP 9A PASSED")
print("Frozen 100-document RoSE holdout recovered correctly.")
print("No modelling or tuning has been performed.")
print("=" * 90)

FROZEN RoSE FINAL HOLDOUT
Shape: (1200, 63)
Unique documents: 100
Unique summaries: 1200

Missing required columns: []

Summaries per document:
12    100
Name: count, dtype: int64

MISSING VALUE CHECK
human_content_coverage       0
rougeL                       0
bertScoreF1                  0
moverScore                   0
manual_hybrid_raw            0
learned_hybrid_prediction    0
dtype: int64

All evaluation values finite: True

✅ STEP 9A PASSED
Frozen 100-document RoSE holdout recovered correctly.
No modelling or tuning has been performed.


In [192]:
# ============================================================
# STEP 9B: Document-bootstrap 95% CIs for main RoSE
# holdout correlations
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr, kendalltau


# ------------------------------------------------------------
# 1. Bootstrap specification
# ------------------------------------------------------------

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


# ------------------------------------------------------------
# 2. Define the five main evaluation methods
#
# IMPORTANT:
# Raw baseline scores are used for correlation/ranking.
# Calibrated predictions are NOT used here.
# ------------------------------------------------------------

CORRELATION_METHODS = {
    "Learned Hybrid":
        "learned_hybrid_prediction",

    "BERTScore F1":
        "bertScoreF1",

    "MoverScore":
        "moverScore",

    "ROUGE-L":
        "rougeL",

    "Manual Hybrid":
        "manual_hybrid_raw"
}

TARGET = "human_content_coverage"


# ------------------------------------------------------------
# 3. Prepare arrays
# ------------------------------------------------------------

y_all = (
    rose_corr_holdout_df[TARGET]
    .astype(float)
    .to_numpy()
)

doc_id_array = (
    rose_corr_holdout_df["doc_id"]
    .to_numpy()
)

unique_docs = np.unique(
    doc_id_array
)

n_docs = len(unique_docs)


# Map each document to its 12 row positions
doc_to_indices = {
    doc_id: np.flatnonzero(
        doc_id_array == doc_id
    )
    for doc_id in unique_docs
}


# ------------------------------------------------------------
# 4. Strong document checks
# ------------------------------------------------------------

assert n_docs == 100

assert all(
    len(indices) == 12
    for indices in doc_to_indices.values()
)


method_arrays = {
    method:
        rose_corr_holdout_df[column]
        .astype(float)
        .to_numpy()

    for method, column
    in CORRELATION_METHODS.items()
}


print("=" * 100)
print("RoSE HOLDOUT CORRELATION BOOTSTRAP")
print("=" * 100)

print("Holdout documents:", n_docs)
print("Holdout summaries:", len(y_all))
print("Methods:", len(CORRELATION_METHODS))
print("Bootstrap replicates:", N_BOOTSTRAP)
print("Random seed:", BOOTSTRAP_SEED)
print("Resampling unit: source document")
print("Sampling: with replacement")


# ------------------------------------------------------------
# 5. Correlation helper
# ------------------------------------------------------------

def calculate_correlations(y_true, scores):

    return {
        "Pearson":
            pearsonr(
                y_true,
                scores
            ).statistic,

        "Spearman":
            spearmanr(
                y_true,
                scores
            ).statistic,

        "Kendall":
            kendalltau(
                y_true,
                scores
            ).statistic
    }


# ------------------------------------------------------------
# 6. First reproduce the OBSERVED holdout correlations
# ------------------------------------------------------------

observed_rows = []

for method, scores in method_arrays.items():

    metrics = calculate_correlations(
        y_all,
        scores
    )

    observed_rows.append({
        "Method": method,
        **metrics
    })


observed_correlation_df = pd.DataFrame(
    observed_rows
)


# Keep your standard reporting order
method_order = list(
    CORRELATION_METHODS.keys()
)

observed_correlation_df["Method"] = pd.Categorical(
    observed_correlation_df["Method"],
    categories=method_order,
    ordered=True
)

observed_correlation_df = (
    observed_correlation_df
    .sort_values("Method")
    .reset_index(drop=True)
)


print("\n" + "=" * 100)
print("OBSERVED FROZEN HOLDOUT CORRELATIONS")
print("=" * 100)

display(
    observed_correlation_df.round(6)
)


# ------------------------------------------------------------
# 7. Run paired document bootstrap
# ------------------------------------------------------------

bootstrap_records = []

print("\nRunning 5,000 document-bootstrap replicates...")


for b in range(N_BOOTSTRAP):

    # Sample 100 document IDs WITH replacement
    sampled_docs = rng.choice(
        unique_docs,
        size=n_docs,
        replace=True
    )


    # Preserve multiplicity:
    # if a document is sampled twice,
    # all 12 summaries appear twice
    sampled_indices = np.concatenate([
        doc_to_indices[doc_id]
        for doc_id in sampled_docs
    ])


    y_b = y_all[
        sampled_indices
    ]


    # Same sampled documents for every method
    for method, scores_all in method_arrays.items():

        scores_b = scores_all[
            sampled_indices
        ]

        metrics_b = calculate_correlations(
            y_b,
            scores_b
        )

        bootstrap_records.append({
            "bootstrap": b + 1,
            "Method": method,
            "Pearson": metrics_b["Pearson"],
            "Spearman": metrics_b["Spearman"],
            "Kendall": metrics_b["Kendall"]
        })


    if (b + 1) % 500 == 0:
        print(
            f"  completed {b + 1:,} / "
            f"{N_BOOTSTRAP:,}"
        )


correlation_bootstrap_raw_df = pd.DataFrame(
    bootstrap_records
)


# ------------------------------------------------------------
# 8. Completion checks
# ------------------------------------------------------------

expected_rows = (
    N_BOOTSTRAP
    * len(CORRELATION_METHODS)
)

print("\nBootstrap rows:",
      len(correlation_bootstrap_raw_df))

print("Expected rows:",
      expected_rows)

assert (
    len(correlation_bootstrap_raw_df)
    == expected_rows
)

assert (
    correlation_bootstrap_raw_df[
        ["Pearson", "Spearman", "Kendall"]
    ]
    .notna()
    .all()
    .all()
)


# ------------------------------------------------------------
# 9. Summarise percentile 95% confidence intervals
# ------------------------------------------------------------

summary_rows = []

observed_lookup = (
    observed_correlation_df
    .set_index("Method")
)


for method in method_order:

    subset = (
        correlation_bootstrap_raw_df[
            correlation_bootstrap_raw_df[
                "Method"
            ] == method
        ]
    )

    row = {
        "Method": method
    }


    for metric in [
        "Pearson",
        "Spearman",
        "Kendall"
    ]:

        values = (
            subset[metric]
            .to_numpy()
        )

        observed = float(
            observed_lookup.loc[
                method,
                metric
            ]
        )

        row[
            f"{metric}_Observed"
        ] = observed

        row[
            f"{metric}_BootstrapMean"
        ] = np.mean(values)

        row[
            f"{metric}_CI_Lower"
        ] = np.percentile(
            values,
            2.5
        )

        row[
            f"{metric}_CI_Upper"
        ] = np.percentile(
            values,
            97.5
        )


    summary_rows.append(row)


correlation_bootstrap_summary_df = pd.DataFrame(
    summary_rows
)


# ------------------------------------------------------------
# 10. Thesis-friendly compact table
# ------------------------------------------------------------

compact_rows = []

for _, row in correlation_bootstrap_summary_df.iterrows():

    compact_rows.append({

        "Method":
            row["Method"],

        "Pearson":
            row["Pearson_Observed"],

        "Pearson 95% CI":
            (
                f"[{row['Pearson_CI_Lower']:.4f}, "
                f"{row['Pearson_CI_Upper']:.4f}]"
            ),

        "Spearman":
            row["Spearman_Observed"],

        "Spearman 95% CI":
            (
                f"[{row['Spearman_CI_Lower']:.4f}, "
                f"{row['Spearman_CI_Upper']:.4f}]"
            ),

        "Kendall":
            row["Kendall_Observed"],

        "Kendall 95% CI":
            (
                f"[{row['Kendall_CI_Lower']:.4f}, "
                f"{row['Kendall_CI_Upper']:.4f}]"
            )
    })


correlation_bootstrap_compact_df = pd.DataFrame(
    compact_rows
)


print("\n" + "=" * 115)
print("RoSE HOLDOUT CORRELATIONS WITH")
print("95% DOCUMENT-BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 115)

display(
    correlation_bootstrap_compact_df.round(6)
)


# ------------------------------------------------------------
# 11. Compare bootstrap means with observed values
# Diagnostic only
# ------------------------------------------------------------

print("\n" + "=" * 115)
print("BOOTSTRAP MEAN SANITY CHECK")
print("=" * 115)

sanity_rows = []

for _, row in correlation_bootstrap_summary_df.iterrows():

    sanity_rows.append({

        "Method": row["Method"],

        "Observed Pearson":
            row["Pearson_Observed"],

        "Bootstrap Mean Pearson":
            row["Pearson_BootstrapMean"],

        "Difference":
            row["Pearson_BootstrapMean"]
            - row["Pearson_Observed"]
    })


bootstrap_sanity_df = pd.DataFrame(
    sanity_rows
)

display(
    bootstrap_sanity_df.round(6)
)


# ------------------------------------------------------------
# 12. Save permanent results
# ------------------------------------------------------------

observed_correlation_df.to_csv(
    "ROSE_HOLDOUT_CORRELATIONS_OBSERVED.csv",
    index=False
)

correlation_bootstrap_raw_df.to_parquet(
    "ROSE_HOLDOUT_CORRELATION_BOOTSTRAP_RAW.parquet",
    index=False
)

correlation_bootstrap_summary_df.to_csv(
    "ROSE_HOLDOUT_CORRELATION_BOOTSTRAP_SUMMARY.csv",
    index=False
)

correlation_bootstrap_compact_df.to_csv(
    "ROSE_HOLDOUT_CORRELATION_BOOTSTRAP_COMPACT.csv",
    index=False
)


print("\nSaved:")
print("ROSE_HOLDOUT_CORRELATIONS_OBSERVED.csv")
print("ROSE_HOLDOUT_CORRELATION_BOOTSTRAP_RAW.parquet")
print("ROSE_HOLDOUT_CORRELATION_BOOTSTRAP_SUMMARY.csv")
print("ROSE_HOLDOUT_CORRELATION_BOOTSTRAP_COMPACT.csv")


print("\n" + "=" * 115)
print("✅ STEP 9B PASSED")
print("Document-bootstrap correlation CIs calculated.")
print("No model fitting, tuning, or holdout modification occurred.")
print("=" * 115)

RoSE HOLDOUT CORRELATION BOOTSTRAP
Holdout documents: 100
Holdout summaries: 1200
Methods: 5
Bootstrap replicates: 5000
Random seed: 42
Resampling unit: source document
Sampling: with replacement

OBSERVED FROZEN HOLDOUT CORRELATIONS


,Method,Pearson,Spearman,Kendall
0,Learned Hybrid,0.708710,0.702939,0.522724
1,BERTScore F1,0.622253,0.608035,0.440607
2,MoverScore,0.573845,0.594102,0.425974
3,ROUGE-L,0.565328,0.567517,0.407581
4,Manual Hybrid,0.461034,0.462555,0.322882



Running 5,000 document-bootstrap replicates...
  completed 500 / 5,000
  completed 1,000 / 5,000
  completed 1,500 / 5,000
  completed 2,000 / 5,000
  completed 2,500 / 5,000
  completed 3,000 / 5,000
  completed 3,500 / 5,000
  completed 4,000 / 5,000
  completed 4,500 / 5,000
  completed 5,000 / 5,000

Bootstrap rows: 25000
Expected rows: 25000

RoSE HOLDOUT CORRELATIONS WITH
95% DOCUMENT-BOOTSTRAP CONFIDENCE INTERVALS


,Method,Pearson,Pearson 95% CI,Spearman,Spearman 95% CI,Kendall,Kendall 95% CI
0,Learned Hybrid,0.708710,"[0.6432, 0.7651]",0.702939,"[0.6291, 0.7652]",0.522724,"[0.4620, 0.5789]"
1,BERTScore F1,0.622253,"[0.5400, 0.6948]",0.608035,"[0.5184, 0.6872]",0.440607,"[0.3698, 0.5083]"
2,MoverScore,0.573845,"[0.4743, 0.6607]",0.594102,"[0.4908, 0.6808]",0.425974,"[0.3485, 0.4961]"
3,ROUGE-L,0.565328,"[0.4736, 0.6464]",0.567517,"[0.4694, 0.6529]",0.407581,"[0.3326, 0.4774]"
4,Manual Hybrid,0.461034,"[0.3695, 0.5438]",0.462555,"[0.3674, 0.5483]",0.322882,"[0.2534, 0.3893]"



BOOTSTRAP MEAN SANITY CHECK


,Method,Observed Pearson,Bootstrap Mean Pearson,Difference
0,Learned Hybrid,0.708710,0.707712,-0.000998
1,BERTScore F1,0.622253,0.620823,-0.001430
2,MoverScore,0.573845,0.573224,-0.000621
3,ROUGE-L,0.565328,0.564537,-0.000792
4,Manual Hybrid,0.461034,0.458963,-0.002072



Saved:
ROSE_HOLDOUT_CORRELATIONS_OBSERVED.csv
ROSE_HOLDOUT_CORRELATION_BOOTSTRAP_RAW.parquet
ROSE_HOLDOUT_CORRELATION_BOOTSTRAP_SUMMARY.csv
ROSE_HOLDOUT_CORRELATION_BOOTSTRAP_COMPACT.csv

✅ STEP 9B PASSED
Document-bootstrap correlation CIs calculated.
No model fitting, tuning, or holdout modification occurred.


In [194]:
# ============================================================
# STEP 9C: Paired document-bootstrap differences
# Learned Hybrid vs each baseline
#
# Reuses the 5,000 bootstrap samples from Step 9B.
# NO new bootstrap sampling is performed.
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Comparison specification
# ------------------------------------------------------------

LEARNED_METHOD = "Learned Hybrid"

BASELINE_METHODS = [
    "BERTScore F1",
    "MoverScore",
    "ROUGE-L",
    "Manual Hybrid"
]

CORRELATION_METRICS = [
    "Pearson",
    "Spearman",
    "Kendall"
]

N_BOOTSTRAP_EXPECTED = 5000


# ------------------------------------------------------------
# 2. Validate raw bootstrap table
# ------------------------------------------------------------

print("=" * 105)
print("PAIRED CORRELATION BOOTSTRAP DIFFERENCE ANALYSIS")
print("=" * 105)

print(
    "Raw bootstrap rows:",
    len(correlation_bootstrap_raw_df)
)

print(
    "Unique bootstrap replicates:",
    correlation_bootstrap_raw_df[
        "bootstrap"
    ].nunique()
)

print(
    "Methods:",
    correlation_bootstrap_raw_df[
        "Method"
    ].nunique()
)

assert (
    correlation_bootstrap_raw_df[
        "bootstrap"
    ].nunique()
    == N_BOOTSTRAP_EXPECTED
)

assert set(
    correlation_bootstrap_raw_df[
        "Method"
    ].unique()
) == set(CORRELATION_METHODS.keys())


# ------------------------------------------------------------
# 3. Observed correlation lookup
# ------------------------------------------------------------

observed_corr_lookup = (
    observed_correlation_df
    .set_index("Method")
)


# ------------------------------------------------------------
# 4. Holm multiple-comparison correction helper
# ------------------------------------------------------------

def holm_adjust(p_values):
    """
    Holm-Bonferroni adjusted p-values.
    Preserves original ordering.
    """

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    m = len(p_values)

    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.empty(
        m,
        dtype=float
    )

    running_max = 0.0

    for i, p in enumerate(sorted_p):

        adjusted = (
            (m - i) * p
        )

        running_max = max(
            running_max,
            adjusted
        )

        adjusted_sorted[i] = min(
            running_max,
            1.0
        )


    adjusted = np.empty(
        m,
        dtype=float
    )

    adjusted[order] = adjusted_sorted

    return adjusted


# ------------------------------------------------------------
# 5. Build paired bootstrap delta distributions
# ------------------------------------------------------------

paired_delta_records = []


for metric in CORRELATION_METRICS:

    # --------------------------------------------------------
    # Pivot so each bootstrap replicate has one row and
    # each method has one column.
    # --------------------------------------------------------

    metric_wide = (
        correlation_bootstrap_raw_df
        .pivot(
            index="bootstrap",
            columns="Method",
            values=metric
        )
        .sort_index()
    )


    assert len(metric_wide) == 5000
    assert metric_wide.notna().all().all()


    learned_boot = (
        metric_wide[
            LEARNED_METHOD
        ]
        .to_numpy()
    )


    for baseline in BASELINE_METHODS:

        baseline_boot = (
            metric_wide[
                baseline
            ]
            .to_numpy()
        )


        # ----------------------------------------------------
        # Positive delta = Learned Hybrid better
        # ----------------------------------------------------

        delta_boot = (
            learned_boot
            - baseline_boot
        )


        # ----------------------------------------------------
        # Observed point-estimate difference
        # ----------------------------------------------------

        observed_delta = (
            float(
                observed_corr_lookup
                .loc[
                    LEARNED_METHOD,
                    metric
                ]
            )
            -
            float(
                observed_corr_lookup
                .loc[
                    baseline,
                    metric
                ]
            )
        )


        # ----------------------------------------------------
        # Percentile confidence interval
        # ----------------------------------------------------

        ci_lower = np.percentile(
            delta_boot,
            2.5
        )

        ci_upper = np.percentile(
            delta_boot,
            97.5
        )


        # ----------------------------------------------------
        # Empirical two-sided bootstrap p-value
        #
        # +1 correction prevents reporting p = 0.
        # ----------------------------------------------------

        count_nonpositive = np.sum(
            delta_boot <= 0
        )

        count_nonnegative = np.sum(
            delta_boot >= 0
        )

        lower_tail = (
            count_nonpositive + 1
        ) / (
            len(delta_boot) + 1
        )

        upper_tail = (
            count_nonnegative + 1
        ) / (
            len(delta_boot) + 1
        )

        p_raw = min(
            1.0,
            2 * min(
                lower_tail,
                upper_tail
            )
        )


        paired_delta_records.append({

            "Metric":
                metric,

            "Comparison":
                f"Learned Hybrid vs {baseline}",

            "Baseline":
                baseline,

            "Observed_Delta":
                observed_delta,

            "Bootstrap_Mean_Delta":
                np.mean(delta_boot),

            "CI_Lower":
                ci_lower,

            "CI_Upper":
                ci_upper,

            "CI_Excludes_Zero":
                (
                    ci_lower > 0
                    or
                    ci_upper < 0
                ),

            "Raw_Bootstrap_P":
                p_raw
        })


paired_correlation_delta_df = pd.DataFrame(
    paired_delta_records
)


# ------------------------------------------------------------
# 6. Holm correction separately for each correlation metric
# ------------------------------------------------------------

paired_correlation_delta_df[
    "Holm_Adjusted_P"
] = np.nan


for metric in CORRELATION_METRICS:

    mask = (
        paired_correlation_delta_df[
            "Metric"
        ] == metric
    )

    raw_p = (
        paired_correlation_delta_df
        .loc[
            mask,
            "Raw_Bootstrap_P"
        ]
        .to_numpy()
    )

    corrected_p = holm_adjust(
        raw_p
    )

    paired_correlation_delta_df.loc[
        mask,
        "Holm_Adjusted_P"
    ] = corrected_p


paired_correlation_delta_df[
    "Significant_Holm_0.05"
] = (
    paired_correlation_delta_df[
        "Holm_Adjusted_P"
    ] < 0.05
)


# ------------------------------------------------------------
# 7. Display complete results
# ------------------------------------------------------------

print("\n" + "=" * 115)
print("LEARNED HYBRID VS BASELINES")
print("Positive delta = Learned Hybrid has stronger correlation")
print("=" * 115)

display(
    paired_correlation_delta_df[
        [
            "Metric",
            "Baseline",
            "Observed_Delta",
            "Bootstrap_Mean_Delta",
            "CI_Lower",
            "CI_Upper",
            "Raw_Bootstrap_P",
            "Holm_Adjusted_P",
            "Significant_Holm_0.05"
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 8. Create thesis-friendly compact table
# ------------------------------------------------------------

compact_rows = []


for baseline in BASELINE_METHODS:

    row = {
        "Baseline":
            baseline
    }


    for metric in CORRELATION_METRICS:

        result = (
            paired_correlation_delta_df[
                (
                    paired_correlation_delta_df[
                        "Baseline"
                    ] == baseline
                )
                &
                (
                    paired_correlation_delta_df[
                        "Metric"
                    ] == metric
                )
            ]
            .iloc[0]
        )


        row[
            f"{metric} Δ"
        ] = result[
            "Observed_Delta"
        ]

        row[
            f"{metric} 95% CI"
        ] = (
            f"[{result['CI_Lower']:.4f}, "
            f"{result['CI_Upper']:.4f}]"
        )

        row[
            f"{metric} Holm p"
        ] = result[
            "Holm_Adjusted_P"
        ]


    compact_rows.append(
        row
    )


paired_correlation_compact_df = pd.DataFrame(
    compact_rows
)


print("\n" + "=" * 130)
print("COMPACT PAIRED CORRELATION DIFFERENCE TABLE")
print("=" * 130)

display(
    paired_correlation_compact_df.round(6)
)


# ------------------------------------------------------------
# 9. Pretty significance summary
# ------------------------------------------------------------

print("\n" + "=" * 105)
print("HOLM-CORRECTED SIGNIFICANCE SUMMARY")
print("=" * 105)


for metric in CORRELATION_METRICS:

    print(f"\n{metric}")
    print("-" * len(metric))

    subset = (
        paired_correlation_delta_df[
            paired_correlation_delta_df[
                "Metric"
            ] == metric
        ]
    )

    for _, row in subset.iterrows():

        p = row[
            "Holm_Adjusted_P"
        ]

        if p < 0.001:
            p_text = "p < .001"
        else:
            p_text = f"p = {p:.4f}"

        significance = (
            "SIGNIFICANT"
            if row[
                "Significant_Holm_0.05"
            ]
            else
            "NOT SIGNIFICANT"
        )

        print(
            f"  vs {row['Baseline']:15s}"
            f" Δ={row['Observed_Delta']:+.4f}"
            f" | 95% CI "
            f"[{row['CI_Lower']:.4f}, "
            f"{row['CI_Upper']:.4f}]"
            f" | {p_text}"
            f" | {significance}"
        )


# ------------------------------------------------------------
# 10. Save permanent outputs
# ------------------------------------------------------------

paired_correlation_delta_df.to_csv(
    "ROSE_HOLDOUT_CORRELATION_PAIRED_SIGNIFICANCE.csv",
    index=False
)

paired_correlation_compact_df.to_csv(
    "ROSE_HOLDOUT_CORRELATION_PAIRED_COMPACT.csv",
    index=False
)


print("\nSaved:")
print(
    "ROSE_HOLDOUT_CORRELATION_PAIRED_SIGNIFICANCE.csv"
)
print(
    "ROSE_HOLDOUT_CORRELATION_PAIRED_COMPACT.csv"
)


print("\n" + "=" * 115)
print("✅ STEP 9C PASSED")
print(
    "Paired Learned-vs-baseline correlation "
    "differences tested using the existing "
    "document-bootstrap replicates."
)
print("=" * 115)

PAIRED CORRELATION BOOTSTRAP DIFFERENCE ANALYSIS
Raw bootstrap rows: 25000
Unique bootstrap replicates: 5000
Methods: 5

LEARNED HYBRID VS BASELINES
Positive delta = Learned Hybrid has stronger correlation


,Metric,Baseline,Observed_Delta,Bootstrap_Mean_Delta,CI_Lower,CI_Upper,Raw_Bootstrap_P,Holm_Adjusted_P,Significant_Holm_0.05
0,Pearson,BERTScore F1,0.086457,0.086889,0.054012,0.123915,0.0004,0.0016,True
1,Pearson,MoverScore,0.134865,0.134487,0.087052,0.186108,0.0004,0.0016,True
2,Pearson,ROUGE-L,0.143381,0.143175,0.083159,0.204516,0.0004,0.0016,True
3,Pearson,Manual Hybrid,0.247676,0.248749,0.191474,0.312283,0.0004,0.0016,True
4,Spearman,BERTScore F1,0.094905,0.094892,0.055477,0.136775,0.0004,0.0016,True
5,Spearman,MoverScore,0.108838,0.109059,0.062644,0.159981,0.0004,0.0016,True
6,Spearman,ROUGE-L,0.135423,0.135303,0.065624,0.206649,0.0008,0.0016,True
7,Spearman,Manual Hybrid,0.240385,0.240697,0.176071,0.309517,0.0004,0.0016,True
8,Kendall,BERTScore F1,0.082117,0.081986,0.049822,0.115567,0.0004,0.0016,True
9,Kendall,MoverScore,0.096750,0.096543,0.059718,0.136274,0.0004,0.0016,True



COMPACT PAIRED CORRELATION DIFFERENCE TABLE


,Baseline,Pearson Δ,Pearson 95% CI,Pearson Holm p,Spearman Δ,Spearman 95% CI,Spearman Holm p,Kendall Δ,Kendall 95% CI,Kendall Holm p
0,BERTScore F1,0.086457,"[0.0540, 0.1239]",0.0016,0.094905,"[0.0555, 0.1368]",0.0016,0.082117,"[0.0498, 0.1156]",0.0016
1,MoverScore,0.134865,"[0.0871, 0.1861]",0.0016,0.108838,"[0.0626, 0.1600]",0.0016,0.096750,"[0.0597, 0.1363]",0.0016
2,ROUGE-L,0.143381,"[0.0832, 0.2045]",0.0016,0.135423,"[0.0656, 0.2066]",0.0016,0.115143,"[0.0612, 0.1681]",0.0016
3,Manual Hybrid,0.247676,"[0.1915, 0.3123]",0.0016,0.240385,"[0.1761, 0.3095]",0.0016,0.199843,"[0.1509, 0.2495]",0.0016



HOLM-CORRECTED SIGNIFICANCE SUMMARY

Pearson
-------
  vs BERTScore F1    Δ=+0.0865 | 95% CI [0.0540, 0.1239] | p = 0.0016 | SIGNIFICANT
  vs MoverScore      Δ=+0.1349 | 95% CI [0.0871, 0.1861] | p = 0.0016 | SIGNIFICANT
  vs ROUGE-L         Δ=+0.1434 | 95% CI [0.0832, 0.2045] | p = 0.0016 | SIGNIFICANT
  vs Manual Hybrid   Δ=+0.2477 | 95% CI [0.1915, 0.3123] | p = 0.0016 | SIGNIFICANT

Spearman
--------
  vs BERTScore F1    Δ=+0.0949 | 95% CI [0.0555, 0.1368] | p = 0.0016 | SIGNIFICANT
  vs MoverScore      Δ=+0.1088 | 95% CI [0.0626, 0.1600] | p = 0.0016 | SIGNIFICANT
  vs ROUGE-L         Δ=+0.1354 | 95% CI [0.0656, 0.2066] | p = 0.0016 | SIGNIFICANT
  vs Manual Hybrid   Δ=+0.2404 | 95% CI [0.1761, 0.3095] | p = 0.0016 | SIGNIFICANT

Kendall
-------
  vs BERTScore F1    Δ=+0.0821 | 95% CI [0.0498, 0.1156] | p = 0.0016 | SIGNIFICANT
  vs MoverScore      Δ=+0.0968 | 95% CI [0.0597, 0.1363] | p = 0.0016 | SIGNIFICANT
  vs ROUGE-L         Δ=+0.1151 | 95% CI [0.0612, 0.1681] | p = 0.0016 

## 16. Micro- and Macro-Level Pairwise Ranking Analysis

This section extends the pairwise ranking evaluation by distinguishing between micro-averaged and macro-averaged document-level accuracy.

The frozen RoSE pairwise results are recovered, after which pairwise ranking performance is calculated for the Learned Hybrid metric and each baseline at both aggregation levels. Document-level bootstrap resampling is then used to estimate uncertainty for macro pairwise accuracy and to compare the Learned Hybrid metric with the baseline methods.

In [196]:
# ============================================================
# STEP 10A: Recover and inspect frozen RoSE pairwise data
# ============================================================

import os
import pandas as pd

PAIRWISE_FILE = "ROSE_FINAL_PAIRWISE_PAIRS.parquet"

print("=" * 90)
print("RoSE FROZEN PAIRWISE DATA")
print("=" * 90)

print("File exists:", os.path.exists(PAIRWISE_FILE))

assert os.path.exists(PAIRWISE_FILE), (
    f"Could not find {PAIRWISE_FILE}"
)

rose_pairwise_df = pd.read_parquet(
    PAIRWISE_FILE
)

print("\nShape:", rose_pairwise_df.shape)

print("\nColumns:")

for i, c in enumerate(
    rose_pairwise_df.columns,
    start=1
):
    print(f"{i:02d}. {c}")


print("\n" + "=" * 90)
print("FIRST 5 ROWS")
print("=" * 90)

display(
    rose_pairwise_df.head()
)


print("\n" + "=" * 90)
print("DATA TYPES")
print("=" * 90)

print(
    rose_pairwise_df.dtypes
)


print("\n" + "=" * 90)
print("MISSING VALUES")
print("=" * 90)

print(
    rose_pairwise_df
    .isna()
    .sum()
)


# ------------------------------------------------------------
# Basic document information
# ------------------------------------------------------------

if "doc_id" in rose_pairwise_df.columns:

    print(
        "\nUnique documents:",
        rose_pairwise_df["doc_id"].nunique()
    )

    print("\nPairs per document:")

    display(
        rose_pairwise_df
        .groupby("doc_id")
        .size()
        .describe()
    )


print("\n" + "=" * 90)
print("✅ STEP 10A COMPLETE")
print("Pairwise schema recovered.")
print("No new metric has been calculated yet.")
print("=" * 90)

RoSE FROZEN PAIRWISE DATA
File exists: True

Shape: (4856, 8)

Columns:
01. doc_id
02. i
03. j
04. Learned Hybrid
05. BERTScore F1
06. MoverScore
07. ROUGE-L
08. Manual Hybrid

FIRST 5 ROWS


,doc_id,i,j,Learned Hybrid,BERTScore F1,MoverScore,ROUGE-L,Manual Hybrid
0,000571afe702684d90c1d222ce70b1e1375c1016,0,1,1.0,0.0,0.0,1.0,1.0
1,000571afe702684d90c1d222ce70b1e1375c1016,0,2,1.0,1.0,0.0,1.0,1.0
2,000571afe702684d90c1d222ce70b1e1375c1016,0,3,1.0,0.0,0.0,0.0,1.0
3,000571afe702684d90c1d222ce70b1e1375c1016,0,4,1.0,0.0,0.0,0.0,1.0
4,000571afe702684d90c1d222ce70b1e1375c1016,0,5,1.0,0.0,0.0,0.0,0.0



DATA TYPES
doc_id             object
i                   int64
j                   int64
Learned Hybrid    float64
BERTScore F1      float64
MoverScore        float64
ROUGE-L           float64
Manual Hybrid     float64
dtype: object

MISSING VALUES
doc_id            0
i                 0
j                 0
Learned Hybrid    0
BERTScore F1      0
MoverScore        0
ROUGE-L           0
Manual Hybrid     0
dtype: int64

Unique documents: 100

Pairs per document:


count    100.000000
mean      48.560000
std       10.620468
min       11.000000
25%       44.000000
50%       52.000000
75%       56.000000
max       62.000000
dtype: float64


✅ STEP 10A COMPLETE
Pairwise schema recovered.
No new metric has been calculated yet.


In [198]:
# ============================================================
# STEP 10B: Calculate MICRO and MACRO document-level
# pairwise ranking accuracy
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Define methods exactly as stored in frozen pairwise file
# ------------------------------------------------------------

PAIRWISE_METHODS = [
    "Learned Hybrid",
    "BERTScore F1",
    "MoverScore",
    "ROUGE-L",
    "Manual Hybrid"
]


# ------------------------------------------------------------
# 2. Verify pair-score encoding
# ------------------------------------------------------------

print("=" * 95)
print("PAIRWISE SCORE ENCODING CHECK")
print("=" * 95)

for method in PAIRWISE_METHODS:

    unique_values = sorted(
        rose_pairwise_df[
            method
        ].unique()
    )

    print(
        f"{method:20s}: "
        f"{unique_values}"
    )

    assert set(
        unique_values
    ).issubset(
        {0.0, 0.5, 1.0}
    ), (
        f"Unexpected pairwise values found "
        f"for {method}: {unique_values}"
    )


# ------------------------------------------------------------
# 3. MICRO pairwise accuracy
#
# This reproduces the existing main result.
# Each VALID PAIR receives equal weight.
# ------------------------------------------------------------

micro_rows = []

for method in PAIRWISE_METHODS:

    micro_accuracy = (
        rose_pairwise_df[
            method
        ].mean()
    )

    micro_rows.append({
        "Method": method,
        "Micro_Pairwise_Accuracy":
            micro_accuracy
    })


micro_pairwise_df = pd.DataFrame(
    micro_rows
)


print("\n" + "=" * 95)
print("MICRO PAIRWISE ACCURACY")
print("Each valid pair receives equal weight")
print("=" * 95)

display(
    micro_pairwise_df.round(6)
)


# ------------------------------------------------------------
# 4. Verify reproduction of original frozen results
# ------------------------------------------------------------

EXPECTED_MICRO = {
    "Learned Hybrid": 0.7184,
    "MoverScore": 0.6972,
    "BERTScore F1": 0.6770,
    "ROUGE-L": 0.6493,
    "Manual Hybrid": 0.6035
}


print("\n" + "=" * 95)
print("ORIGINAL RESULT REPRODUCTION CHECK")
print("=" * 95)

for method, expected in EXPECTED_MICRO.items():

    actual = float(
        micro_pairwise_df
        .loc[
            micro_pairwise_df[
                "Method"
            ] == method,
            "Micro_Pairwise_Accuracy"
        ]
        .iloc[0]
    )

    difference = (
        actual - expected
    )

    print(
        f"{method:20s} "
        f"actual={actual:.6f} | "
        f"reported≈{expected:.4f} | "
        f"difference={difference:+.6f}"
    )

    # Original table was rounded to four decimals,
    # so allow small rounding difference.
    assert abs(
        round(actual, 4)
        - expected
    ) <= 0.0001


# ------------------------------------------------------------
# 5. Calculate accuracy WITHIN each document
# ------------------------------------------------------------

document_pairwise_accuracy_df = (
    rose_pairwise_df
    .groupby(
        "doc_id",
        as_index=False
    )
    .agg(
        Valid_Pairs=(
            "Learned Hybrid",
            "size"
        ),

        **{
            method: (
                method,
                "mean"
            )
            for method
            in PAIRWISE_METHODS
        }
    )
)


print("\n" + "=" * 95)
print("DOCUMENT-LEVEL PAIRWISE ACCURACY TABLE")
print("=" * 95)

print(
    "Shape:",
    document_pairwise_accuracy_df.shape
)

display(
    document_pairwise_accuracy_df.head()
)


assert (
    len(document_pairwise_accuracy_df)
    == 100
)

assert (
    document_pairwise_accuracy_df[
        "Valid_Pairs"
    ].sum()
    == 4856
)


# ------------------------------------------------------------
# 6. MACRO pairwise accuracy
#
# Every source document receives equal weight,
# regardless of how many valid human pairs it contains.
# ------------------------------------------------------------

macro_rows = []

for method in PAIRWISE_METHODS:

    doc_scores = (
        document_pairwise_accuracy_df[
            method
        ]
        .astype(float)
    )

    macro_rows.append({

        "Method":
            method,

        "Macro_Document_Accuracy":
            doc_scores.mean(),

        "Document_SD":
            doc_scores.std(ddof=1),

        "Document_Min":
            doc_scores.min(),

        "Document_Max":
            doc_scores.max()
    })


macro_pairwise_df = pd.DataFrame(
    macro_rows
)


# ------------------------------------------------------------
# 7. Combine micro + macro
# ------------------------------------------------------------

pairwise_micro_macro_df = (
    micro_pairwise_df
    .merge(
        macro_pairwise_df,
        on="Method",
        how="inner"
    )
)


pairwise_micro_macro_df[
    "Macro_minus_Micro"
] = (
    pairwise_micro_macro_df[
        "Macro_Document_Accuracy"
    ]
    -
    pairwise_micro_macro_df[
        "Micro_Pairwise_Accuracy"
    ]
)


# Sort by macro accuracy
pairwise_micro_macro_df = (
    pairwise_micro_macro_df
    .sort_values(
        "Macro_Document_Accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 105)
print("RoSE PAIRWISE ACCURACY: MICRO VS MACRO")
print("=" * 105)

display(
    pairwise_micro_macro_df[
        [
            "Method",
            "Micro_Pairwise_Accuracy",
            "Macro_Document_Accuracy",
            "Macro_minus_Micro",
            "Document_SD"
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 8. Ranking comparison
# ------------------------------------------------------------

micro_ranking = (
    pairwise_micro_macro_df[
        [
            "Method",
            "Micro_Pairwise_Accuracy"
        ]
    ]
    .sort_values(
        "Micro_Pairwise_Accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)

macro_ranking = (
    pairwise_micro_macro_df[
        [
            "Method",
            "Macro_Document_Accuracy"
        ]
    ]
    .sort_values(
        "Macro_Document_Accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 95)
print("MICRO RANKING")
print("=" * 95)

display(
    micro_ranking.round(6)
)


print("\n" + "=" * 95)
print("MACRO DOCUMENT-LEVEL RANKING")
print("=" * 95)

display(
    macro_ranking.round(6)
)


# ------------------------------------------------------------
# 9. Save results
# ------------------------------------------------------------

document_pairwise_accuracy_df.to_parquet(
    "ROSE_PAIRWISE_DOCUMENT_LEVEL_ACCURACY.parquet",
    index=False
)

pairwise_micro_macro_df.to_csv(
    "ROSE_PAIRWISE_MICRO_MACRO_RESULTS.csv",
    index=False
)


print("\nSaved:")
print(
    "ROSE_PAIRWISE_DOCUMENT_LEVEL_ACCURACY.parquet"
)
print(
    "ROSE_PAIRWISE_MICRO_MACRO_RESULTS.csv"
)


# ------------------------------------------------------------
# 10. Final checks
# ------------------------------------------------------------

assert (
    pairwise_micro_macro_df[
        [
            "Micro_Pairwise_Accuracy",
            "Macro_Document_Accuracy"
        ]
    ]
    .notna()
    .all()
    .all()
)

print("\n" + "=" * 105)
print("✅ STEP 10B PASSED")
print(
    "Micro and macro document-level pairwise "
    "accuracies calculated successfully."
)
print("=" * 105)

PAIRWISE SCORE ENCODING CHECK
Learned Hybrid      : [0.0, 0.5, 1.0]
BERTScore F1        : [0.0, 0.5, 1.0]
MoverScore          : [0.0, 0.5, 1.0]
ROUGE-L             : [0.0, 0.5, 1.0]
Manual Hybrid       : [0.0, 0.5, 1.0]

MICRO PAIRWISE ACCURACY
Each valid pair receives equal weight


,Method,Micro_Pairwise_Accuracy
0,Learned Hybrid,0.718390
1,BERTScore F1,0.676998
2,MoverScore,0.697179
3,ROUGE-L,0.649300
4,Manual Hybrid,0.603480



ORIGINAL RESULT REPRODUCTION CHECK
Learned Hybrid       actual=0.718390 | reported≈0.7184 | difference=-0.000010
MoverScore           actual=0.697179 | reported≈0.6972 | difference=-0.000021
BERTScore F1         actual=0.676998 | reported≈0.6770 | difference=-0.000002
ROUGE-L              actual=0.649300 | reported≈0.6493 | difference=-0.000000
Manual Hybrid        actual=0.603480 | reported≈0.6035 | difference=-0.000020

DOCUMENT-LEVEL PAIRWISE ACCURACY TABLE
Shape: (100, 7)


,doc_id,Valid_Pairs,Learned Hybrid,BERTScore F1,MoverScore,ROUGE-L,Manual Hybrid
0,000571afe702684d90c1d222ce70b1e1375c1016,60,0.866667,0.633333,0.466667,0.683333,0.433333
1,00110802bc6eae0e8e4d3d22e27f458f41be2b22,42,0.880952,0.904762,0.880952,0.857143,0.833333
2,0021fe8d65bd0d6d76d5fefba2ac02f0c48a43f4,52,0.442308,0.442308,0.384615,0.519231,0.480769
3,002da0eaa8d2d09ac66228ba1c49cdc152fab925,56,0.839286,0.607143,0.642857,0.392857,0.303571
4,0034b7c223e24477e046cf3ee085dd006be38b27,59,0.949153,0.762712,0.627119,0.762712,0.677966



RoSE PAIRWISE ACCURACY: MICRO VS MACRO


...


MICRO RANKING


,Method,Micro_Pairwise_Accuracy
0,Learned Hybrid,0.718390
1,MoverScore,0.697179
2,BERTScore F1,0.676998
3,ROUGE-L,0.649300
4,Manual Hybrid,0.603480



MACRO DOCUMENT-LEVEL RANKING


,Method,Macro_Document_Accuracy
0,Learned Hybrid,0.713429
1,MoverScore,0.706819
2,BERTScore F1,0.681106
3,ROUGE-L,0.658217
4,Manual Hybrid,0.612174



Saved:
ROSE_PAIRWISE_DOCUMENT_LEVEL_ACCURACY.parquet
ROSE_PAIRWISE_MICRO_MACRO_RESULTS.csv

✅ STEP 10B PASSED
Micro and macro document-level pairwise accuracies calculated successfully.


In [200]:
# ============================================================
# STEP 10C: Document-bootstrap uncertainty for
# MACRO pairwise accuracy
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Specification
# ------------------------------------------------------------

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

PAIRWISE_METHODS = [
    "Learned Hybrid",
    "MoverScore",
    "BERTScore F1",
    "ROUGE-L",
    "Manual Hybrid"
]

LEARNED_METHOD = "Learned Hybrid"

BASELINE_METHODS = [
    "MoverScore",
    "BERTScore F1",
    "ROUGE-L",
    "Manual Hybrid"
]


# ------------------------------------------------------------
# 2. Validate document-level table
# ------------------------------------------------------------

print("=" * 100)
print("MACRO PAIRWISE DOCUMENT-BOOTSTRAP SPECIFICATION")
print("=" * 100)

print(
    "Documents:",
    len(document_pairwise_accuracy_df)
)

print(
    "Methods:",
    len(PAIRWISE_METHODS)
)

print(
    "Bootstrap replicates:",
    N_BOOTSTRAP
)

print(
    "Random seed:",
    BOOTSTRAP_SEED
)

print(
    "Resampling unit: source document"
)


assert len(
    document_pairwise_accuracy_df
) == 100

assert (
    document_pairwise_accuracy_df[
        "doc_id"
    ].nunique()
    == 100
)

assert all(
    method in document_pairwise_accuracy_df.columns
    for method in PAIRWISE_METHODS
)


# ------------------------------------------------------------
# 3. Observed macro accuracies
# ------------------------------------------------------------

observed_macro = {
    method:
        float(
            document_pairwise_accuracy_df[
                method
            ].mean()
        )
    for method in PAIRWISE_METHODS
}


observed_macro_df = pd.DataFrame([
    {
        "Method": method,
        "Macro_Accuracy":
            observed_macro[method]
    }
    for method in PAIRWISE_METHODS
])


print("\n" + "=" * 100)
print("OBSERVED MACRO DOCUMENT ACCURACY")
print("=" * 100)

display(
    observed_macro_df.round(6)
)


# ------------------------------------------------------------
# 4. Convert method scores to arrays
# ------------------------------------------------------------

method_arrays = {
    method:
        document_pairwise_accuracy_df[
            method
        ]
        .astype(float)
        .to_numpy()

    for method in PAIRWISE_METHODS
}


n_docs = len(
    document_pairwise_accuracy_df
)


# ------------------------------------------------------------
# 5. Bootstrap
# ------------------------------------------------------------

bootstrap_records = []

print("\nRunning bootstrap...")


for b in range(N_BOOTSTRAP):

    # Sample DOCUMENT POSITIONS with replacement
    sampled_idx = rng.choice(
        np.arange(n_docs),
        size=n_docs,
        replace=True
    )


    for method in PAIRWISE_METHODS:

        score = np.mean(
            method_arrays[
                method
            ][sampled_idx]
        )

        bootstrap_records.append({
            "bootstrap": b + 1,
            "Method": method,
            "Macro_Accuracy": score
        })


    if (b + 1) % 500 == 0:

        print(
            f"  completed "
            f"{b + 1:,} / "
            f"{N_BOOTSTRAP:,}"
        )


macro_bootstrap_raw_df = pd.DataFrame(
    bootstrap_records
)


expected_rows = (
    N_BOOTSTRAP
    * len(PAIRWISE_METHODS)
)


print(
    "\nBootstrap rows:",
    len(macro_bootstrap_raw_df)
)

print(
    "Expected rows:",
    expected_rows
)


assert len(
    macro_bootstrap_raw_df
) == expected_rows


# ------------------------------------------------------------
# 6. Individual method confidence intervals
# ------------------------------------------------------------

macro_ci_rows = []


for method in PAIRWISE_METHODS:

    values = (
        macro_bootstrap_raw_df[
            macro_bootstrap_raw_df[
                "Method"
            ] == method
        ][
            "Macro_Accuracy"
        ]
        .to_numpy()
    )

    macro_ci_rows.append({

        "Method":
            method,

        "Observed":
            observed_macro[method],

        "Bootstrap_Mean":
            np.mean(values),

        "CI_Lower":
            np.percentile(
                values,
                2.5
            ),

        "CI_Upper":
            np.percentile(
                values,
                97.5
            )
    })


macro_pairwise_ci_df = pd.DataFrame(
    macro_ci_rows
)


print("\n" + "=" * 105)
print("MACRO PAIRWISE ACCURACY WITH")
print("95% DOCUMENT-BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 105)

display(
    macro_pairwise_ci_df.round(6)
)


# ------------------------------------------------------------
# 7. Pivot bootstrap data for PAIRED comparisons
# ------------------------------------------------------------

bootstrap_wide = (
    macro_bootstrap_raw_df
    .pivot(
        index="bootstrap",
        columns="Method",
        values="Macro_Accuracy"
    )
    .sort_index()
)


assert len(
    bootstrap_wide
) == N_BOOTSTRAP


# ------------------------------------------------------------
# 8. Holm adjustment helper
# ------------------------------------------------------------

def holm_adjust(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    m = len(p_values)

    order = np.argsort(
        p_values
    )

    sorted_p = p_values[
        order
    ]

    adjusted_sorted = np.zeros(
        m,
        dtype=float
    )

    running_max = 0.0

    for i, p in enumerate(sorted_p):

        adjusted = (
            (m - i) * p
        )

        running_max = max(
            running_max,
            adjusted
        )

        adjusted_sorted[i] = min(
            running_max,
            1.0
        )


    adjusted = np.empty(
        m,
        dtype=float
    )

    adjusted[
        order
    ] = adjusted_sorted

    return adjusted


# ------------------------------------------------------------
# 9. Learned Hybrid vs each baseline
# ------------------------------------------------------------

paired_rows = []


learned_boot = (
    bootstrap_wide[
        LEARNED_METHOD
    ]
    .to_numpy()
)


for baseline in BASELINE_METHODS:

    baseline_boot = (
        bootstrap_wide[
            baseline
        ]
        .to_numpy()
    )


    # Positive = Learned Hybrid better
    delta = (
        learned_boot
        - baseline_boot
    )


    observed_delta = (
        observed_macro[
            LEARNED_METHOD
        ]
        -
        observed_macro[
            baseline
        ]
    )


    ci_lower = np.percentile(
        delta,
        2.5
    )

    ci_upper = np.percentile(
        delta,
        97.5
    )


    # Empirical two-sided bootstrap p-value
    # +1 correction avoids p = 0
    lower_tail = (
        np.sum(
            delta <= 0
        ) + 1
    ) / (
        len(delta) + 1
    )

    upper_tail = (
        np.sum(
            delta >= 0
        ) + 1
    ) / (
        len(delta) + 1
    )

    p_raw = min(
        1.0,
        2 * min(
            lower_tail,
            upper_tail
        )
    )


    paired_rows.append({

        "Baseline":
            baseline,

        "Observed_Delta":
            observed_delta,

        "Bootstrap_Mean_Delta":
            np.mean(delta),

        "CI_Lower":
            ci_lower,

        "CI_Upper":
            ci_upper,

        "CI_Excludes_Zero":
            (
                ci_lower > 0
                or
                ci_upper < 0
            ),

        "Raw_Bootstrap_P":
            p_raw
    })


macro_pairwise_paired_df = pd.DataFrame(
    paired_rows
)


# ------------------------------------------------------------
# 10. Holm correction across four comparisons
# ------------------------------------------------------------

macro_pairwise_paired_df[
    "Holm_Adjusted_P"
] = holm_adjust(
    macro_pairwise_paired_df[
        "Raw_Bootstrap_P"
    ].to_numpy()
)


macro_pairwise_paired_df[
    "Significant_Holm_0.05"
] = (
    macro_pairwise_paired_df[
        "Holm_Adjusted_P"
    ] < 0.05
)


print("\n" + "=" * 110)
print("LEARNED HYBRID VS BASELINES")
print("MACRO DOCUMENT-LEVEL PAIRWISE ACCURACY")
print("Positive delta = Learned Hybrid better")
print("=" * 110)

display(
    macro_pairwise_paired_df.round(6)
)


# ------------------------------------------------------------
# 11. Readable significance summary
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("HOLM-CORRECTED SIGNIFICANCE SUMMARY")
print("=" * 100)


for _, row in macro_pairwise_paired_df.iterrows():

    p = row[
        "Holm_Adjusted_P"
    ]

    if p < 0.001:
        p_text = "p < .001"
    else:
        p_text = (
            f"p = {p:.4f}"
        )


    significance = (
        "SIGNIFICANT"
        if row[
            "Significant_Holm_0.05"
        ]
        else
        "NOT SIGNIFICANT"
    )


    print(
        f"vs {row['Baseline']:15s}"
        f" Δ={row['Observed_Delta']:+.4f}"
        f" | 95% CI "
        f"[{row['CI_Lower']:.4f}, "
        f"{row['CI_Upper']:.4f}]"
        f" | {p_text}"
        f" | {significance}"
    )


# ------------------------------------------------------------
# 12. Save
# ------------------------------------------------------------

macro_bootstrap_raw_df.to_parquet(
    "ROSE_PAIRWISE_MACRO_BOOTSTRAP_RAW.parquet",
    index=False
)

macro_pairwise_ci_df.to_csv(
    "ROSE_PAIRWISE_MACRO_BOOTSTRAP_CI.csv",
    index=False
)

macro_pairwise_paired_df.to_csv(
    "ROSE_PAIRWISE_MACRO_PAIRED_SIGNIFICANCE.csv",
    index=False
)


print("\nSaved:")
print(
    "ROSE_PAIRWISE_MACRO_BOOTSTRAP_RAW.parquet"
)
print(
    "ROSE_PAIRWISE_MACRO_BOOTSTRAP_CI.csv"
)
print(
    "ROSE_PAIRWISE_MACRO_PAIRED_SIGNIFICANCE.csv"
)


print("\n" + "=" * 110)
print("✅ STEP 10C PASSED")
print(
    "Macro document-level pairwise uncertainty "
    "and paired significance calculated."
)
print("=" * 110)

MACRO PAIRWISE DOCUMENT-BOOTSTRAP SPECIFICATION
Documents: 100
Methods: 5
Bootstrap replicates: 5000
Random seed: 42
Resampling unit: source document

OBSERVED MACRO DOCUMENT ACCURACY


,Method,Macro_Accuracy
0,Learned Hybrid,0.713429
1,MoverScore,0.706819
2,BERTScore F1,0.681106
3,ROUGE-L,0.658217
4,Manual Hybrid,0.612174



Running bootstrap...
  completed 500 / 5,000
  completed 1,000 / 5,000
  completed 1,500 / 5,000
  completed 2,000 / 5,000
  completed 2,500 / 5,000
  completed 3,000 / 5,000
  completed 3,500 / 5,000
  completed 4,000 / 5,000
  completed 4,500 / 5,000
  completed 5,000 / 5,000

Bootstrap rows: 25000
Expected rows: 25000

MACRO PAIRWISE ACCURACY WITH
95% DOCUMENT-BOOTSTRAP CONFIDENCE INTERVALS


,Method,Observed,Bootstrap_Mean,CI_Lower,CI_Upper
0,Learned Hybrid,0.713429,0.713285,0.683308,0.740860
1,MoverScore,0.706819,0.706852,0.676199,0.737826
2,BERTScore F1,0.681106,0.680971,0.651726,0.709592
3,ROUGE-L,0.658217,0.658097,0.626694,0.690245
4,Manual Hybrid,0.612174,0.611960,0.580142,0.644414



LEARNED HYBRID VS BASELINES
MACRO DOCUMENT-LEVEL PAIRWISE ACCURACY
Positive delta = Learned Hybrid better


,Baseline,Observed_Delta,Bootstrap_Mean_Delta,CI_Lower,CI_Upper,CI_Excludes_Zero,Raw_Bootstrap_P,Holm_Adjusted_P,Significant_Holm_0.05
0,MoverScore,0.006609,0.006434,-0.029698,0.040628,False,0.716657,0.716657,False
1,BERTScore F1,0.032323,0.032314,0.006108,0.059090,True,0.012797,0.025595,True
2,ROUGE-L,0.055212,0.055188,0.023342,0.087447,True,0.001600,0.004799,True
3,Manual Hybrid,0.101255,0.101325,0.060889,0.140957,True,0.000400,0.001600,True



HOLM-CORRECTED SIGNIFICANCE SUMMARY
vs MoverScore      Δ=+0.0066 | 95% CI [-0.0297, 0.0406] | p = 0.7167 | NOT SIGNIFICANT
vs BERTScore F1    Δ=+0.0323 | 95% CI [0.0061, 0.0591] | p = 0.0256 | SIGNIFICANT
vs ROUGE-L         Δ=+0.0552 | 95% CI [0.0233, 0.0874] | p = 0.0048 | SIGNIFICANT
vs Manual Hybrid   Δ=+0.1013 | 95% CI [0.0609, 0.1410] | p = 0.0016 | SIGNIFICANT

Saved:
ROSE_PAIRWISE_MACRO_BOOTSTRAP_RAW.parquet
ROSE_PAIRWISE_MACRO_BOOTSTRAP_CI.csv
ROSE_PAIRWISE_MACRO_PAIRED_SIGNIFICANCE.csv

✅ STEP 10C PASSED
Macro document-level pairwise uncertainty and paired significance calculated.


## 17. Development-Mean Sanity Baseline

This section evaluates a simple development-mean predictor as an additional sanity-check baseline on the frozen RoSE holdout partition.

The predictor uses the mean human content-coverage score from the development data as a constant prediction for every holdout summary. Its error and agreement statistics provide a low-complexity reference point for interpreting the performance of the Learned Hybrid metric and the automatic-metric baselines.

In [202]:
# ============================================================
# STEP 11: Development-mean predictor baseline
# Evaluated on frozen RoSE holdout
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)


# ------------------------------------------------------------
# 1. Human target
# ------------------------------------------------------------

TARGET = "human_content_coverage"


# ------------------------------------------------------------
# 2. Calculate constant prediction using DEVELOPMENT ONLY
# ------------------------------------------------------------

development_mean = float(
    rose_dev_df[TARGET].mean()
)

holdout_human_mean = float(
    rose_corr_holdout_df[TARGET].mean()
)


print("=" * 95)
print("DEVELOPMENT-MEAN BASELINE")
print("=" * 95)

print(
    f"Development target mean: "
    f"{development_mean:.6f}"
)

print(
    f"Holdout target mean:     "
    f"{holdout_human_mean:.6f}"
)

print(
    f"Mean shift:              "
    f"{holdout_human_mean - development_mean:+.6f}"
)


# ------------------------------------------------------------
# 3. Create constant holdout predictions
# ------------------------------------------------------------

y_holdout = (
    rose_corr_holdout_df[TARGET]
    .astype(float)
    .to_numpy()
)

mean_baseline_prediction = np.full(
    shape=len(y_holdout),
    fill_value=development_mean,
    dtype=float
)


assert len(
    mean_baseline_prediction
) == 1200

assert np.isfinite(
    mean_baseline_prediction
).all()


# ------------------------------------------------------------
# 4. Evaluate prediction error
# ------------------------------------------------------------

mean_baseline_mae = mean_absolute_error(
    y_holdout,
    mean_baseline_prediction
)

mean_baseline_rmse = np.sqrt(
    mean_squared_error(
        y_holdout,
        mean_baseline_prediction
    )
)

mean_baseline_ccc = (
    concordance_correlation_coefficient(
        y_holdout,
        mean_baseline_prediction
    )
)

mean_baseline_bias = float(
    np.mean(
        mean_baseline_prediction
        - y_holdout
    )
)


# ------------------------------------------------------------
# 5. Learned Hybrid comparison
# ------------------------------------------------------------

learned_prediction = (
    rose_corr_holdout_df[
        "learned_hybrid_prediction"
    ]
    .astype(float)
    .to_numpy()
)

learned_mae = mean_absolute_error(
    y_holdout,
    learned_prediction
)

learned_rmse = np.sqrt(
    mean_squared_error(
        y_holdout,
        learned_prediction
    )
)

learned_ccc = (
    concordance_correlation_coefficient(
        y_holdout,
        learned_prediction
    )
)


# ------------------------------------------------------------
# 6. Build comparison table
# ------------------------------------------------------------

mean_baseline_results_df = pd.DataFrame([

    {
        "Method":
            "Development Mean Predictor",

        "MAE":
            mean_baseline_mae,

        "RMSE":
            mean_baseline_rmse,

        "CCC":
            mean_baseline_ccc,

        "Mean_Bias":
            mean_baseline_bias
    },

    {
        "Method":
            "Learned Hybrid",

        "MAE":
            learned_mae,

        "RMSE":
            learned_rmse,

        "CCC":
            learned_ccc,

        "Mean_Bias":
            float(
                np.mean(
                    learned_prediction
                    - y_holdout
                )
            )
    }
])


print("\n" + "=" * 95)
print("MEAN-PREDICTOR SANITY BASELINE")
print("=" * 95)

display(
    mean_baseline_results_df.round(6)
)


# ------------------------------------------------------------
# 7. Improvement of Learned Hybrid over constant baseline
# ------------------------------------------------------------

improvement_df = pd.DataFrame([{

    "MAE_Reduction":
        mean_baseline_mae
        - learned_mae,

    "MAE_Percent_Reduction":
        100 * (
            mean_baseline_mae
            - learned_mae
        ) / mean_baseline_mae,

    "RMSE_Reduction":
        mean_baseline_rmse
        - learned_rmse,

    "RMSE_Percent_Reduction":
        100 * (
            mean_baseline_rmse
            - learned_rmse
        ) / mean_baseline_rmse,

    "CCC_Improvement":
        learned_ccc
        - mean_baseline_ccc
}])


print("\n" + "=" * 95)
print("LEARNED HYBRID IMPROVEMENT OVER MEAN PREDICTOR")
print("=" * 95)

display(
    improvement_df.round(6)
)


# ------------------------------------------------------------
# 8. Important note about correlation
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("CORRELATION NOTE")
print("=" * 95)

print("""
Pearson, Spearman and Kendall are not reported for the
development-mean predictor because its predictions are constant.
A constant vector has zero variance, so correlation is undefined.

The mean predictor is therefore used only as an error/agreement
sanity baseline.
""")


# ------------------------------------------------------------
# 9. Save
# ------------------------------------------------------------

mean_baseline_results_df.to_csv(
    "ROSE_MEAN_PREDICTOR_BASELINE.csv",
    index=False
)

improvement_df.to_csv(
    "ROSE_LEARNED_VS_MEAN_BASELINE.csv",
    index=False
)


print("Saved:")
print("ROSE_MEAN_PREDICTOR_BASELINE.csv")
print("ROSE_LEARNED_VS_MEAN_BASELINE.csv")


print("\n" + "=" * 95)
print("✅ STEP 11 PASSED")
print(
    "Development-mean sanity baseline evaluated "
    "on the frozen RoSE holdout."
)
print("=" * 95)

DEVELOPMENT-MEAN BASELINE
Development target mean: 0.409063
Holdout target mean:     0.387140
Mean shift:              -0.021922

MEAN-PREDICTOR SANITY BASELINE


,Method,MAE,RMSE,CCC,Mean_Bias
0,Development Mean Predictor,0.19646,0.240352,0.000000,0.021922
1,Learned Hybrid,0.13372,0.171185,0.631329,0.017845



LEARNED HYBRID IMPROVEMENT OVER MEAN PREDICTOR


,MAE_Reduction,MAE_Percent_Reduction,RMSE_Reduction,RMSE_Percent_Reduction,CCC_Improvement
0,0.06274,31.935188,0.069167,28.777537,0.631329



CORRELATION NOTE

Pearson, Spearman and Kendall are not reported for the
development-mean predictor because its predictions are constant.
A constant vector has zero variance, so correlation is undefined.

The mean predictor is therefore used only as an error/agreement
sanity baseline.

Saved:
ROSE_MEAN_PREDICTOR_BASELINE.csv
ROSE_LEARNED_VS_MEAN_BASELINE.csv

✅ STEP 11 PASSED
Development-mean sanity baseline evaluated on the frozen RoSE holdout.


## 18. Summary-Level and System-Level Evaluation

This section evaluates the automatic metrics under complementary levels of aggregation using the frozen RoSE holdout partition.

In addition to pooled candidate-summary evaluation, correlations are calculated at the summary level and at the summarisation-system level. System-level evaluation averages the human and automatic metric scores for each of the 12 summarisation systems before computing Pearson, Spearman, and Kendall correlations across systems.

In [204]:
# ============================================================
# STEP 12A: Verify RoSE holdout structure for
# summary-level and system-level meta-evaluation
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Work from frozen final holdout
# ------------------------------------------------------------

rose_meta_df = (
    rose_corr_holdout_df
    .copy()
)


print("=" * 100)
print("RoSE META-EVALUATION STRUCTURE CHECK")
print("=" * 100)

print(
    "Rows:",
    len(rose_meta_df)
)

print(
    "Documents:",
    rose_meta_df["doc_id"].nunique()
)

print(
    "Systems:",
    rose_meta_df["model"].nunique()
)


# ------------------------------------------------------------
# 2. List the exact systems
# ------------------------------------------------------------

systems = sorted(
    rose_meta_df[
        "model"
    ]
    .unique()
)

print("\n" + "=" * 100)
print("SYSTEMS IN FROZEN HOLDOUT")
print("=" * 100)

for i, system in enumerate(
    systems,
    start=1
):
    print(
        f"{i:02d}. {system}"
    )


# ------------------------------------------------------------
# 3. Number of systems/summaries per document
# ------------------------------------------------------------

systems_per_doc = (
    rose_meta_df
    .groupby("doc_id")[
        "model"
    ]
    .nunique()
)

rows_per_doc = (
    rose_meta_df
    .groupby("doc_id")
    .size()
)


print("\n" + "=" * 100)
print("SYSTEMS PER DOCUMENT")
print("=" * 100)

print(
    systems_per_doc
    .value_counts()
    .sort_index()
)


print("\nRows per document:")

print(
    rows_per_doc
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 4. Number of documents represented by each system
# ------------------------------------------------------------

system_counts = (
    rose_meta_df
    .groupby("model")
    .agg(
        Rows=("summary_id", "size"),
        Documents=("doc_id", "nunique")
    )
    .sort_index()
)


print("\n" + "=" * 100)
print("SYSTEM COVERAGE")
print("=" * 100)

display(
    system_counts
)


# ------------------------------------------------------------
# 5. Verify every document has exactly same system set
# ------------------------------------------------------------

expected_system_set = set(
    systems
)

docs_with_wrong_system_set = []


for doc_id, group in rose_meta_df.groupby(
    "doc_id"
):

    doc_system_set = set(
        group["model"]
    )

    if doc_system_set != expected_system_set:

        docs_with_wrong_system_set.append(
            doc_id
        )


print(
    "\nDocuments with incomplete/different "
    "system sets:",
    len(docs_with_wrong_system_set)
)


# ------------------------------------------------------------
# 6. Check duplicate system within a document
# ------------------------------------------------------------

duplicate_doc_system = (
    rose_meta_df
    .groupby(
        ["doc_id", "model"]
    )
    .size()
)

duplicate_pairs = (
    duplicate_doc_system[
        duplicate_doc_system != 1
    ]
)


print(
    "Document-system combinations "
    "not occurring exactly once:",
    len(duplicate_pairs)
)


# ------------------------------------------------------------
# 7. Required metric columns
# ------------------------------------------------------------

META_METHODS = {

    "Learned Hybrid":
        "learned_hybrid_prediction",

    "BERTScore F1":
        "bertScoreF1",

    "MoverScore":
        "moverScore",

    "ROUGE-L":
        "rougeL",

    "Manual Hybrid":
        "manual_hybrid_raw"
}


required_meta_columns = [
    "doc_id",
    "model",
    "human_content_coverage",
    *META_METHODS.values()
]


missing_meta_columns = [
    c
    for c in required_meta_columns
    if c not in rose_meta_df.columns
]


print("\nMissing required columns:",
      missing_meta_columns)


# ------------------------------------------------------------
# 8. Check within-document target variation
#
# Correlation cannot be calculated if all 12 human scores
# for a document are identical.
# ------------------------------------------------------------

human_unique_per_doc = (
    rose_meta_df
    .groupby("doc_id")[
        "human_content_coverage"
    ]
    .nunique()
)


constant_human_docs = (
    human_unique_per_doc[
        human_unique_per_doc <= 1
    ]
    .index
    .tolist()
)


print("\n" + "=" * 100)
print("WITHIN-DOCUMENT HUMAN SCORE VARIATION")
print("=" * 100)

print(
    "Documents with constant human ACU:",
    len(constant_human_docs)
)

if constant_human_docs:
    print(
        "Constant-target documents:",
        constant_human_docs
    )


print(
    "\nHuman-score unique-value distribution:"
)

print(
    human_unique_per_doc
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 9. Check within-document variation for every metric
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("WITHIN-DOCUMENT METRIC VARIATION")
print("=" * 100)


constant_metric_summary = []


for method, column in META_METHODS.items():

    nunique_by_doc = (
        rose_meta_df
        .groupby("doc_id")[
            column
        ]
        .nunique()
    )

    n_constant = int(
        (nunique_by_doc <= 1).sum()
    )

    constant_metric_summary.append({
        "Method": method,
        "Documents_with_constant_metric":
            n_constant
    })


constant_metric_summary_df = pd.DataFrame(
    constant_metric_summary
)

display(
    constant_metric_summary_df
)


# ------------------------------------------------------------
# 10. Final assertions
# ------------------------------------------------------------

assert len(rose_meta_df) == 1200

assert (
    rose_meta_df[
        "doc_id"
    ].nunique()
    == 100
)

assert (
    rose_meta_df[
        "model"
    ].nunique()
    == 12
)

assert (
    systems_per_doc == 12
).all()

assert (
    rows_per_doc == 12
).all()

assert (
    system_counts[
        "Rows"
    ] == 100
).all()

assert (
    system_counts[
        "Documents"
    ] == 100
).all()

assert (
    len(
        docs_with_wrong_system_set
    ) == 0
)

assert (
    len(
        duplicate_pairs
    ) == 0
)

assert (
    len(
        missing_meta_columns
    ) == 0
)


print("\n" + "=" * 100)
print("✅ STEP 12A PASSED")
print(
    "Frozen holdout structure is valid for "
    "RoSE-style summary/system meta-evaluation."
)
print(
    "No correlations have been calculated yet."
)
print("=" * 100)

RoSE META-EVALUATION STRUCTURE CHECK
Rows: 1200
Documents: 100
Systems: 12

SYSTEMS IN FROZEN HOLDOUT
01. bart
02. brio
03. brio-ext
04. cliff
05. ctrlsum
06. frost
07. glob
08. gold
09. gsum
10. matchsum
11. pegasus
12. simcls

SYSTEMS PER DOCUMENT
model
12    100
Name: count, dtype: int64

Rows per document:
12    100
Name: count, dtype: int64

SYSTEM COVERAGE


,Rows,Documents
model,,
bart,100,100
brio,100,100
brio-ext,100,100
cliff,100,100
ctrlsum,100,100
frost,100,100
glob,100,100
gold,100,100
gsum,100,100



Documents with incomplete/different system sets: 0
Document-system combinations not occurring exactly once: 0

Missing required columns: []

WITHIN-DOCUMENT HUMAN SCORE VARIATION
Documents with constant human ACU: 0

Human-score unique-value distribution:
human_content_coverage
2     5
3    15
4    18
5    30
6    26
7     4
8     1
9     1
Name: count, dtype: int64

WITHIN-DOCUMENT METRIC VARIATION


,Method,Documents_with_constant_metric
0,Learned Hybrid,0
1,BERTScore F1,0
2,MoverScore,0
3,ROUGE-L,0
4,Manual Hybrid,0



✅ STEP 12A PASSED
Frozen holdout structure is valid for RoSE-style summary/system meta-evaluation.
No correlations have been calculated yet.


In [206]:
# ============================================================
# STEP 12B: RoSE-style summary-level and system-level
# correlation analysis
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau
)


# ------------------------------------------------------------
# 1. Fixed target and methods
# ------------------------------------------------------------

TARGET = "human_content_coverage"

META_METHODS = {

    "Learned Hybrid":
        "learned_hybrid_prediction",

    "BERTScore F1":
        "bertScoreF1",

    "MoverScore":
        "moverScore",

    "ROUGE-L":
        "rougeL",

    "Manual Hybrid":
        "manual_hybrid_raw"
}


# ------------------------------------------------------------
# 2. Correlation helper
# ------------------------------------------------------------

def compute_three_correlations(
    human_scores,
    metric_scores
):

    human_scores = np.asarray(
        human_scores,
        dtype=float
    )

    metric_scores = np.asarray(
        metric_scores,
        dtype=float
    )

    return {

        "Pearson":
            pearsonr(
                human_scores,
                metric_scores
            ).statistic,

        "Spearman":
            spearmanr(
                human_scores,
                metric_scores
            ).statistic,

        "Kendall":
            kendalltau(
                human_scores,
                metric_scores
            ).statistic
    }


# ============================================================
# PART A — POOLED CORRELATION
# Existing main analysis
# ============================================================

pooled_rows = []

for method, metric_col in META_METHODS.items():

    corr = compute_three_correlations(

        rose_meta_df[
            TARGET
        ].to_numpy(),

        rose_meta_df[
            metric_col
        ].to_numpy()
    )

    pooled_rows.append({

        "Method":
            method,

        "Pearson":
            corr["Pearson"],

        "Spearman":
            corr["Spearman"],

        "Kendall":
            corr["Kendall"]
    })


pooled_meta_df = pd.DataFrame(
    pooled_rows
)


print("=" * 110)
print("A. POOLED CANDIDATE-LEVEL CORRELATION")
print("All 1,200 summaries treated together")
print("=" * 110)

display(
    pooled_meta_df.round(6)
)


# ------------------------------------------------------------
# Confirm existing frozen values
# ------------------------------------------------------------

expected_pooled = {

    "Learned Hybrid":
        (0.708710, 0.702939, 0.522724),

    "BERTScore F1":
        (0.622253, 0.608035, 0.440607),

    "MoverScore":
        (0.573845, 0.594102, 0.425974),

    "ROUGE-L":
        (0.565328, 0.567517, 0.407581),

    "Manual Hybrid":
        (0.461034, 0.462555, 0.322882)
}


for _, row in pooled_meta_df.iterrows():

    expected = expected_pooled[
        row["Method"]
    ]

    assert np.isclose(
        row["Pearson"],
        expected[0],
        atol=1e-6
    )

    assert np.isclose(
        row["Spearman"],
        expected[1],
        atol=1e-6
    )

    assert np.isclose(
        row["Kendall"],
        expected[2],
        atol=1e-6
    )


print(
    "\n✅ Existing pooled correlations reproduced exactly."
)


# ============================================================
# PART B — SUMMARY-LEVEL CORRELATION
#
# For EACH document:
# correlate the 12 systems against human ACU.
#
# Then average across the 100 documents.
# ============================================================

summary_level_records = []


for doc_id, doc_group in rose_meta_df.groupby(
    "doc_id",
    sort=True
):

    # Defensive check
    assert len(doc_group) == 12

    human_doc = (
        doc_group[
            TARGET
        ]
        .astype(float)
        .to_numpy()
    )


    for method, metric_col in META_METHODS.items():

        metric_doc = (
            doc_group[
                metric_col
            ]
            .astype(float)
            .to_numpy()
        )


        correlations = (
            compute_three_correlations(
                human_doc,
                metric_doc
            )
        )


        summary_level_records.append({

            "doc_id":
                doc_id,

            "Method":
                method,

            "Pearson":
                correlations[
                    "Pearson"
                ],

            "Spearman":
                correlations[
                    "Spearman"
                ],

            "Kendall":
                correlations[
                    "Kendall"
                ]
        })


summary_level_per_doc_df = pd.DataFrame(
    summary_level_records
)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

expected_rows = (
    100
    * len(META_METHODS)
)

assert len(
    summary_level_per_doc_df
) == expected_rows

assert (
    summary_level_per_doc_df[
        ["Pearson", "Spearman", "Kendall"]
    ]
    .notna()
    .all()
    .all()
)


# ------------------------------------------------------------
# Average document correlations
# ------------------------------------------------------------

summary_level_results_df = (

    summary_level_per_doc_df

    .groupby(
        "Method",
        as_index=False
    )

    .agg(

        Pearson=(
            "Pearson",
            "mean"
        ),

        Spearman=(
            "Spearman",
            "mean"
        ),

        Kendall=(
            "Kendall",
            "mean"
        ),

        Pearson_SD=(
            "Pearson",
            "std"
        ),

        Spearman_SD=(
            "Spearman",
            "std"
        ),

        Kendall_SD=(
            "Kendall",
            "std"
        )
    )
)


# Reorder methods
method_order = list(
    META_METHODS.keys()
)

summary_level_results_df[
    "Method"
] = pd.Categorical(

    summary_level_results_df[
        "Method"
    ],

    categories=method_order,
    ordered=True
)

summary_level_results_df = (

    summary_level_results_df
    .sort_values("Method")
    .reset_index(drop=True)
)


print("\n" + "=" * 110)
print("B. SUMMARY-LEVEL CORRELATION")
print(
    "Correlation across 12 systems within each document,"
)
print(
    "then macro-averaged across 100 documents"
)
print("=" * 110)

display(

    summary_level_results_df[
        [
            "Method",
            "Pearson",
            "Spearman",
            "Kendall",
            "Pearson_SD",
            "Spearman_SD",
            "Kendall_SD"
        ]
    ]
    .round(6)
)


# ============================================================
# PART C — SYSTEM-LEVEL CORRELATION
#
# Average each system over 100 documents.
# Then correlate 12 system averages.
# ============================================================

aggregation_dict = {

    TARGET:
        "mean"
}


for metric_col in META_METHODS.values():

    aggregation_dict[
        metric_col
    ] = "mean"


system_mean_df = (

    rose_meta_df

    .groupby(
        "model",
        as_index=False
    )

    .agg(
        aggregation_dict
    )
)


assert len(
    system_mean_df
) == 12


print("\n" + "=" * 110)
print("SYSTEM-LEVEL MEAN SCORES")
print("Each row represents one summarisation system")
print("=" * 110)

display(
    system_mean_df.round(6)
)


# ------------------------------------------------------------
# Correlate 12 system averages
# ------------------------------------------------------------

system_level_rows = []

human_system_mean = (

    system_mean_df[
        TARGET
    ]
    .astype(float)
    .to_numpy()
)


for method, metric_col in META_METHODS.items():

    metric_system_mean = (

        system_mean_df[
            metric_col
        ]
        .astype(float)
        .to_numpy()
    )


    correlations = (
        compute_three_correlations(
            human_system_mean,
            metric_system_mean
        )
    )


    system_level_rows.append({

        "Method":
            method,

        "Pearson":
            correlations[
                "Pearson"
            ],

        "Spearman":
            correlations[
                "Spearman"
            ],

        "Kendall":
            correlations[
                "Kendall"
            ]
    })


system_level_results_df = pd.DataFrame(
    system_level_rows
)


print("\n" + "=" * 110)
print("C. SYSTEM-LEVEL CORRELATION")
print(
    "Correlation across the 12 system-level averages"
)
print("=" * 110)

display(
    system_level_results_df.round(6)
)


# ============================================================
# PART D — COMBINE ALL THREE VIEWS
# ============================================================

comparison_rows = []


for method in method_order:

    pooled_row = (
        pooled_meta_df[
            pooled_meta_df[
                "Method"
            ] == method
        ]
        .iloc[0]
    )

    summary_row = (
        summary_level_results_df[
            summary_level_results_df[
                "Method"
            ] == method
        ]
        .iloc[0]
    )

    system_row = (
        system_level_results_df[
            system_level_results_df[
                "Method"
            ] == method
        ]
        .iloc[0]
    )


    comparison_rows.append({

        "Method":
            method,

        "Pooled_Pearson":
            pooled_row[
                "Pearson"
            ],

        "Summary_Pearson":
            summary_row[
                "Pearson"
            ],

        "System_Pearson":
            system_row[
                "Pearson"
            ],

        "Pooled_Spearman":
            pooled_row[
                "Spearman"
            ],

        "Summary_Spearman":
            summary_row[
                "Spearman"
            ],

        "System_Spearman":
            system_row[
                "Spearman"
            ],

        "Pooled_Kendall":
            pooled_row[
                "Kendall"
            ],

        "Summary_Kendall":
            summary_row[
                "Kendall"
            ],

        "System_Kendall":
            system_row[
                "Kendall"
            ]
    })


meta_comparison_df = pd.DataFrame(
    comparison_rows
)


print("\n" + "=" * 120)
print("POOLED VS SUMMARY-LEVEL VS SYSTEM-LEVEL")
print("=" * 120)

display(
    meta_comparison_df.round(6)
)


# ============================================================
# PART E — Ranking under each aggregation
# ============================================================

print("\n" + "=" * 110)
print("SUMMARY-LEVEL PEARSON RANKING")
print("=" * 110)

display(

    summary_level_results_df[
        [
            "Method",
            "Pearson",
            "Spearman",
            "Kendall"
        ]
    ]

    .sort_values(
        "Pearson",
        ascending=False
    )

    .reset_index(drop=True)

    .round(6)
)


print("\n" + "=" * 110)
print("SYSTEM-LEVEL PEARSON RANKING")
print("=" * 110)

display(

    system_level_results_df

    .sort_values(
        "Pearson",
        ascending=False
    )

    .reset_index(drop=True)

    .round(6)
)


# ============================================================
# PART F — Save permanent results
# ============================================================

summary_level_per_doc_df.to_parquet(
    "ROSE_SUMMARY_LEVEL_CORRELATIONS_PER_DOCUMENT.parquet",
    index=False
)

summary_level_results_df.to_csv(
    "ROSE_SUMMARY_LEVEL_CORRELATIONS.csv",
    index=False
)

system_mean_df.to_csv(
    "ROSE_SYSTEM_LEVEL_MEAN_SCORES.csv",
    index=False
)

system_level_results_df.to_csv(
    "ROSE_SYSTEM_LEVEL_CORRELATIONS.csv",
    index=False
)

meta_comparison_df.to_csv(
    "ROSE_POOLED_SUMMARY_SYSTEM_CORRELATIONS.csv",
    index=False
)


print("\nSaved:")
print(
    "ROSE_SUMMARY_LEVEL_CORRELATIONS_PER_DOCUMENT.parquet"
)
print(
    "ROSE_SUMMARY_LEVEL_CORRELATIONS.csv"
)
print(
    "ROSE_SYSTEM_LEVEL_MEAN_SCORES.csv"
)
print(
    "ROSE_SYSTEM_LEVEL_CORRELATIONS.csv"
)
print(
    "ROSE_POOLED_SUMMARY_SYSTEM_CORRELATIONS.csv"
)


print("\n" + "=" * 120)
print("✅ STEP 12B PASSED")
print(
    "Pooled, summary-level, and system-level "
    "correlations calculated successfully."
)
print("=" * 120)

A. POOLED CANDIDATE-LEVEL CORRELATION
All 1,200 summaries treated together


,Method,Pearson,Spearman,Kendall
0,Learned Hybrid,0.708710,0.702939,0.522724
1,BERTScore F1,0.622253,0.608035,0.440607
2,MoverScore,0.573845,0.594102,0.425974
3,ROUGE-L,0.565328,0.567517,0.407581
4,Manual Hybrid,0.461034,0.462555,0.322882



✅ Existing pooled correlations reproduced exactly.

B. SUMMARY-LEVEL CORRELATION
Correlation across 12 systems within each document,
then macro-averaged across 100 documents


,Method,Pearson,Spearman,Kendall,Pearson_SD,Spearman_SD,Kendall_SD
0,Learned Hybrid,0.508444,0.456832,0.370192,0.281543,0.280359,0.237232
1,BERTScore F1,0.422859,0.379970,0.305463,0.308291,0.291904,0.239196
2,MoverScore,0.449951,0.423227,0.343648,0.304410,0.298773,0.246591
3,ROUGE-L,0.354385,0.328305,0.262411,0.340297,0.332298,0.267906
4,Manual Hybrid,0.214592,0.230882,0.182744,0.333782,0.333649,0.270138



SYSTEM-LEVEL MEAN SCORES
Each row represents one summarisation system


...


SUMMARY-LEVEL PEARSON RANKING


,Method,Pearson,Spearman,Kendall
0,Learned Hybrid,0.508444,0.456832,0.370192
1,MoverScore,0.449951,0.423227,0.343648
2,BERTScore F1,0.422859,0.379970,0.305463
3,ROUGE-L,0.354385,0.328305,0.262411
4,Manual Hybrid,0.214592,0.230882,0.182744



SYSTEM-LEVEL PEARSON RANKING


,Method,Pearson,Spearman,Kendall
0,Learned Hybrid,0.908961,0.867133,0.696970
1,BERTScore F1,0.255136,0.307692,0.181818
2,MoverScore,0.018367,0.069930,0.121212
3,ROUGE-L,-0.223181,-0.146853,-0.090909
4,Manual Hybrid,-0.472745,-0.335664,-0.242424



Saved:
ROSE_SUMMARY_LEVEL_CORRELATIONS_PER_DOCUMENT.parquet
ROSE_SUMMARY_LEVEL_CORRELATIONS.csv
ROSE_SYSTEM_LEVEL_MEAN_SCORES.csv
ROSE_SYSTEM_LEVEL_CORRELATIONS.csv
ROSE_POOLED_SUMMARY_SYSTEM_CORRELATIONS.csv

✅ STEP 12B PASSED
Pooled, summary-level, and system-level correlations calculated successfully.


## 19. Summary-Level and System-Level Bootstrap Inference

This section quantifies uncertainty in the summary-level and system-level RoSE correlation results using document-level bootstrap resampling.

A total of 5,000 source-document bootstrap replicates are used to estimate 95% confidence intervals for Pearson, Spearman, and Kendall correlations at both aggregation levels. Paired bootstrap comparisons are then performed between the Learned Hybrid metric and each baseline to assess whether differences in correlation performance are statistically meaningful.

In [208]:
# ============================================================
# STEP 12C: Document-bootstrap confidence intervals for
# SUMMARY-LEVEL and SYSTEM-LEVEL correlations
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau
)


# ------------------------------------------------------------
# 1. Bootstrap specification
# ------------------------------------------------------------

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

METRICS = [
    "Pearson",
    "Spearman",
    "Kendall"
]

METHOD_ORDER = list(
    META_METHODS.keys()
)


# ------------------------------------------------------------
# 2. Prepare document IDs
# ------------------------------------------------------------

unique_docs = np.array(
    sorted(
        rose_meta_df[
            "doc_id"
        ].unique()
    )
)

n_docs = len(unique_docs)

assert n_docs == 100


print("=" * 105)
print("RoSE SUMMARY/SYSTEM CORRELATION BOOTSTRAP")
print("=" * 105)

print("Documents:", n_docs)
print("Systems:", rose_meta_df["model"].nunique())
print("Methods:", len(METHOD_ORDER))
print("Bootstrap replicates:", N_BOOTSTRAP)
print("Random seed:", BOOTSTRAP_SEED)
print("Resampling unit: source document")
print("Sampling: with replacement")


# ============================================================
# PART A
# Prepare summary-level per-document correlations
# ============================================================

# Pivot existing per-document correlation results
# into easy lookup tables.

summary_lookup = {}

for metric in METRICS:

    summary_lookup[metric] = (
        summary_level_per_doc_df
        .pivot(
            index="doc_id",
            columns="Method",
            values=metric
        )
        .reindex(
            index=unique_docs,
            columns=METHOD_ORDER
        )
    )

    assert (
        summary_lookup[metric]
        .notna()
        .all()
        .all()
    )


# ============================================================
# PART B
# Prepare raw document-system rows for system-level bootstrap
# ============================================================

# Each document must contain exactly 12 systems.
# We will select documents WITH replacement and preserve
# repeated documents in the bootstrap sample.

system_order = sorted(
    rose_meta_df[
        "model"
    ].unique()
)

assert len(system_order) == 12


# Create per-document dataframes for fast retrieval
doc_frames = {
    doc_id:
        rose_meta_df[
            rose_meta_df[
                "doc_id"
            ] == doc_id
        ]
        .copy()

    for doc_id in unique_docs
}

assert all(
    len(df) == 12
    for df in doc_frames.values()
)


# ------------------------------------------------------------
# 3. Helper for three correlations
# ------------------------------------------------------------

def three_corr(y, x):

    y = np.asarray(
        y,
        dtype=float
    )

    x = np.asarray(
        x,
        dtype=float
    )

    return {
        "Pearson":
            pearsonr(
                y,
                x
            ).statistic,

        "Spearman":
            spearmanr(
                y,
                x
            ).statistic,

        "Kendall":
            kendalltau(
                y,
                x
            ).statistic
    }


# ------------------------------------------------------------
# 4. Bootstrap
# ------------------------------------------------------------

summary_bootstrap_records = []
system_bootstrap_records = []

print("\nRunning 5,000 document-bootstrap replicates...")


for b in range(N_BOOTSTRAP):

    # --------------------------------------------------------
    # Sample 100 source documents WITH replacement
    # --------------------------------------------------------

    sampled_docs = rng.choice(
        unique_docs,
        size=n_docs,
        replace=True
    )


    # ========================================================
    # A. SUMMARY-LEVEL
    #
    # Each sampled source document contributes its
    # per-document correlation.
    # ========================================================

    for metric in METRICS:

        table = summary_lookup[
            metric
        ]

        sampled_values = (
            table
            .loc[
                sampled_docs
            ]
            .to_numpy()
        )

        # shape:
        # 100 sampled documents × 5 methods

        mean_correlations = np.mean(
            sampled_values,
            axis=0
        )


        for method_idx, method in enumerate(
            METHOD_ORDER
        ):

            summary_bootstrap_records.append({

                "bootstrap":
                    b + 1,

                "Method":
                    method,

                "Metric":
                    metric,

                "Correlation":
                    float(
                        mean_correlations[
                            method_idx
                        ]
                    )
            })


    # ========================================================
    # B. SYSTEM-LEVEL
    #
    # Rebuild the bootstrap sample, average every system
    # across sampled documents, then correlate the 12
    # resulting system means.
    # ========================================================

    sampled_frames = []

    for sample_position, doc_id in enumerate(
        sampled_docs
    ):

        temp = (
            doc_frames[
                doc_id
            ]
            .copy()
        )

        # Give repeated sampled documents distinct bootstrap
        # instance IDs so multiplicity is explicitly retained.
        temp[
            "_bootstrap_doc_instance"
        ] = sample_position

        sampled_frames.append(
            temp
        )


    boot_df = pd.concat(
        sampled_frames,
        ignore_index=True
    )


    # Expected:
    # 100 sampled document instances × 12 systems
    assert len(
        boot_df
    ) == 1200


    # Average over sampled document instances
    system_boot_mean_df = (
        boot_df
        .groupby(
            "model",
            as_index=False
        )
        .agg({

            TARGET:
                "mean",

            **{
                metric_col:
                    "mean"

                for metric_col
                in META_METHODS.values()
            }
        })
        .set_index(
            "model"
        )
        .reindex(
            system_order
        )
        .reset_index()
    )


    assert len(
        system_boot_mean_df
    ) == 12


    human_system = (
        system_boot_mean_df[
            TARGET
        ]
        .astype(float)
        .to_numpy()
    )


    for method, metric_col in META_METHODS.items():

        metric_system = (
            system_boot_mean_df[
                metric_col
            ]
            .astype(float)
            .to_numpy()
        )

        correlations = three_corr(
            human_system,
            metric_system
        )


        for metric in METRICS:

            system_bootstrap_records.append({

                "bootstrap":
                    b + 1,

                "Method":
                    method,

                "Metric":
                    metric,

                "Correlation":
                    correlations[
                        metric
                    ]
            })


    if (b + 1) % 500 == 0:

        print(
            f"  completed "
            f"{b + 1:,} / "
            f"{N_BOOTSTRAP:,}"
        )


# ------------------------------------------------------------
# 5. Convert to dataframes
# ------------------------------------------------------------

summary_meta_bootstrap_df = pd.DataFrame(
    summary_bootstrap_records
)

system_meta_bootstrap_df = pd.DataFrame(
    system_bootstrap_records
)


expected_each = (
    N_BOOTSTRAP
    * len(METHOD_ORDER)
    * len(METRICS)
)


print("\nSummary bootstrap rows:",
      len(summary_meta_bootstrap_df))

print("System bootstrap rows:",
      len(system_meta_bootstrap_df))

print("Expected each:",
      expected_each)


assert (
    len(summary_meta_bootstrap_df)
    == expected_each
)

assert (
    len(system_meta_bootstrap_df)
    == expected_each
)


# ============================================================
# PART C
# Summarise percentile 95% confidence intervals
# ============================================================

def bootstrap_ci_table(
    raw_bootstrap_df,
    observed_df,
    level_name
):

    rows = []


    for method in METHOD_ORDER:

        observed_row = (
            observed_df[
                observed_df[
                    "Method"
                ] == method
            ]
            .iloc[0]
        )


        for metric in METRICS:

            values = (
                raw_bootstrap_df[
                    (
                        raw_bootstrap_df[
                            "Method"
                        ] == method
                    )
                    &
                    (
                        raw_bootstrap_df[
                            "Metric"
                        ] == metric
                    )
                ][
                    "Correlation"
                ]
                .to_numpy()
            )


            rows.append({

                "Level":
                    level_name,

                "Method":
                    method,

                "Metric":
                    metric,

                "Observed":
                    float(
                        observed_row[
                            metric
                        ]
                    ),

                "Bootstrap_Mean":
                    float(
                        np.mean(
                            values
                        )
                    ),

                "CI_Lower":
                    float(
                        np.percentile(
                            values,
                            2.5
                        )
                    ),

                "CI_Upper":
                    float(
                        np.percentile(
                            values,
                            97.5
                        )
                    )
            })


    return pd.DataFrame(
        rows
    )


summary_meta_ci_df = bootstrap_ci_table(

    summary_meta_bootstrap_df,

    summary_level_results_df,

    "Summary-level"
)


system_meta_ci_df = bootstrap_ci_table(

    system_meta_bootstrap_df,

    system_level_results_df,

    "System-level"
)


# ------------------------------------------------------------
# 6. Thesis-friendly compact tables
# ------------------------------------------------------------

def create_compact_ci_table(
    ci_df
):

    rows = []

    for method in METHOD_ORDER:

        subset = (
            ci_df[
                ci_df[
                    "Method"
                ] == method
            ]
            .set_index(
                "Metric"
            )
        )


        row = {
            "Method":
                method
        }


        for metric in METRICS:

            row[
                metric
            ] = subset.loc[
                metric,
                "Observed"
            ]

            row[
                f"{metric} 95% CI"
            ] = (
                f"["
                f"{subset.loc[metric, 'CI_Lower']:.4f}, "
                f"{subset.loc[metric, 'CI_Upper']:.4f}"
                f"]"
            )


        rows.append(
            row
        )


    return pd.DataFrame(
        rows
    )


summary_meta_compact_df = (
    create_compact_ci_table(
        summary_meta_ci_df
    )
)

system_meta_compact_df = (
    create_compact_ci_table(
        system_meta_ci_df
    )
)


print("\n" + "=" * 120)
print("SUMMARY-LEVEL CORRELATIONS")
print("95% DOCUMENT-BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 120)

display(
    summary_meta_compact_df.round(6)
)


print("\n" + "=" * 120)
print("SYSTEM-LEVEL CORRELATIONS")
print("95% DOCUMENT-BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 120)

display(
    system_meta_compact_df.round(6)
)


# ------------------------------------------------------------
# 7. Bootstrap mean sanity check
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("BOOTSTRAP MEAN SANITY CHECK")
print("=" * 120)

sanity_df = pd.concat(
    [
        summary_meta_ci_df,
        system_meta_ci_df
    ],
    ignore_index=True
)

sanity_df[
    "Mean_minus_Observed"
] = (
    sanity_df[
        "Bootstrap_Mean"
    ]
    -
    sanity_df[
        "Observed"
    ]
)

display(
    sanity_df[
        [
            "Level",
            "Method",
            "Metric",
            "Observed",
            "Bootstrap_Mean",
            "Mean_minus_Observed"
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------

summary_meta_bootstrap_df.to_parquet(
    "ROSE_SUMMARY_LEVEL_BOOTSTRAP_RAW.parquet",
    index=False
)

system_meta_bootstrap_df.to_parquet(
    "ROSE_SYSTEM_LEVEL_BOOTSTRAP_RAW.parquet",
    index=False
)

summary_meta_ci_df.to_csv(
    "ROSE_SUMMARY_LEVEL_BOOTSTRAP_CI.csv",
    index=False
)

system_meta_ci_df.to_csv(
    "ROSE_SYSTEM_LEVEL_BOOTSTRAP_CI.csv",
    index=False
)

summary_meta_compact_df.to_csv(
    "ROSE_SUMMARY_LEVEL_BOOTSTRAP_COMPACT.csv",
    index=False
)

system_meta_compact_df.to_csv(
    "ROSE_SYSTEM_LEVEL_BOOTSTRAP_COMPACT.csv",
    index=False
)


print("\nSaved:")
print("ROSE_SUMMARY_LEVEL_BOOTSTRAP_RAW.parquet")
print("ROSE_SYSTEM_LEVEL_BOOTSTRAP_RAW.parquet")
print("ROSE_SUMMARY_LEVEL_BOOTSTRAP_CI.csv")
print("ROSE_SYSTEM_LEVEL_BOOTSTRAP_CI.csv")
print("ROSE_SUMMARY_LEVEL_BOOTSTRAP_COMPACT.csv")
print("ROSE_SYSTEM_LEVEL_BOOTSTRAP_COMPACT.csv")


print("\n" + "=" * 120)
print("✅ STEP 12C PASSED")
print(
    "Summary-level and system-level bootstrap "
    "confidence intervals calculated."
)
print("=" * 120)

RoSE SUMMARY/SYSTEM CORRELATION BOOTSTRAP
Documents: 100
Systems: 12
Methods: 5
Bootstrap replicates: 5000
Random seed: 42
Resampling unit: source document
Sampling: with replacement

Running 5,000 document-bootstrap replicates...
  completed 500 / 5,000
  completed 1,000 / 5,000
  completed 1,500 / 5,000
  completed 2,000 / 5,000
  completed 2,500 / 5,000
  completed 3,000 / 5,000
  completed 3,500 / 5,000
  completed 4,000 / 5,000
  completed 4,500 / 5,000
  completed 5,000 / 5,000

Summary bootstrap rows: 75000
System bootstrap rows: 75000
Expected each: 75000

SUMMARY-LEVEL CORRELATIONS
95% DOCUMENT-BOOTSTRAP CONFIDENCE INTERVALS


,Method,Pearson,Pearson 95% CI,Spearman,Spearman 95% CI,Kendall,Kendall 95% CI
0,Learned Hybrid,0.508444,"[0.4511, 0.5637]",0.456832,"[0.4005, 0.5118]",0.370192,"[0.3226, 0.4163]"
1,BERTScore F1,0.422859,"[0.3608, 0.4823]",0.379970,"[0.3213, 0.4368]",0.305463,"[0.2569, 0.3517]"
2,MoverScore,0.449951,"[0.3890, 0.5097]",0.423227,"[0.3640, 0.4832]",0.343648,"[0.2943, 0.3929]"
3,ROUGE-L,0.354385,"[0.2874, 0.4196]",0.328305,"[0.2626, 0.3920]",0.262411,"[0.2099, 0.3139]"
4,Manual Hybrid,0.214592,"[0.1482, 0.2789]",0.230882,"[0.1655, 0.2969]",0.182744,"[0.1291, 0.2365]"



SYSTEM-LEVEL CORRELATIONS
95% DOCUMENT-BOOTSTRAP CONFIDENCE INTERVALS


,Method,Pearson,Pearson 95% CI,Spearman,Spearman 95% CI,Kendall,Kendall 95% CI
0,Learned Hybrid,0.908961,"[0.6482, 0.9440]",0.867133,"[0.6222, 0.9441]",0.696970,"[0.4242, 0.8485]"
1,BERTScore F1,0.255136,"[-0.0179, 0.5355]",0.307692,"[-0.0490, 0.5455]",0.181818,"[-0.0606, 0.3939]"
2,MoverScore,0.018367,"[-0.2357, 0.4307]",0.069930,"[-0.2867, 0.4266]",0.121212,"[-0.2121, 0.3333]"
3,ROUGE-L,-0.223181,"[-0.4524, 0.2552]",-0.146853,"[-0.4685, 0.2657]",-0.090909,"[-0.3333, 0.1818]"
4,Manual Hybrid,-0.472745,"[-0.6768, 0.0736]",-0.335664,"[-0.6923, 0.1119]",-0.242424,"[-0.5152, 0.0909]"



BOOTSTRAP MEAN SANITY CHECK


,Level,Method,Metric,Observed,Bootstrap_Mean,Mean_minus_Observed
0,Summary-level,Learned Hybrid,Pearson,0.508444,0.508331,-0.000113
1,Summary-level,Learned Hybrid,Spearman,0.456832,0.456668,-0.000164
2,Summary-level,Learned Hybrid,Kendall,0.370192,0.370060,-0.000132
3,Summary-level,BERTScore F1,Pearson,0.422859,0.422651,-0.000208
4,Summary-level,BERTScore F1,Spearman,0.379970,0.379697,-0.000272
5,Summary-level,BERTScore F1,Kendall,0.305463,0.305196,-0.000267
6,Summary-level,MoverScore,Pearson,0.449951,0.449835,-0.000116
7,Summary-level,MoverScore,Spearman,0.423227,0.423110,-0.000117
8,Summary-level,MoverScore,Kendall,0.343648,0.343551,-0.000097
9,Summary-level,ROUGE-L,Pearson,0.354385,0.354092,-0.000293



Saved:
ROSE_SUMMARY_LEVEL_BOOTSTRAP_RAW.parquet
ROSE_SYSTEM_LEVEL_BOOTSTRAP_RAW.parquet
ROSE_SUMMARY_LEVEL_BOOTSTRAP_CI.csv
ROSE_SYSTEM_LEVEL_BOOTSTRAP_CI.csv
ROSE_SUMMARY_LEVEL_BOOTSTRAP_COMPACT.csv
ROSE_SYSTEM_LEVEL_BOOTSTRAP_COMPACT.csv

✅ STEP 12C PASSED
Summary-level and system-level bootstrap confidence intervals calculated.


In [210]:
# ============================================================
# STEP 12D: Paired bootstrap significance for
# summary-level and system-level correlations
# ============================================================

import numpy as np
import pandas as pd


LEARNED_METHOD = "Learned Hybrid"

BASELINES = [
    "BERTScore F1",
    "MoverScore",
    "ROUGE-L",
    "Manual Hybrid"
]

CORR_METRICS = [
    "Pearson",
    "Spearman",
    "Kendall"
]


# ------------------------------------------------------------
# 1. Holm adjustment helper
# ------------------------------------------------------------

def holm_adjust(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    m = len(p_values)

    order = np.argsort(
        p_values
    )

    sorted_p = p_values[
        order
    ]

    adjusted_sorted = np.empty(
        m,
        dtype=float
    )

    running_max = 0.0

    for i, p in enumerate(sorted_p):

        adjusted = (
            (m - i) * p
        )

        running_max = max(
            running_max,
            adjusted
        )

        adjusted_sorted[i] = min(
            running_max,
            1.0
        )

    adjusted = np.empty(
        m,
        dtype=float
    )

    adjusted[
        order
    ] = adjusted_sorted

    return adjusted


# ------------------------------------------------------------
# 2. Paired comparison function
# ------------------------------------------------------------

def paired_meta_comparisons(
    bootstrap_df,
    observed_df,
    level_name
):

    records = []

    observed_lookup = (
        observed_df
        .set_index("Method")
    )


    for metric in CORR_METRICS:

        # --------------------------------------------
        # One row per bootstrap replicate,
        # one column per method
        # --------------------------------------------

        metric_df = (
            bootstrap_df[
                bootstrap_df[
                    "Metric"
                ] == metric
            ]
            .pivot(
                index="bootstrap",
                columns="Method",
                values="Correlation"
            )
            .sort_index()
        )


        assert len(
            metric_df
        ) == 5000

        assert (
            metric_df
            .notna()
            .all()
            .all()
        )


        learned_boot = (
            metric_df[
                LEARNED_METHOD
            ]
            .to_numpy()
        )


        for baseline in BASELINES:

            baseline_boot = (
                metric_df[
                    baseline
                ]
                .to_numpy()
            )


            # Positive = Learned better
            delta = (
                learned_boot
                - baseline_boot
            )


            observed_delta = (
                float(
                    observed_lookup.loc[
                        LEARNED_METHOD,
                        metric
                    ]
                )
                -
                float(
                    observed_lookup.loc[
                        baseline,
                        metric
                    ]
                )
            )


            ci_lower = np.percentile(
                delta,
                2.5
            )

            ci_upper = np.percentile(
                delta,
                97.5
            )


            # ----------------------------------------
            # Empirical two-sided bootstrap p-value
            # +1 correction prevents p = 0
            # ----------------------------------------

            lower_tail = (
                np.sum(
                    delta <= 0
                ) + 1
            ) / (
                len(delta) + 1
            )

            upper_tail = (
                np.sum(
                    delta >= 0
                ) + 1
            ) / (
                len(delta) + 1
            )

            p_raw = min(
                1.0,
                2 * min(
                    lower_tail,
                    upper_tail
                )
            )


            records.append({

                "Level":
                    level_name,

                "Metric":
                    metric,

                "Baseline":
                    baseline,

                "Observed_Delta":
                    observed_delta,

                "Bootstrap_Mean_Delta":
                    np.mean(delta),

                "CI_Lower":
                    ci_lower,

                "CI_Upper":
                    ci_upper,

                "CI_Excludes_Zero":
                    (
                        ci_lower > 0
                        or
                        ci_upper < 0
                    ),

                "Raw_Bootstrap_P":
                    p_raw
            })


    result_df = pd.DataFrame(
        records
    )


    # --------------------------------------------------------
    # Holm correction separately inside each metric
    # --------------------------------------------------------

    result_df[
        "Holm_Adjusted_P"
    ] = np.nan


    for metric in CORR_METRICS:

        mask = (
            result_df[
                "Metric"
            ] == metric
        )

        corrected = holm_adjust(
            result_df
            .loc[
                mask,
                "Raw_Bootstrap_P"
            ]
            .to_numpy()
        )

        result_df.loc[
            mask,
            "Holm_Adjusted_P"
        ] = corrected


    result_df[
        "Significant_Holm_0.05"
    ] = (
        result_df[
            "Holm_Adjusted_P"
        ] < 0.05
    )


    return result_df


# ------------------------------------------------------------
# 3. Summary-level paired analysis
# ------------------------------------------------------------

summary_meta_paired_df = (
    paired_meta_comparisons(

        summary_meta_bootstrap_df,

        summary_level_results_df,

        "Summary-level"
    )
)


# ------------------------------------------------------------
# 4. System-level paired analysis
# ------------------------------------------------------------

system_meta_paired_df = (
    paired_meta_comparisons(

        system_meta_bootstrap_df,

        system_level_results_df,

        "System-level"
    )
)


# ------------------------------------------------------------
# 5. Combine
# ------------------------------------------------------------

meta_paired_all_df = pd.concat(
    [
        summary_meta_paired_df,
        system_meta_paired_df
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 6. Display detailed results
# ------------------------------------------------------------

print("=" * 125)
print("SUMMARY-LEVEL: LEARNED HYBRID VS BASELINES")
print("Positive delta = Learned Hybrid stronger")
print("=" * 125)

display(
    summary_meta_paired_df[
        [
            "Metric",
            "Baseline",
            "Observed_Delta",
            "Bootstrap_Mean_Delta",
            "CI_Lower",
            "CI_Upper",
            "Raw_Bootstrap_P",
            "Holm_Adjusted_P",
            "Significant_Holm_0.05"
        ]
    ].round(6)
)


print("\n" + "=" * 125)
print("SYSTEM-LEVEL: LEARNED HYBRID VS BASELINES")
print("Positive delta = Learned Hybrid stronger")
print("=" * 125)

display(
    system_meta_paired_df[
        [
            "Metric",
            "Baseline",
            "Observed_Delta",
            "Bootstrap_Mean_Delta",
            "CI_Lower",
            "CI_Upper",
            "Raw_Bootstrap_P",
            "Holm_Adjusted_P",
            "Significant_Holm_0.05"
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 7. Human-readable summary
# ------------------------------------------------------------

for level_name, result_df in [

    (
        "SUMMARY-LEVEL",
        summary_meta_paired_df
    ),

    (
        "SYSTEM-LEVEL",
        system_meta_paired_df
    )
]:

    print(
        "\n"
        + "=" * 105
    )

    print(
        level_name,
        "HOLM-CORRECTED SIGNIFICANCE"
    )

    print(
        "=" * 105
    )


    for metric in CORR_METRICS:

        print(
            f"\n{metric}"
        )

        print(
            "-" * len(metric)
        )


        subset = result_df[
            result_df[
                "Metric"
            ] == metric
        ]


        for _, row in subset.iterrows():

            p = row[
                "Holm_Adjusted_P"
            ]

            if p < 0.001:

                p_text = (
                    "p < .001"
                )

            else:

                p_text = (
                    f"p = {p:.4f}"
                )


            status = (
                "SIGNIFICANT"
                if row[
                    "Significant_Holm_0.05"
                ]
                else
                "NOT SIGNIFICANT"
            )


            print(

                f"  vs "
                f"{row['Baseline']:15s}"

                f" Δ="
                f"{row['Observed_Delta']:+.4f}"

                f" | 95% CI "
                f"[{row['CI_Lower']:.4f}, "
                f"{row['CI_Upper']:.4f}]"

                f" | {p_text}"

                f" | {status}"
            )


# ------------------------------------------------------------
# 8. Compact thesis-friendly table
# ------------------------------------------------------------

compact_rows = []


for _, row in meta_paired_all_df.iterrows():

    compact_rows.append({

        "Level":
            row[
                "Level"
            ],

        "Metric":
            row[
                "Metric"
            ],

        "Baseline":
            row[
                "Baseline"
            ],

        "Delta":
            row[
                "Observed_Delta"
            ],

        "95% CI":
            (
                f"["
                f"{row['CI_Lower']:.4f}, "
                f"{row['CI_Upper']:.4f}"
                f"]"
            ),

        "Holm p":
            row[
                "Holm_Adjusted_P"
            ],

        "Significant":
            row[
                "Significant_Holm_0.05"
            ]
    })


meta_paired_compact_df = pd.DataFrame(
    compact_rows
)


print(
    "\n"
    + "=" * 125
)

print(
    "COMPACT SUMMARY/SYSTEM PAIRED RESULTS"
)

print(
    "=" * 125
)


display(
    meta_paired_compact_df.round(6)
)


# ------------------------------------------------------------
# 9. Save
# ------------------------------------------------------------

summary_meta_paired_df.to_csv(
    "ROSE_SUMMARY_LEVEL_PAIRED_SIGNIFICANCE.csv",
    index=False
)

system_meta_paired_df.to_csv(
    "ROSE_SYSTEM_LEVEL_PAIRED_SIGNIFICANCE.csv",
    index=False
)

meta_paired_compact_df.to_csv(
    "ROSE_SUMMARY_SYSTEM_PAIRED_COMPACT.csv",
    index=False
)


print("\nSaved:")
print(
    "ROSE_SUMMARY_LEVEL_PAIRED_SIGNIFICANCE.csv"
)
print(
    "ROSE_SYSTEM_LEVEL_PAIRED_SIGNIFICANCE.csv"
)
print(
    "ROSE_SUMMARY_SYSTEM_PAIRED_COMPACT.csv"
)


print(
    "\n"
    + "=" * 125
)

print(
    "✅ STEP 12D PASSED"
)

print(
    "Paired summary-level and system-level "
    "correlation differences tested."
)

print(
    "=" * 125
)

SUMMARY-LEVEL: LEARNED HYBRID VS BASELINES
Positive delta = Learned Hybrid stronger


,Metric,Baseline,Observed_Delta,Bootstrap_Mean_Delta,CI_Lower,CI_Upper,Raw_Bootstrap_P,Holm_Adjusted_P,Significant_Holm_0.05
0,Pearson,BERTScore F1,0.085585,0.085680,0.036327,0.136170,0.000800,0.001600,True
1,Pearson,MoverScore,0.058493,0.058497,-0.001828,0.117591,0.056389,0.056389,False
2,Pearson,ROUGE-L,0.154059,0.154239,0.095897,0.216499,0.000400,0.001600,True
3,Pearson,Manual Hybrid,0.293852,0.294539,0.218682,0.371613,0.000400,0.001600,True
4,Spearman,BERTScore F1,0.076863,0.076971,0.029284,0.126881,0.001200,0.002400,True
5,Spearman,MoverScore,0.033605,0.033559,-0.028196,0.096733,0.281944,0.281944,False
6,Spearman,ROUGE-L,0.128527,0.128632,0.066845,0.194280,0.000400,0.001600,True
7,Spearman,Manual Hybrid,0.225950,0.226403,0.149434,0.304620,0.000400,0.001600,True
8,Kendall,BERTScore F1,0.064729,0.064865,0.023734,0.108133,0.002799,0.005599,True
9,Kendall,MoverScore,0.026544,0.026510,-0.026779,0.081169,0.326735,0.326735,False



SYSTEM-LEVEL: LEARNED HYBRID VS BASELINES
Positive delta = Learned Hybrid stronger


,Metric,Baseline,Observed_Delta,Bootstrap_Mean_Delta,CI_Lower,CI_Upper,Raw_Bootstrap_P,Holm_Adjusted_P,Significant_Holm_0.05
0,Pearson,BERTScore F1,0.653825,0.559815,0.264989,0.835732,0.001200,0.001600,True
1,Pearson,MoverScore,0.890594,0.733569,0.396172,1.060410,0.000400,0.001600,True
2,Pearson,ROUGE-L,1.132142,0.940502,0.563741,1.290051,0.000400,0.001600,True
3,Pearson,Manual Hybrid,1.381706,1.151588,0.748428,1.517016,0.000400,0.001600,True
4,Spearman,BERTScore F1,0.559441,0.565161,0.251748,0.860140,0.003599,0.003599,True
5,Spearman,MoverScore,0.797203,0.738996,0.377622,1.111888,0.000800,0.001600,True
6,Spearman,ROUGE-L,1.013986,0.926621,0.552273,1.286713,0.000400,0.001600,True
7,Spearman,Manual Hybrid,1.202797,1.115915,0.699301,1.510490,0.000400,0.001600,True
8,Kendall,BERTScore F1,0.515152,0.473679,0.181818,0.727273,0.003999,0.003999,True
9,Kendall,MoverScore,0.575758,0.590788,0.272727,0.909091,0.002000,0.003999,True



SUMMARY-LEVEL HOLM-CORRECTED SIGNIFICANCE

Pearson
-------
  vs BERTScore F1    Δ=+0.0856 | 95% CI [0.0363, 0.1362] | p = 0.0016 | SIGNIFICANT
  vs MoverScore      Δ=+0.0585 | 95% CI [-0.0018, 0.1176] | p = 0.0564 | NOT SIGNIFICANT
  vs ROUGE-L         Δ=+0.1541 | 95% CI [0.0959, 0.2165] | p = 0.0016 | SIGNIFICANT
  vs Manual Hybrid   Δ=+0.2939 | 95% CI [0.2187, 0.3716] | p = 0.0016 | SIGNIFICANT

Spearman
--------
  vs BERTScore F1    Δ=+0.0769 | 95% CI [0.0293, 0.1269] | p = 0.0024 | SIGNIFICANT
  vs MoverScore      Δ=+0.0336 | 95% CI [-0.0282, 0.0967] | p = 0.2819 | NOT SIGNIFICANT
  vs ROUGE-L         Δ=+0.1285 | 95% CI [0.0668, 0.1943] | p = 0.0016 | SIGNIFICANT
  vs Manual Hybrid   Δ=+0.2259 | 95% CI [0.1494, 0.3046] | p = 0.0016 | SIGNIFICANT

Kendall
-------
  vs BERTScore F1    Δ=+0.0647 | 95% CI [0.0237, 0.1081] | p = 0.0056 | SIGNIFICANT
  vs MoverScore      Δ=+0.0265 | 95% CI [-0.0268, 0.0812] | p = 0.3267 | NOT SIGNIFICANT
  vs ROUGE-L         Δ=+0.1078 | 95% CI [0.0571, 

,Level,Metric,Baseline,Delta,95% CI,Holm p,Significant
0,Summary-level,Pearson,BERTScore F1,0.085585,"[0.0363, 0.1362]",0.001600,True
1,Summary-level,Pearson,MoverScore,0.058493,"[-0.0018, 0.1176]",0.056389,False
2,Summary-level,Pearson,ROUGE-L,0.154059,"[0.0959, 0.2165]",0.001600,True
3,Summary-level,Pearson,Manual Hybrid,0.293852,"[0.2187, 0.3716]",0.001600,True
4,Summary-level,Spearman,BERTScore F1,0.076863,"[0.0293, 0.1269]",0.002400,True
5,Summary-level,Spearman,MoverScore,0.033605,"[-0.0282, 0.0967]",0.281944,False
6,Summary-level,Spearman,ROUGE-L,0.128527,"[0.0668, 0.1943]",0.001600,True
7,Summary-level,Spearman,Manual Hybrid,0.225950,"[0.1494, 0.3046]",0.001600,True
8,Summary-level,Kendall,BERTScore F1,0.064729,"[0.0237, 0.1081]",0.005599,True
9,Summary-level,Kendall,MoverScore,0.026544,"[-0.0268, 0.0812]",0.326735,False



Saved:
ROSE_SUMMARY_LEVEL_PAIRED_SIGNIFICANCE.csv
ROSE_SYSTEM_LEVEL_PAIRED_SIGNIFICANCE.csv
ROSE_SUMMARY_SYSTEM_PAIRED_COMPACT.csv

✅ STEP 12D PASSED
Paired summary-level and system-level correlation differences tested.


## 20. Final Frozen Model Specification

This section documents the final frozen RoSE Learned Hybrid model and its fitted preprocessing parameters.

The saved MinMax scaler, Ridge Regression model, complete pipeline, and coefficient table are reloaded to verify the exact model used for evaluation. The final feature order, scaling parameters, Ridge coefficients, intercept, regularisation setting, and clipping configuration are reported to provide a transparent specification of the frozen Learned Hybrid metric.

In [1]:
from pathlib import Path
import joblib

SCALER_PATH = Path("ROSE_feature_minmax_scaler.joblib")

print("Scaler exists:", SCALER_PATH.exists())
print("Full path:", SCALER_PATH.resolve())

feature_scaler = joblib.load(SCALER_PATH)

print(feature_scaler)

Scaler exists: True
Full path: /Users/sudip/Downloads/ROSE_feature_minmax_scaler.joblib
MinMaxScaler(clip=True)


In [5]:
import joblib
import pandas as pd
from pathlib import Path

SCALER_PATH = Path("ROSE_feature_minmax_scaler.joblib")
MODEL_PATH = Path("ROSE_FINAL_LEARNED_HYBRID_RIDGE.joblib")
PIPELINE_PATH = Path("ROSE_FINAL_LEARNED_HYBRID_PIPELINE.joblib")
COEF_PATH = Path("ROSE_FINAL_RIDGE_COEFFICIENTS.csv")

print("Scaler exists:", SCALER_PATH.exists())
print("Model exists:", MODEL_PATH.exists())
print("Pipeline exists:", PIPELINE_PATH.exists())
print("Coefficient CSV exists:", COEF_PATH.exists())

feature_scaler = joblib.load(SCALER_PATH)
final_ridge_model = joblib.load(MODEL_PATH)
final_pipeline = joblib.load(PIPELINE_PATH)

ridge_coefficients_df = pd.read_csv(COEF_PATH)

print("\nSCALER:")
print(feature_scaler)

print("\nRIDGE MODEL:")
print(final_ridge_model)

print("\nFINAL PIPELINE:")
print(final_pipeline)

print("\nCOEFFICIENT FILE COLUMNS:")
print(ridge_coefficients_df.columns.tolist())

print("\nCOEFFICIENT TABLE:")
display(ridge_coefficients_df)

print("\nRIDGE INTERCEPT:")
print(repr(float(final_ridge_model.intercept_)))

print("\nSCALER FEATURE NAMES:")
if hasattr(feature_scaler, "feature_names_in_"):
    print(feature_scaler.feature_names_in_)
else:
    print("No feature_names_in_ stored")

Scaler exists: True
Model exists: True
Pipeline exists: True
Coefficient CSV exists: True

SCALER:
MinMaxScaler(clip=True)

RIDGE MODEL:
Ridge()

FINAL PIPELINE:
Pipeline(steps=[('scaler', MinMaxScaler(clip=True)), ('model', Ridge())])

COEFFICIENT FILE COLUMNS:
['Feature', 'Coefficient', 'Absolute_Coefficient']

COEFFICIENT TABLE:


,Feature,Coefficient,Absolute_Coefficient
0,bertScoreF1,0.839712,0.839712
1,moverScore,0.348279,0.348279
2,average_sentence_length,0.296204,0.296204
3,length_adequacy,-0.270745,0.270745
4,rougeL,0.236657,0.236657
5,nli_mean_entailment,0.092490,0.092490
6,nli_mean_neutral,-0.071075,0.071075
7,nli_contradicted_sentence_ratio,-0.062953,0.062953
8,nli_min_entailment,-0.050336,0.050336
9,redundancy_quality,0.044007,0.044007



RIDGE INTERCEPT:
-0.10694132383904054

SCALER FEATURE NAMES:
['rougeL' 'bertScoreF1' 'moverScore' 'nli_mean_entailment'
 'nli_mean_contradiction' 'nli_mean_neutral' 'nli_max_contradiction'
 'nli_min_entailment' 'nli_supported_sentence_ratio'
 'nli_contradicted_sentence_ratio' 'length_adequacy'
 'sentence_count_ratio' 'redundancy_quality' 'average_sentence_length']


In [7]:
import pandas as pd

# Use the complete frozen pipeline
scaler = final_pipeline.named_steps["scaler"]
model = final_pipeline.named_steps["model"]

# Exact feature order used by the fitted scaler/model
features = list(scaler.feature_names_in_)

assert len(features) == 14
assert len(model.coef_) == 14

rose_appendix = pd.DataFrame({
    "Feature": features,
    "Data minimum": scaler.data_min_,
    "Data maximum": scaler.data_max_,
    "Scaler scale": scaler.scale_,
    "Min adjustment": scaler.min_,
    "Coefficient": model.coef_
})

display(rose_appendix.round(6))

print("\nINTERCEPT:")
print(f"{float(model.intercept_):.10f}")

print("\nMODEL CHECK:")
print("Alpha:", model.alpha)
print("Scaler clip:", scaler.clip)

,Feature,Data minimum,Data maximum,Scaler scale,Min adjustment,Coefficient
0,rougeL,0.027778,0.981481,1.048544,-0.029126,0.236657
1,bertScoreF1,-0.200838,0.946296,0.871737,0.175078,0.839712
2,moverScore,0.471747,0.868020,2.523509,-1.190457,0.348279
3,nli_mean_entailment,0.000773,0.998226,1.002554,-0.000775,0.092490
4,nli_mean_contradiction,0.000633,0.725900,1.378803,-0.000873,-0.029504
5,nli_mean_neutral,0.000765,0.997675,1.003099,-0.000767,-0.071075
6,nli_max_contradiction,0.000948,0.999179,1.001772,-0.000950,0.008209
7,nli_min_entailment,0.000093,0.997563,1.002536,-0.000093,-0.050336
8,nli_supported_sentence_ratio,0.000000,1.000000,1.000000,0.000000,-0.036984
9,nli_contradicted_sentence_ratio,0.000000,0.750000,1.333333,0.000000,-0.062953



INTERCEPT:
-0.1069413238

MODEL CHECK:
Alpha: 1.0
Scaler clip: True
